In [2]:
# SPUR DuckDB extension setup cell v1
# requirements: {"active":[{"packageName":"duckrouting","source":"community"},{"packageName":"shellfs","source":"community"}],"pending":[]}
# Managed by Jute. Change packages through the extension sidebar.
# SPUR DuckDB bootstrap
import duckdb
import sys
_SPUR_DUCKDB_EXTENSION_PATH = "/Users/kevintruong/.spur/extensions/spur_rest.duckdb_extension"
# Shared by notebook setup and classic SQL cells. Preserve the generic path for
# older bundles; new bundles carry exact native ABI artifacts per engine.
from pathlib import Path as _SpurExtensionPath

_spur_extension = _SpurExtensionPath(_SPUR_DUCKDB_EXTENSION_PATH)
_spur_versioned_extension = (
    _spur_extension.parent
    / ("duckdb-v" + duckdb.__version__)
    / _spur_extension.name
)
if _spur_versioned_extension.is_file():
    _SPUR_DUCKDB_EXTENSION_PATH = str(_spur_versioned_extension)

if (
    sys.platform in ("linux", "darwin")
    and duckdb.__version__ in ("1.4.4", "1.5.5")
    and "_SPUR_DUCKDB_HOST_SYMBOLS" not in globals()
):
    import ctypes as _spur_ctypes
    import _duckdb as _spur_host_duckdb

    # Promote the already-loaded Python engine's symbols. This opens the same
    # module handle; it neither imports another engine nor replaces connections.
    _SPUR_DUCKDB_HOST_SYMBOLS = _spur_ctypes.CDLL(
        _spur_host_duckdb.__file__, mode=_spur_ctypes.RTLD_GLOBAL
    )

_SPUR_DUCKDB_EXTENSION_SQL = _SPUR_DUCKDB_EXTENSION_PATH.replace("'", "''")

if "_SPUR_DUCKDB_CONNECTION" not in globals():
    _SPUR_DUCKDB_CONNECTION = duckdb.connect(
        database=":memory:",
        config={"allow_unsigned_extensions": "true",
                 "autoload_known_extensions": "false",
                 "autoinstall_known_extensions": "false"},
    )

duckdb.set_default_connection(_SPUR_DUCKDB_CONNECTION)
# Embedded helper source contains no token, credential path or capability lease.
# The approved local launcher passes a private inherited socket descriptor, not
# executable source or a notebook/widget message. Same-kernel code is trusted.
def _spur_managed_connection(db):
    """Only the host's selected connection is subject to managed policy."""
    import types
    connection = db.default_connection() if isinstance(db, types.ModuleType) else db
    if connection is not globals().get("_SPUR_DUCKDB_CONNECTION"):
        return None
    try:
        connection.sql("SET autoload_known_extensions = false")
        connection.sql("SET autoinstall_known_extensions = false")
    except Exception:
        return None  # existing locked/custom connections have an explicit boundary
    return connection


# Required SpurLab baseline: JSON/parquet are statically linked in the managed
# wheel; httpfs supplies remote files and REST gateway dependencies; spur_rest is
# the separately packaged, host-verified gateway (never a repository download).
# Remote datasource attach code explicitly installs/loads its provider adapter.
# Spatial and Jev are optional. No browsing/query path calls operate("install").
# Fixed metadata query only. Identifiers and descriptions remain inert JSON; no
# catalog value is interpolated as SQL. The limits bound the shell reply too.
def _spur_function_catalog(db):
    import json
    limit = 10000
    rows = db.sql("SELECT database_name, schema_name, function_name, function_type, "
                  "parameters, parameter_types, varargs, return_type, description, examples, function_oid "
                  "FROM duckdb_functions() ORDER BY database_name, schema_name, function_name, "
                  "function_type, parameter_types, varargs LIMIT 10001").fetchall()
    functions, size = [], 0
    complete = len(rows) <= limit
    for row in rows[:limit]:
        value = dict(zip(("database_name", "schema_name", "name", "function_type", "parameters",
                          "parameter_types", "varargs", "return_type", "description", "examples", "function_oid"), row))
        value["ownership"] = "unknown"
        encoded = json.dumps(value, ensure_ascii=True)
        # Oversized metadata is omitted, with incomplete coverage made explicit.
        if len(encoded) > 65536 or size + len(encoded) > 4 * 1024 * 1024:
            complete = False
            continue
        functions.append(value)
        size += len(encoded)
    return {"status": "complete" if complete else "incomplete", "functions": functions,
            "truncated": not complete}


def _spur_function_identity(value):
    return (value["database_name"], value["schema_name"], value["name"],
            value["function_type"], tuple(value["parameter_types"]), value["varargs"])


def _spur_snapshot_function_registration(db):
    try:
        snapshot = _spur_function_catalog(db)
        if snapshot["status"] == "complete" and snapshot.get("truncated") is False:
            return snapshot
    except Exception:
        pass  # Optional metadata must not prevent an authorized extension load.
    return None


def _spur_record_function_ownership(db, extension, before):
    """Retain registrations observed around a verified load on this connection."""
    if before is None:
        return
    try:
        if before["status"] != "complete" or before.get("truncated") is not False:
            return
        after = _spur_snapshot_function_registration(db)
        if after is None:
            return
        def key(row):
            return (*_spur_function_identity(row), row.get("return_type"), row["function_oid"])
        previous = {key(row) for row in before["functions"]}
        registrations = globals().setdefault("_SPUR_DUCKDB_FUNCTION_OWNERSHIP", {})
        retained = registrations.get(id(db))
        owners = dict(retained[1]) if retained is not None and retained[0] is db else {}
        additions = [key(row) for row in after["functions"]
                     if row["function_oid"] is not None and key(row) not in previous]
        for identity in additions:
            owners[identity] = set(owners.get(identity, ())) | {extension}
        registrations[id(db)] = (db, owners)
    except Exception:
        pass  # Incomplete/failed evidence stays unattributed.


class _SpurDuckDBRuntime:
    BASELINE = ("json", "parquet", "httpfs", "spur_rest")
    REPOSITORIES = {
        # DuckDB's built-in installer handles HTTP without httpfs; HTTPS itself
        # requires httpfs. Use the official signed distribution bootstrap, never
        # temporarily re-enable autoload to download its own dependency.
        "httpfs": "http://extensions.duckdb.org",
        "spatial": "http://extensions.duckdb.org",
        "jev": "http://community-extensions.duckdb.org",
    }

    def __init__(self, db, platform):
        self.db, self.platform = db, platform
        self.entries = {}
        self.inventories = {}  # inert metadata; never activation authority
        self.verified = {}  # at most one private verified copy per curated entry
        self.loaded = set()  # artifact verification is not selected-connection load

    def _entry(self, name):
        return self.entries.setdefault(name, {
            "name": name, "installed": None,
            "desired_enabled": name in self.BASELINE, "loaded": None,
            "provider_health": "unknown", "restart_required": False,
            "activation_transport_restart": False,
            "stage": "idle", "reason": None, "policy": "unknown",
            "artifact_digest": None, "signature_verified": False,
        })

    def _observe(self, db, name):
        return db.sql(
            "SELECT loaded, installed, install_path, installed_from, install_mode "
            "FROM duckdb_extensions() WHERE extension_name = ?", params=[name]
        ).fetchone()

    def _refresh(self, entry):
        row = self._observe(self.db, entry["name"])
        entry["loaded"] = bool(row and row[0])
        entry["installed"] = bool(row and row[1])
        return row

    def _artifact(self, name, install, cancelled):
        # Verification happens in a separate database with signature and ABI
        # checks ON, even when the selected gateway connection allows unsigned
        # bundled code. A repository label or hash alone is not signing evidence.
        import duckdb
        import hashlib
        from pathlib import Path
        import shutil
        import tempfile
        directory = self.db.sql("SELECT current_setting('extension_directory')").fetchone()[0]
        config = {"allow_unsigned_extensions": "false",
                  "allow_community_extensions": "true" if name == "jev" else "false",
                  "allow_extensions_metadata_mismatch": "false",
                  "autoload_known_extensions": "false",
                  "autoinstall_known_extensions": "false"}
        config["home_directory"] = self.db.sql("SELECT current_setting('home_directory')").fetchone()[0]
        if directory:
            config["extension_directory"] = directory
        entry = self._entry(name)
        entry["signature_verified"] = False
        with duckdb.connect(config=config) as verifier:
            if (verifier.sql("SELECT version()").fetchone()[0] != "v1.5.5"
                    or verifier.sql("PRAGMA platform").fetchone()[0] != self.platform):
                raise ValueError("runtime_mismatch")
            entry["stage"] = "installing" if install else "verifying"
            row = self._observe(verifier, name)
            if install and (row is None or not row[1]):
                # Install only when absent. DuckDB rejects INSTALL across HTTP/
                # HTTPS origin spellings even for the same official repository.
                # Cached artifacts follow the same origin/signature/ABI checks
                # below; never FORCE/UPDATE or silently replace rejected bytes.
                verifier.install_extension(name, repository_url=self.REPOSITORIES[name])
                row = self._observe(verifier, name)
            if row is None or not row[1]:
                raise ValueError("not_installed")
            entry["installed"] = True
            if cancelled():
                raise InterruptedError()
            entry["stage"] = "verifying"
            repository = self.REPOSITORIES[name]
            aliases = (repository, repository.replace("http:", "https:"),
                       "community" if name == "jev" else "core")
            if row[3] not in aliases or row[4] != "REPOSITORY":
                raise ValueError("repository_mismatch")
            # Freeze the bytes in a private directory before the verifier loads
            # them. Load those same bytes into the selected connection, never a
            # mutable cache alias. Same-user/kernel arbitrary code is trusted.
            private = tempfile.TemporaryDirectory(prefix="spur-duckdb-verified-")
            try:
                path = Path(private.name) / (name + ".duckdb_extension")
                shutil.copyfile(row[2], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).hexdigest()
                try:
                    before = _spur_function_catalog(verifier)
                except Exception:
                    before = None
                verifier.load_extension(str(path))  # engine checks signature/ABI
                if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
                    raise ValueError("artifact_changed")
                try:
                    after = _spur_function_catalog(verifier)
                    if before is None:
                        raise ValueError("inventory_failed")
                    previous_rows = {_spur_function_identity(f): f for f in before["functions"]}
                    functions = [f for f in after["functions"]
                                 if previous_rows.get(_spur_function_identity(f)) != f]
                    complete = before["status"] == after["status"] == "complete"
                    inventory = {"status": "complete" if complete else "incomplete",
                                 "functions": functions, "truncated": not complete}
                except Exception:
                    inventory = {"status": "failed", "functions": [], "truncated": False}
                inventory.update(artifact_digest=digest, provenance="isolated_signed_verifier_load_delta")
                self.inventories[name] = inventory
            except BaseException:
                private.cleanup()
                raise
            previous = self.verified.get(name)
            if previous is not None:
                previous.cleanup()
            self.verified[name] = private
            return str(path), digest

    def operate(self, name, action, cancelled=lambda: False):
        """Explicit, nonsecret job; called only on the kernel shell queue.

        Cancellation is cooperative at stages. It never interrupts a user query;
        installed but not loaded artifacts remain available for an explicit retry.
        There is no UNLOAD in DuckDB: disable reports loaded + restart_required.
        """
        entry = self._entry(name)
        entry["stage"], entry["reason"] = "resolving", None
        if action == "disable" and name not in self.BASELINE:
            entry["desired_enabled"] = False
            if name == "jev":
                _spur_close_jev_channel()
        try:
            if name not in (*self.BASELINE, "spatial", "jev"):
                raise ValueError("unsupported_extension")
            if self.db is None:
                raise ValueError("no_connection")
            if self.db is not globals().get("_SPUR_DUCKDB_CONNECTION"):
                entry["policy"] = "unenforceable"
                raise ValueError("unmanaged_connection")
            if (self.db.sql("SELECT version()").fetchone()[0] != "v1.5.5"
                    or self.db.sql("PRAGMA platform").fetchone()[0] != self.platform):
                raise ValueError("runtime_mismatch")
            if _spur_managed_connection(self.db) is None:
                entry["policy"] = "unenforceable"
                raise ValueError("policy_unenforceable")
            entry["policy"] = "enforced"
            self._refresh(entry)
            if action == "disable":
                if name in self.BASELINE:
                    raise ValueError("required_baseline")
                entry["desired_enabled"] = False
                entry["restart_required"] = entry["loaded"] or name == "jev"
                entry["activation_transport_restart"] = name == "jev"
                entry["reason"] = "restart_required" if entry["restart_required"] else None
                entry["stage"] = "succeeded"
                return dict(entry)
            if cancelled():
                raise InterruptedError()
            if action == "invalidate_credential_channel" and name == "jev":
                _spur_close_jev_channel()
                entry["restart_required"] = True
                entry["activation_transport_restart"] = True
                entry["reason"] = "restart_required"
            elif action == "enable":
                entry["desired_enabled"] = True
                if name == "jev" and not self._channel_ready():
                    entry["restart_required"] = True
                    entry["activation_transport_restart"] = True
                    entry["reason"] = self._channel_reason()
            elif action in ("install", "revalidate", "load", "prepare_verification"):
                if action == "prepare_verification" and (name not in self.REPOSITORIES or not entry["installed"]
                        or not entry["signature_verified"] or not entry["artifact_digest"]):
                    raise ValueError("not_installed")
                if action == "load" and not entry["desired_enabled"]:
                    raise ValueError("setup_required_without_autoload")
                if name in ("json", "parquet"):
                    row = self._observe(self.db, name)
                    if not row or row[4] != "STATICALLY_LINKED" or not row[0]:
                        raise ValueError("baseline_unavailable")
                elif name == "spur_rest":
                    # Owned by the bundled artifact installer/bootstrap. Never
                    # weaken signature policy or download a substitute here.
                    if not entry["loaded"]:
                        raise ValueError("baseline_unavailable")
                else:
                    if entry["loaded"] and name == "httpfs" and name not in self.loaded:
                        # Connection setup may have loaded the same signed baseline
                        # first. Accept only its exact-connection frozen-byte proof.
                        from pathlib import Path
                        import hashlib
                        proof = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {}).get((id(self.db), name))
                        if proof and proof[0] is self.db and proof[1] == "core":
                            try:
                                digest = hashlib.sha256((Path(proof[2].name) / "httpfs.duckdb_extension").read_bytes()).digest()
                                if digest == proof[3]:
                                    self.verified[name] = proof[2]
                                    self.loaded.add(name)
                                    entry.update(signature_verified=True, artifact_digest=digest.hex(),
                                                 restart_required=False, activation_transport_restart=False)
                            except OSError:
                                pass
                    if entry["loaded"] and name not in self.loaded:
                        entry["signature_verified"] = False
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = False
                        raise ValueError("preexisting_load_unverified")
                    if entry["loaded"] and action in ("load", "prepare_verification"):
                        # Explicit retry may reacquire discovery in the same isolated
                        # signature verifier; it must not change selected enablement.
                        if action == "prepare_verification" and self.inventories.get(name, {}).get("status") != "complete":
                            loaded_digest = entry["artifact_digest"]
                            _, refreshed_digest = self._artifact(name, False, cancelled)
                            if refreshed_digest != loaded_digest:
                                entry["restart_required"] = True
                                entry["activation_transport_restart"] = False
                                raise ValueError("artifact_changed")
                            entry["signature_verified"] = True
                        if name == "jev" and action == "load" and not self._channel_ready():
                            entry["restart_required"] = True
                            entry["activation_transport_restart"] = True
                            entry["reason"] = self._channel_reason()
                        if entry["restart_required"] and entry["activation_transport_restart"]:
                            entry["reason"] = self._channel_reason()
                        entry["stage"] = "succeeded"
                        return dict(entry)
                    if entry["loaded"]:
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = False
                        raise ValueError("restart_required")
                    path, digest = self._artifact(name, action == "install", cancelled)
                    entry["installed"] = True
                    entry["artifact_digest"] = digest
                    entry["signature_verified"] = True
                    if cancelled():
                        raise InterruptedError()
                    if action in ("load", "prepare_verification") and not entry["loaded"]:
                        entry["stage"] = "loading"
                        before = _spur_snapshot_function_registration(self.db)
                        self.db.load_extension(path)
                        self._refresh(entry)
                        if not entry["loaded"]:
                            raise ValueError("load_failed")
                        self.loaded.add(name)
                        _spur_record_function_ownership(self.db, name, before)
                    if name == "jev" and action != "prepare_verification" and not self._channel_ready():
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = True
                        entry["reason"] = self._channel_reason()
            elif action != "inspect":
                raise ValueError("unsupported_operation")
            if entry["restart_required"] and entry["activation_transport_restart"]:
                entry["reason"] = self._channel_reason()
            entry["stage"] = "succeeded"
        except (KeyboardInterrupt, InterruptedError):
            entry["reason"] = "cancelled"
        except ValueError as error:
            # Only our fixed reasons are exposed; never native exception text.
            allowed = {"runtime_mismatch", "unsupported_extension", "unmanaged_connection",
                       "required_baseline", "setup_required_without_autoload", "not_installed",
                       "repository_mismatch", "artifact_changed", "baseline_unavailable",
                       "preexisting_load_unverified", "load_failed", "unsupported_operation", "no_connection",
                       "policy_unenforceable", "restart_required"}
            entry["reason"] = str(error) if str(error) in allowed else "verification_failed"
        except Exception:
            entry["reason"] = {"installing": "install_failed", "loading": "load_failed",
                               "verifying": "verification_failed"}.get(entry["stage"], "policy_failed")
        return dict(entry)

    @staticmethod
    def _channel_ready():
        import os
        if os.name != "posix":
            return False
        try:
            channel = _spur_jev_channel()
            return channel is not None and channel.available()
        except Exception:
            return False

    @staticmethod
    def _channel_reason():
        import os
        return "restart_required" if os.name == "posix" else "credential_transport_unavailable"

    def close(self):
        """Release verified temporary copies after closing this connection."""
        for directory in self.verified.values():
            directory.cleanup()
        self.verified.clear()


def _spur_duckdb_job(platform, name, action, cancel_path=None):
    """Entry point for typed host jobs; no caller SQL and no credential payload."""
    from pathlib import Path
    global _SPUR_DUCKDB_RUNTIME
    db = globals().get("_SPUR_DUCKDB_CONNECTION")
    current = globals().get("_SPUR_DUCKDB_RUNTIME")
    if current is None or current.db is not db or current.platform != platform:
        current = _SPUR_DUCKDB_RUNTIME = _SpurDuckDBRuntime(db, platform)
    return current.operate(name, action, lambda: bool(cancel_path and Path(cancel_path).exists()))


def _spur_duckdb_baseline():
    """Load approved cached baseline only; missing artifacts need explicit setup.

    Used by both root bootstrap and the library's shared SQL session helper.
    The bundled gateway loader stays with its existing host-owned installer.
    """
    db = globals().get("_SPUR_DUCKDB_CONNECTION")
    if db is None or db.sql("SELECT version()").fetchone()[0] != "v1.5.5":
        return
    platform = db.sql("PRAGMA platform").fetchone()[0]
    current = globals().get("_SPUR_DUCKDB_RUNTIME")
    if current is not None and current.db is not db:
        current = None
    for name in ("json", "parquet", "httpfs"):
        if current is None or not current._entry(name)["loaded"]:
            _spur_duckdb_job(platform, name, "load")


class _SpurJevChannel:
    def __init__(self, descriptor):
        import socket
        self.socket = socket.socket(fileno=int(descriptor))
        self.socket.set_inheritable(False)
        self.lease = self._frame()

    def _read(self, count):
        chunks = bytearray()
        while len(chunks) < count:
            chunk = self.socket.recv(count - len(chunks))
            if not chunk:
                raise RuntimeError("Jev private credential channel closed")
            chunks.extend(chunk)
        return bytes(chunks)

    def _frame(self):
        return self._read(int.from_bytes(self._read(4), "big"))

    def _authorize(self, resolve):
        self.socket.sendall(len(self.lease).to_bytes(4, "big") + self.lease + bytes([resolve]))
        status = self._read(1)
        if status == b"\x02":
            return False
        if status != b"\x01":
            raise RuntimeError("denied")
        return True

    def available(self):
        try:
            return self._authorize(False)
        except Exception:
            self.socket.close()
            raise RuntimeError("Jev credential request denied or unavailable. Verify and enable the current managed local kernel in Settings.") from None

    def request(self):
        try:
            if not self._authorize(True):
                return None
            token = self._frame().decode("ascii")
            if not token or any(not 33 <= ord(c) <= 126 for c in token):
                raise ValueError("invalid credential")
            return token
        except Exception:
            self.socket.close()
            raise RuntimeError("Jev credential request denied or unavailable. Verify and enable the current managed local kernel in Settings.") from None


# Repeated generated SQL preambles preserve the captured channel. Failed
# attachment remains failed until kernel restart; never retry a legacy file.
if "_SPUR_JEV_BROKER" not in globals():
    _SPUR_JEV_BROKER = None
    _SPUR_JEV_BROKER_FAILED = False


def _spur_jev_channel():
    import os
    global _SPUR_JEV_BROKER, _SPUR_JEV_BROKER_FAILED
    if _SPUR_JEV_BROKER_FAILED:
        raise RuntimeError("Jev private credential channel is unavailable. Restart the managed local kernel.")
    descriptor = os.environ.pop("SPUR_JEV_BROKER_FD", None)
    if descriptor is not None:
        try:
            if _SPUR_JEV_BROKER is not None:
                raise RuntimeError("duplicate channel")
            _SPUR_JEV_BROKER = _SpurJevChannel(descriptor)
        except Exception:
            _SPUR_JEV_BROKER_FAILED = True
            raise RuntimeError("Jev private credential channel is unavailable. Restart the managed local kernel.") from None
    return _SPUR_JEV_BROKER


def _spur_close_jev_channel():
    """Disable/revoke/replacement is terminal for this inherited descriptor."""
    import os
    global _SPUR_JEV_BROKER_FAILED
    _SPUR_JEV_BROKER_FAILED = True
    if _SPUR_JEV_BROKER is not None:
        _SPUR_JEV_BROKER.socket.close()
    descriptor = os.environ.pop("SPUR_JEV_BROKER_FD", None)
    if descriptor is not None:
        try:
            os.close(int(descriptor))
        except (ValueError, OSError):
            pass


class _SpurJevSession:
    def __init__(self, db):
        self.db = db
        self.token = None
        self.previous = None

    def __enter__(self):
        import os
        _spur_duckdb_baseline()
        runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
        connection = _spur_managed_connection(self.db)
        if runtime is None or runtime.db is not connection:
            return self
        entry = runtime._entry("jev")
        if not (entry["desired_enabled"] and entry["loaded"] and entry["signature_verified"]
                and "jev" in runtime.loaded):
            return self
        try:
            channel = _spur_jev_channel()
            if channel is None or not channel.available():
                entry["restart_required"] = True
                entry["activation_transport_restart"] = True
                entry["reason"] = "restart_required"
                return self
        except Exception:
            # A dead/revoked channel cannot renew in-place. Do not block ordinary
            # SQL, read a legacy file or install/load anything as a fallback.
            entry["restart_required"] = True
            entry["activation_transport_restart"] = True
            entry["reason"] = "restart_required"
            return self
        endpoint = self.db.sql("SELECT current_setting('jev_api_url')").fetchone()[0]
        if endpoint != "https://api.typesafe.ai/v1/systemone":
            raise RuntimeError("Managed Jev credentials require the TypeSafe HTTPS endpoint. Reset jev_api_url before running this cell.")
        self.db.sql("RESET jev_api_key")
        # Resolve only after installation and exact endpoint validation. The
        # broker rechecks lease, enablement and generation on every request.
        try:
            token = channel.request()
        except Exception:
            entry["restart_required"] = True
            entry["reason"] = "restart_required"
            return self
        if token is None:  # disabled while setup was in flight
            return self
        self.previous = os.environ.get("TYPESAFE_API_KEY")
        self.token = token
        os.environ["TYPESAFE_API_KEY"] = token
        return self

    def __exit__(self, error_type, error, traceback):
        import os
        import traceback as traceback_module

        if self.token is None:
            return False
        token = self.token
        self.token = None
        if self.previous is None:
            os.environ.pop("TYPESAFE_API_KEY", None)
        else:
            os.environ["TYPESAFE_API_KEY"] = self.previous
        self.previous = None
        # A remote API can echo a header in an error. Suppress the original
        # traceback chain and remove the exact managed token from the diagnostic.
        if error is not None and token in "".join(traceback_module.format_exception(error_type, error, traceback)):
            message = str(error).replace(token, "[redacted]")
            token = None
            error = None
            traceback = None
            raise RuntimeError(message) from None
        return False

def _spur_restore_catalog_extensions(connection, requirements):
    """Restore optional membership without making unrelated SQL depend on its success."""
    import sys
    failures = globals().setdefault("_SPUR_CATALOG_EXTENSION_FAILURES", {})
    errors = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        key = (id(connection), name, source)
        if _spur_catalog_extensions_ready(connection, [item]):
            failures.pop(key, None)
            continue
        previous = failures.get(key)
        if previous and previous[0] is connection:
            errors[name] = previous[1]
            continue
        try:
            _spur_prepare_catalog_extensions(connection, [item])
        except Exception as error:
            errors[name] = str(error)
            failures[key] = (connection, str(error))
            print(f"SPUR: optional extension {name} could not be restored: {error}. Retry or remove it in Extensions.", file=sys.stderr)
    return errors


def _spur_catalog_extensions_ready(connection, requirements):
    """Read current connection identity and proofs without installing or loading code."""
    import re
    rows = connection.sql(
        "SELECT extension_name, installed, loaded, install_mode, installed_from, "
        "install_path, aliases FROM duckdb_extensions()"
    ).fetchall()
    inventory = {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}
    proofs = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            return False
        row = inventory.get(name)
        if not row or not row[2]:
            return False
        if row[3] == "STATICALLY_LINKED" and source == "core":
            continue
        proof = proofs.get((id(connection), row[0]))
        if not proof or proof[0] is not connection or proof[1] != source:
            return False
    return True


def _spur_prepare_connection_extensions(statements):
    """Use the same verifier for host-generated connection INSTALL/LOAD pairs."""
    import re
    requirements, loads = {}, set()
    for statement in statements:
        install = re.fullmatch(r"INSTALL ([a-z][a-z0-9_]*)(?: FROM (core|community))?", statement)
        load = re.fullmatch(r"LOAD ([a-z][a-z0-9_]*)", statement)
        if install:
            name, source = install.group(1), install.group(2) or "core"
            if name in requirements and requirements[name] != source:
                raise ValueError("Conflicting managed connection repositories")
            requirements[name] = source
        elif load:
            loads.add(load.group(1))
        else:
            raise ValueError("Invalid managed connection extension statement")
    if loads != set(requirements):
        raise ValueError("Managed connection extension installation and load must match")
    connection = globals().get("_SPUR_DUCKDB_CONNECTION")
    if connection is None:
        connection = duckdb.default_connection() if hasattr(duckdb, "default_connection") else duckdb
    _spur_prepare_catalog_extensions(connection, [
        {"packageName": name, "source": source} for name, source in requirements.items()
    ])


def _spur_prepare_catalog_extensions(connection, requirements):
    """Prepare signed packages on the notebook's shared DuckDB connection."""
    import duckdb as engine
    import hashlib
    from pathlib import Path
    import re
    import shutil
    import tempfile

    checked = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            raise ValueError("Invalid managed DuckDB extension identity")
        if name in checked and checked[name] != source:
            raise ValueError("One package cannot use two repositories in the same runtime")
        checked[name] = source
        # Explicit preparation is an intentional retry; ordinary SQL uses restore's memo.
        globals().get("_SPUR_CATALOG_EXTENSION_FAILURES", {}).pop((id(connection), name, source), None)

    def inventory(db):
        rows = db.sql(
            "SELECT extension_name, installed, loaded, install_mode, installed_from, "
            "install_path, aliases FROM duckdb_extensions()"
        ).fetchall()
        return {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}

    def matches(row, source):
        origin = row[4] or ""
        repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
        return origin == source or origin.rstrip("/") in (
            "http://" + repository, "https://" + repository)

    # Proofs belong to this exact connection, surviving redefinition on SQL replay.
    # Retain private immutable copies for the lifetime of the kernel.
    proofs = globals().setdefault("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for name, source in checked.items():
        row = inventory(connection).get(name)
        if row and row[3] == "STATICALLY_LINKED" and source == "core":
            if not row[2]:
                connection.load_extension(name)
            continue
        proof = proofs.get((id(connection), row[0] if row else name))
        if row and row[2]:
            # The required HTTPFS baseline has an existing signed verifier and
            # frozen copy. Share that exact proof, never a bare loaded flag or
            # repository label. This also supports already-running baseline code.
            if proof is None and row[0] == "httpfs" and source == "core":
                runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
                if runtime is not None and runtime.db is connection and "httpfs" in runtime.loaded:
                    baseline = runtime._entry("httpfs")
                    private = runtime.verified.get("httpfs")
                    if baseline.get("signature_verified") is True and baseline.get("loaded") is True and private is not None:
                        try:
                            digest = hashlib.sha256((Path(private.name) / "httpfs.duckdb_extension").read_bytes()).digest()
                            if digest.hex() == baseline.get("artifact_digest"):
                                proof = (connection, source, private, digest)
                                proofs[(id(connection), row[0])] = proof
                        except OSError:
                            pass
            if proof and proof[0] is connection and proof[1] == source:
                continue
            raise RuntimeError(f"{name} is already loaded without this setup's verified artifact; restart the kernel and retry Install in notebook")

        # The shared connection permits the app's bundled unsigned gateway.
        # Always verify third-party bytes in an isolated engine with signing ON.
        config = {
            "allow_unsigned_extensions": "false",
            "allow_extensions_metadata_mismatch": "false",
            "allow_community_extensions": "true" if source == "community" else "false",
            "autoload_known_extensions": "false",
            "autoinstall_known_extensions": "false",
            "home_directory": connection.sql("SELECT current_setting('home_directory')").fetchone()[0],
        }
        directory = connection.sql("SELECT current_setting('extension_directory')").fetchone()[0]
        if directory:
            config["extension_directory"] = directory
        private = None
        try:
            with engine.connect(config=config) as verifier:
                for query in ("SELECT version()", "PRAGMA platform"):
                    if verifier.sql(query).fetchone() != connection.sql(query).fetchone():
                        raise RuntimeError("DuckDB runtime changed during extension verification")
                artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
                    verifier.install_extension(name, repository_url="http://" + repository)
                    artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    raise RuntimeError(f"{name} installation is unavailable for this DuckDB runtime")
                if artifact[3] != "REPOSITORY" or not matches(artifact, source):
                    raise RuntimeError(f"{name} is cached from another repository; resolve its source before retrying")
                private = tempfile.TemporaryDirectory(prefix="spur-duckdb-catalog-")
                path = Path(private.name) / (artifact[0] + ".duckdb_extension")
                shutil.copyfile(artifact[5], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).digest()
                verifier.load_extension(str(path))  # signature and ABI checked by DuckDB
                if hashlib.sha256(path.read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
            snapshot = globals().get("_spur_snapshot_function_registration")
            before = snapshot(connection) if callable(snapshot) else None
            connection.load_extension(str(path))  # load exactly the verified bytes
            observed = inventory(connection).get(name)
            if not observed or not observed[2]:
                raise RuntimeError(f"{name} did not load into the notebook connection")
            proofs[(id(connection), observed[0])] = (connection, source, private, digest)
            record = globals().get("_spur_record_function_ownership")
            if callable(record):
                record(connection, observed[0], before)
            private = None
        finally:
            if private is not None:
                private.cleanup()


_spur_duckdb_baseline()
def _spur_restore_catalog_extensions(connection, requirements):
    """Restore optional membership without making unrelated SQL depend on its success."""
    import sys
    failures = globals().setdefault("_SPUR_CATALOG_EXTENSION_FAILURES", {})
    errors = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        key = (id(connection), name, source)
        if _spur_catalog_extensions_ready(connection, [item]):
            failures.pop(key, None)
            continue
        previous = failures.get(key)
        if previous and previous[0] is connection:
            errors[name] = previous[1]
            continue
        try:
            _spur_prepare_catalog_extensions(connection, [item])
        except Exception as error:
            errors[name] = str(error)
            failures[key] = (connection, str(error))
            print(f"SPUR: optional extension {name} could not be restored: {error}. Retry or remove it in Extensions.", file=sys.stderr)
    return errors


def _spur_catalog_extensions_ready(connection, requirements):
    """Read current connection identity and proofs without installing or loading code."""
    import re
    rows = connection.sql(
        "SELECT extension_name, installed, loaded, install_mode, installed_from, "
        "install_path, aliases FROM duckdb_extensions()"
    ).fetchall()
    inventory = {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}
    proofs = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            return False
        row = inventory.get(name)
        if not row or not row[2]:
            return False
        if row[3] == "STATICALLY_LINKED" and source == "core":
            continue
        proof = proofs.get((id(connection), row[0]))
        if not proof or proof[0] is not connection or proof[1] != source:
            return False
    return True


def _spur_prepare_connection_extensions(statements):
    """Use the same verifier for host-generated connection INSTALL/LOAD pairs."""
    import re
    requirements, loads = {}, set()
    for statement in statements:
        install = re.fullmatch(r"INSTALL ([a-z][a-z0-9_]*)(?: FROM (core|community))?", statement)
        load = re.fullmatch(r"LOAD ([a-z][a-z0-9_]*)", statement)
        if install:
            name, source = install.group(1), install.group(2) or "core"
            if name in requirements and requirements[name] != source:
                raise ValueError("Conflicting managed connection repositories")
            requirements[name] = source
        elif load:
            loads.add(load.group(1))
        else:
            raise ValueError("Invalid managed connection extension statement")
    if loads != set(requirements):
        raise ValueError("Managed connection extension installation and load must match")
    connection = globals().get("_SPUR_DUCKDB_CONNECTION")
    if connection is None:
        connection = duckdb.default_connection() if hasattr(duckdb, "default_connection") else duckdb
    _spur_prepare_catalog_extensions(connection, [
        {"packageName": name, "source": source} for name, source in requirements.items()
    ])


def _spur_prepare_catalog_extensions(connection, requirements):
    """Prepare signed packages on the notebook's shared DuckDB connection."""
    import duckdb as engine
    import hashlib
    from pathlib import Path
    import re
    import shutil
    import tempfile

    checked = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            raise ValueError("Invalid managed DuckDB extension identity")
        if name in checked and checked[name] != source:
            raise ValueError("One package cannot use two repositories in the same runtime")
        checked[name] = source
        # Explicit preparation is an intentional retry; ordinary SQL uses restore's memo.
        globals().get("_SPUR_CATALOG_EXTENSION_FAILURES", {}).pop((id(connection), name, source), None)

    def inventory(db):
        rows = db.sql(
            "SELECT extension_name, installed, loaded, install_mode, installed_from, "
            "install_path, aliases FROM duckdb_extensions()"
        ).fetchall()
        return {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}

    def matches(row, source):
        origin = row[4] or ""
        repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
        return origin == source or origin.rstrip("/") in (
            "http://" + repository, "https://" + repository)

    # Proofs belong to this exact connection, surviving redefinition on SQL replay.
    # Retain private immutable copies for the lifetime of the kernel.
    proofs = globals().setdefault("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for name, source in checked.items():
        row = inventory(connection).get(name)
        if row and row[3] == "STATICALLY_LINKED" and source == "core":
            if not row[2]:
                connection.load_extension(name)
            continue
        proof = proofs.get((id(connection), row[0] if row else name))
        if row and row[2]:
            # The required HTTPFS baseline has an existing signed verifier and
            # frozen copy. Share that exact proof, never a bare loaded flag or
            # repository label. This also supports already-running baseline code.
            if proof is None and row[0] == "httpfs" and source == "core":
                runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
                if runtime is not None and runtime.db is connection and "httpfs" in runtime.loaded:
                    baseline = runtime._entry("httpfs")
                    private = runtime.verified.get("httpfs")
                    if baseline.get("signature_verified") is True and baseline.get("loaded") is True and private is not None:
                        try:
                            digest = hashlib.sha256((Path(private.name) / "httpfs.duckdb_extension").read_bytes()).digest()
                            if digest.hex() == baseline.get("artifact_digest"):
                                proof = (connection, source, private, digest)
                                proofs[(id(connection), row[0])] = proof
                        except OSError:
                            pass
            if proof and proof[0] is connection and proof[1] == source:
                continue
            raise RuntimeError(f"{name} is already loaded without this setup's verified artifact; restart the kernel and retry Install in notebook")

        # The shared connection permits the app's bundled unsigned gateway.
        # Always verify third-party bytes in an isolated engine with signing ON.
        config = {
            "allow_unsigned_extensions": "false",
            "allow_extensions_metadata_mismatch": "false",
            "allow_community_extensions": "true" if source == "community" else "false",
            "autoload_known_extensions": "false",
            "autoinstall_known_extensions": "false",
            "home_directory": connection.sql("SELECT current_setting('home_directory')").fetchone()[0],
        }
        directory = connection.sql("SELECT current_setting('extension_directory')").fetchone()[0]
        if directory:
            config["extension_directory"] = directory
        private = None
        try:
            with engine.connect(config=config) as verifier:
                for query in ("SELECT version()", "PRAGMA platform"):
                    if verifier.sql(query).fetchone() != connection.sql(query).fetchone():
                        raise RuntimeError("DuckDB runtime changed during extension verification")
                artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
                    verifier.install_extension(name, repository_url="http://" + repository)
                    artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    raise RuntimeError(f"{name} installation is unavailable for this DuckDB runtime")
                if artifact[3] != "REPOSITORY" or not matches(artifact, source):
                    raise RuntimeError(f"{name} is cached from another repository; resolve its source before retrying")
                private = tempfile.TemporaryDirectory(prefix="spur-duckdb-catalog-")
                path = Path(private.name) / (artifact[0] + ".duckdb_extension")
                shutil.copyfile(artifact[5], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).digest()
                verifier.load_extension(str(path))  # signature and ABI checked by DuckDB
                if hashlib.sha256(path.read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
            snapshot = globals().get("_spur_snapshot_function_registration")
            before = snapshot(connection) if callable(snapshot) else None
            connection.load_extension(str(path))  # load exactly the verified bytes
            observed = inventory(connection).get(name)
            if not observed or not observed[2]:
                raise RuntimeError(f"{name} did not load into the notebook connection")
            proofs[(id(connection), observed[0])] = (connection, source, private, digest)
            record = globals().get("_spur_record_function_ownership")
            if callable(record):
                record(connection, observed[0], before)
            private = None
        finally:
            if private is not None:
                private.cleanup()

import json as _spur_extension_json
_spur_prepare_catalog_extensions(_SPUR_DUCKDB_CONNECTION, _spur_extension_json.loads("[{\"packageName\":\"duckrouting\",\"source\":\"community\"},{\"packageName\":\"shellfs\",\"source\":\"community\"}]"))


# Panic-path exploration: Spur graph + shellfs

This notebook explores [Finding panics with Call Graph Analyzer](https://tweedegolf.nl/en/blog/242/finding-panics-with-call-graph-analyzer/) using the Spur source graph. **This is a source-level experiment, not CGA's MIR analysis or a panic-freedom proof.**

Run the cells from top to bottom. The managed setup cell restores the signed `shellfs` and `duckrouting` extensions. The remaining cells require `rg` in the notebook kernel's PATH and the local repository/analyst paths shown in the SQL. Source scans are live; the graph database is attached read-only.

- `shellfs` streams a fixed, read-only `rg --json` command into DuckDB.
- The analyst index supplies candidate call edges and symbol ranges. The index used for this run reported indexed commit `46b85c5` while the working checkout was `ea25f4f`. Refresh the graph before drawing current coverage conclusions; restart the kernel or detach the existing read-only attachment to pick up a refreshed database.
- `duckrouting` computes directed paths over recorded source-level call edges, including heuristic dynamic edges. No recorded path means only that this graph did not find one.
- Text matches include tests and comments; unresolved or heuristic edges are not compiler-verified. The candidate query includes only `.expect` and `.unwrap` labels; it does not cover implicit checks, dependency internals, or all panic syntax.


In [4]:
SELECT extension_name, installed, loaded, install_mode FROM duckdb_extensions() WHERE extension_name IN ('shellfs', 'duckrouting') ORDER BY extension_name;

extension_name,installed,loaded,install_mode
duckrouting,True,True,REPOSITORY
shellfs,True,True,REPOSITORY


In [5]:
SELECT * FROM read_csv($$printf 'name,score\nprobe,7\n' |$$, header = true);

name,score
probe,7


In [7]:
ATTACH IF NOT EXISTS '/Users/kevintruong/project/SpurCode/.spur/analyst.duckdb' AS spur_graph (READ_ONLY); SELECT graph_content_hash, graph_index_version, extractor_version, complete FROM spur_graph._meta;

graph_content_hash,graph_index_version,extractor_version,complete
8599245794571718b3153894d70d281c091e04ef3300eff4d66c0f3eb9fe6d59,4,2026-08-30-html-css-extraction-v1,True


In [8]:
SELECT * FROM read_csv($$command -v rg || printf 'RG_NOT_IN_PATH\n' |$$, columns = {'path': 'VARCHAR'}, header = false);

path
/opt/homebrew/bin/rg


In [9]:
SELECT table_name, column_name, data_type FROM information_schema.columns WHERE table_catalog = 'spur_graph' AND table_name IN ('nodes','edges','edges_unresolved') ORDER BY table_name, ordinal_position;

table_name,column_name,data_type
edges,source_stable_id,VARCHAR
edges,target_stable_id,VARCHAR
edges,src_id,BIGINT
edges,dst_id,BIGINT
edges,target_label,VARCHAR
edges,relation,VARCHAR
edges,confidence,VARCHAR
edges,confidence_score,FLOAT
edges,edge_kind,VARCHAR
edges,bind_method,VARCHAR


In [10]:
SELECT node_id, stable_symbol_id, qualified_name, file_path, line_start FROM spur_graph.nodes WHERE stable_symbol_id IN ('63f6898cbb3fcfdd', '5fb06d53a2477ff7', '10dad659a68a8b0f') ORDER BY file_path, line_start;

node_id,stable_symbol_id,qualified_name,file_path,line_start
36667,63f6898cbb3fcfdd,build_with_temporal_jobs_override,crates/spur-cli/src/commands/graph.rs,108
35096,5fb06d53a2477ff7,run_full_walk_into,crates/spur-graph/src/git_walk.rs,910
6133,10dad659a68a8b0f,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,920


In [25]:
CREATE OR REPLACE TEMP TABLE panic_source_hits AS
SELECT
  replace(json_extract_string(data, '$.path.text'), '/Users/kevintruong/project/SpurCode/', '') AS file_path,
  try_cast(json_extract_string(data, '$.line_number') AS INTEGER) AS line_number,
  trim(json_extract_string(data, '$.lines.text')) AS source_line
FROM read_json(
  $$rg --json --glob '*.rs' -e '\.(unwrap|expect)\s*\(' -e '\b(panic|unreachable|todo|unimplemented)!\s*[\(\{\[]' /Users/kevintruong/project/SpurCode/crates/spur-graph/src |$$,
  format = 'newline_delimited',
  columns = {'type': 'VARCHAR', 'data': 'JSON'}
)
WHERE type = 'match';

SELECT count(*) AS matching_lines_including_tests_and_comments,
       count(DISTINCT file_path) AS files
FROM panic_source_hits;

matching_lines_including_tests_and_comments,files
1818,39


In [13]:
-- Node IDs are derived from the current graph snapshot; they can change on rebuild.
SET VARIABLE route_start = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd');
SET VARIABLE route_target = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = '10dad659a68a8b0f');

SELECT path.seq, path.node, n.qualified_name, n.file_path, path.agg_cost AS hops
FROM duckrouting_dijkstra(
  $$SELECT row_number() OVER ()::BIGINT AS id,
           src_id AS source, dst_id AS target,
           1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
    FROM spur_graph.edges
    WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')$$,
  getvariable('route_start'), getvariable('route_target')
) AS path
JOIN spur_graph.nodes n ON n.node_id = path.node
ORDER BY path.seq;

seq,node,qualified_name,file_path,hops
1,36667,build_with_temporal_jobs_override,crates/spur-cli/src/commands/graph.rs,0.0
2,35096,run_full_walk_into,crates/spur-graph/src/git_walk.rs,1.0
3,6133,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,2.0


In [14]:
SELECT DISTINCT n.qualified_name, h.file_path, h.line_number, e.target_label,
       e.confidence AS graph_confidence, h.source_line
FROM panic_source_hits h
JOIN spur_graph.nodes n ON n.file_path = h.file_path
  AND h.line_number BETWEEN n.line_start AND n.line_end
JOIN spur_graph.edges_unresolved e ON e.source_stable_id = n.stable_symbol_id
WHERE n.stable_symbol_id = '10dad659a68a8b0f'
  AND e.target_label IN ('unwrap', 'expect')
  AND regexp_matches(h.source_line, '\.' || e.target_label || '\s*\(')
ORDER BY h.line_number;

qualified_name,file_path,line_number,target_label,graph_confidence,source_line
run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,958,expect,syntax_exact,"let base = base.expect(""incremental base checked above"");\n"


In [16]:
-- These are candidates, including possible test/comment matches, not proven panics.
SET VARIABLE route_start = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd');

WITH reachable AS (
  SELECT node, agg_cost AS hops
  FROM duckrouting_driving_distance(
    $$SELECT row_number() OVER ()::BIGINT AS id,
             src_id AS source, dst_id AS target,
             1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
      FROM spur_graph.edges
      WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')$$,
    getvariable('route_start'), 3.0
  )
), candidates AS (
  SELECT r.hops, n.qualified_name, n.file_path, h.line_number, e.target_label,
         h.source_line,
         row_number() OVER (
           PARTITION BY h.file_path, h.line_number, e.target_label
           ORDER BY n.line_end - n.line_start
         ) AS smallest_symbol
  FROM reachable r
  JOIN spur_graph.nodes n ON n.node_id = r.node
  JOIN spur_graph.edges_unresolved e ON e.source_stable_id = n.stable_symbol_id
  JOIN panic_source_hits h ON h.file_path = n.file_path
       AND h.line_number BETWEEN n.line_start AND n.line_end
       AND regexp_matches(h.source_line, '\.' || e.target_label || '\s*\(')
  WHERE n.file_path LIKE 'crates/spur-graph/src/%'
    AND e.target_label IN ('unwrap', 'expect')
)
SELECT hops, qualified_name, file_path, line_number, target_label, source_line
FROM candidates
WHERE smallest_symbol = 1
ORDER BY hops, file_path, line_number
LIMIT 30;

hops,qualified_name,file_path,line_number,target_label,source_line
2.0,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,958,expect,"let base = base.expect(""incremental base checked above"");\n"
2.0,write_edges,crates/spur-graph/src/store/parquet.rs,1720,expect,".expect(""resolved edge has target"")\n"
3.0,rebuild_from_buckets,crates/spur-graph/src/store/build.rs,2530,expect,".expect(""exhausted synthetic file NodeId space"");\n"
3.0,stable_file_id_from_path,crates/spur-graph/src/store/build.rs,2671,unwrap,u64::from_be_bytes(hasher.finalize()[..8].try_into().unwrap())\n


In [17]:
SET VARIABLE route_start = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd');
SET VARIABLE route_target = (SELECT node_id FROM spur_graph.nodes WHERE stable_symbol_id = 'f7e5a680e7069785');
SELECT path.seq, n.qualified_name, n.file_path, path.agg_cost AS hops
FROM duckrouting_dijkstra(
  $$SELECT row_number() OVER ()::BIGINT AS id,
           src_id AS source, dst_id AS target,
           1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
    FROM spur_graph.edges
    WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')$$,
  getvariable('route_start'), getvariable('route_target')
) AS path
JOIN spur_graph.nodes n ON n.node_id = path.node
ORDER BY path.seq;

seq,qualified_name,file_path,hops
1,build_with_temporal_jobs_override,crates/spur-cli/src/commands/graph.rs,0.0
2,artifact_from_facts_incremental,crates/spur-graph/src/store/build.rs,1.0
3,compose_artifact,crates/spur-graph/src/store/build.rs,2.0
4,rebuild_from_buckets,crates/spur-graph/src/store/build.rs,3.0


In [18]:
-- The solver's witness, evaluated with wide SQL integers.
-- It says nothing about whether such an input can actually enter this function.
WITH model AS (
  SELECT CAST('18446744073709551614' AS HUGEINT) AS max_existing,
         true AS missing_file_id
)
SELECT max_existing, missing_file_id, max_existing + 1 AS next_synthetic,
       try_cast(max_existing + 2 AS UBIGINT) IS NULL AS next_checked_add_would_fail
FROM model;

max_existing,missing_file_id,next_synthetic,next_checked_add_would_fail
18446744073709551614,True,18446744073709551615,True


In [19]:
-- Exhaust the Boolean abstraction used by the previous solver check.
WITH inputs(fast_forward, base_present) AS (
  VALUES (false, false), (false, true), (true, false), (true, true)
), model AS (
  SELECT *, fast_forward AND base_present AS at_expect FROM inputs
)
SELECT count(*) AS assignments,
       count(*) FILTER (WHERE at_expect) AS assignments_reaching_expect,
       count(*) FILTER (WHERE at_expect AND NOT base_present) AS assignments_that_panic
FROM model;

assignments,assignments_reaching_expect,assignments_that_panic
4,1,0


## Interpretation and solver checks

The three-hop candidate query above identified four matching lines on the indexed call graph in this run. Those are **candidates**, not four demonstrated panics. The graph has unresolved call labels and source spans belong to symbols, so matching a current line to an older indexed function is a heuristic.

- `git_walk.rs:958`: at the `expect`, `use_incremental_base = fast_forward AND base_present`. A preflighted `solve_constraints` query found the branch satisfiable, and the branch plus `NOT base_present` **unsatisfiable**. The Boolean enumeration below independently evaluates all four input combinations. This result covers that manual abstraction, not other calls or paths.
- `store/build.rs:2530`: the 64-bit solver query returned **sat** for `max_existing = 0xfffffffffffffffe` and a file with no node ID. The next synthetic ID is `0xffffffffffffffff`, so incrementing it fails. The SQL witness below evaluates this arithmetic. This is a model-level boundary case; the experiment has not shown that a normal or accepted artifact can produce it.
- `store/parquet.rs:1720`: `edge_rows` creates resolved rows from `Some(target)`; proving every possible path into `write_edges` preserves that invariant needs a separate review.
- `store/build.rs:2671`: the source slices eight bytes from a SHA-256 digest before conversion. This suggests an invariant-based triage, but it was not separately solver-verified here.

Solver model for the 64-bit case (all constraints hard): `max_existing != all_ones`, `next_synthetic == bv_add(max_existing, 1)`, `missing_file_id == true`, and `next_synthetic == all_ones`. The typed request passed `solve_constraint_check` before `solve_constraints` returned the model. The solver operated separately through MCP; the SQL cell evaluates its recorded model rather than invoking Z3.

Useful next steps are a fresh graph rebuild, compiler-backed CGA output for the chosen binary, and a targeted test or invariant check for the boundary case. CGA and the source graph have different completeness and false-positive limits.


## Continuation: isolated current-worktree graph

The shared analyst index still served the earlier `859924...` graph. To continue without replacing that index, this section uses a separate graph built from the current, dirty worktree with `spur graph build --no-analyst --no-section-embeddings --no-code-symbol-embeddings`. Its content hash is `0855a502...`; its manifest has `indexed_commit_oid = null` because it is a filesystem snapshot, not a clean commit. The files are local under `.spur/graph/experiments/` and are not tracked in Git.

The source-OID check below verifies the specific files used by the findings. It does not certify every file in the snapshot or eliminate ambiguous, heuristic, unresolved, or conditional edges. Source-graph reachability remains a candidate, not proof of an execution path.


In [21]:
DESCRIBE SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/nodes.parquet');

column_name,column_type,null,key,default,extra
stable_symbol_id,VARCHAR,YES,NaN,NaN,NaN
node_id,BIGINT,YES,NaN,NaN,NaN
file_path,VARCHAR,YES,NaN,NaN,NaN
byte_range_start,BIGINT,YES,NaN,NaN,NaN
byte_range_end,BIGINT,YES,NaN,NaN,NaN
line_start,INTEGER,YES,NaN,NaN,NaN
line_end,INTEGER,YES,NaN,NaN,NaN
entity_name,VARCHAR,YES,NaN,NaN,NaN
qualified_name,VARCHAR,YES,NaN,NaN,NaN
symbol_kind,VARCHAR,YES,NaN,NaN,NaN


In [6]:
CREATE OR REPLACE TEMP VIEW current_nodes AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/nodes.parquet');
CREATE OR REPLACE TEMP VIEW current_edges AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/edges.parquet');
CREATE OR REPLACE TEMP VIEW current_unresolved AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/edges_unresolved.parquet');
CREATE OR REPLACE TEMP VIEW current_manifests AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/file_manifests.parquet');
SELECT table_name, column_name, data_type FROM information_schema.columns
WHERE table_name IN ('current_nodes', 'current_edges', 'current_unresolved', 'current_manifests')
ORDER BY table_name, ordinal_position;

table_name,column_name,data_type
current_edges,source_stable_id,VARCHAR
current_edges,target_stable_id,VARCHAR
current_edges,src_id,BIGINT
current_edges,dst_id,BIGINT
current_edges,target_label,VARCHAR
current_edges,relation,VARCHAR
current_edges,confidence,VARCHAR
current_edges,confidence_score,FLOAT
current_edges,edge_kind,VARCHAR
current_edges,bind_method,VARCHAR


In [23]:
SELECT column_name, data_type FROM information_schema.columns WHERE table_name = 'current_manifests' ORDER BY ordinal_position;

column_name,data_type
stable_file_id,VARCHAR
path,VARCHAR
content_oid,VARCHAR
node_ids,BIGINT[]


In [34]:
-- Verify the six source files used by the sampled findings.
WITH live AS (
  SELECT * FROM read_csv(
    $$for f in crates/spur-cli/src/commands/graph.rs crates/spur-graph/src/git_walk.rs crates/spur-graph/src/store/build.rs crates/spur-graph/src/store/parquet.rs crates/spur-graph/src/validation.rs crates/spur-graph/src/extract/tree_sitter.rs; do printf '%s,' "$f"; git -C /Users/kevintruong/project/SpurCode hash-object --no-filters "$f"; done |$$,
    header = false, columns = {'path': 'VARCHAR', 'live_oid': 'VARCHAR'}
  )
)
SELECT live.path, live.live_oid, m.content_oid AS indexed_oid,
       live.live_oid = m.content_oid AS content_matches
FROM live
LEFT JOIN current_manifests m ON m.path = live.path
ORDER BY live.path;

path,live_oid,indexed_oid,content_matches
crates/spur-cli/src/commands/graph.rs,6e4f6ea108705455894cd320e0f045450c3a5042,6e4f6ea108705455894cd320e0f045450c3a5042,True
crates/spur-graph/src/extract/tree_sitter.rs,055dcf989b46a5de12e54774dcfbaf8a147fe1aa,055dcf989b46a5de12e54774dcfbaf8a147fe1aa,True
crates/spur-graph/src/git_walk.rs,aa96dae0a8e806b8fc3bb1adb33a655b98c9b229,aa96dae0a8e806b8fc3bb1adb33a655b98c9b229,True
crates/spur-graph/src/store/build.rs,4df2bddf13df7ac40c1f0e38df9ed7c6790523c8,4df2bddf13df7ac40c1f0e38df9ed7c6790523c8,True
crates/spur-graph/src/store/parquet.rs,d2d7a08b5bf0a491174fa1769f74f34c6008d194,d2d7a08b5bf0a491174fa1769f74f34c6008d194,True
crates/spur-graph/src/validation.rs,3ba906e69ca1b4691070453a6944a72b24a847a2,3ba906e69ca1b4691070453a6944a72b24a847a2,True


In [26]:
CREATE OR REPLACE TEMP TABLE current_call_edges AS
SELECT row_number() OVER ()::BIGINT AS id, src_id AS source, dst_id AS target,
       1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
FROM current_edges
WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')
  AND src_id IS NOT NULL AND dst_id IS NOT NULL;
SELECT count(*) AS directed_edges FROM current_call_edges;

directed_edges
40345


In [7]:
SET VARIABLE current_root = (
  SELECT node_id FROM current_nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd'
);
-- The routing extension prepares its query on a separate connection, so it
-- reads the Parquet artifact directly instead of using a temporary table.
CREATE OR REPLACE TEMP TABLE current_reachable AS
SELECT node, agg_cost AS hops
FROM duckrouting_driving_distance(
  $$SELECT row_number() OVER ()::BIGINT AS id,
           src_id AS source, dst_id AS target,
           1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
    FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/edges.parquet')
    WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')
      AND src_id IS NOT NULL AND dst_id IS NOT NULL$$,
  getvariable('current_root'), 5.0
);
SELECT hops, count(*) AS reachable_symbols
FROM current_reachable
GROUP BY hops ORDER BY hops;

hops,reachable_symbols
0.0,1
1.0,26
2.0,48
3.0,59
4.0,54
5.0,82


In [29]:
CREATE OR REPLACE TEMP TABLE current_candidates AS
WITH matched AS (
  SELECT r.hops, n.stable_symbol_id, n.qualified_name, n.file_path,
         h.line_number, e.target_label, h.source_line,
         row_number() OVER (
           PARTITION BY h.file_path, h.line_number, e.target_label
           ORDER BY n.line_end - n.line_start
         ) AS smallest_symbol
  FROM current_reachable r
  JOIN current_nodes n ON n.node_id = r.node
  JOIN current_unresolved e ON e.source_stable_id = n.stable_symbol_id
  JOIN panic_source_hits h ON h.file_path = n.file_path
       AND h.line_number BETWEEN n.line_start AND n.line_end
       AND regexp_matches(h.source_line, '\.' || e.target_label || '\s*\(')
  WHERE n.file_path LIKE 'crates/spur-graph/src/%'
    AND n.qualified_name NOT LIKE '%tests::%'
    AND e.relation = 'calls'
    AND e.target_label IN ('unwrap', 'expect')
)
SELECT hops, stable_symbol_id, qualified_name, file_path, line_number, target_label, source_line
FROM matched WHERE smallest_symbol = 1;
SELECT hops, qualified_name, file_path, line_number, target_label, source_line
FROM current_candidates ORDER BY hops, file_path, line_number LIMIT 50;

hops,qualified_name,file_path,line_number,target_label,source_line
2.0,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,958,expect,"let base = base.expect(""incremental base checked above"");\n"
2.0,write_edges,crates/spur-graph/src/store/parquet.rs,1720,expect,".expect(""resolved edge has target"")\n"
3.0,rebuild_from_buckets,crates/spur-graph/src/store/build.rs,2530,expect,".expect(""exhausted synthetic file NodeId space"");\n"
3.0,stable_file_id_from_path,crates/spur-graph/src/store/build.rs,2671,unwrap,u64::from_be_bytes(hasher.finalize()[..8].try_into().unwrap())\n
4.0,impl ExternalImportRegistry::write_to_buckets,crates/spur-graph/src/store/build.rs,1460,expect,".expect(""exhausted synthetic external NodeId space"");\n"
4.0,writer_properties,crates/spur-graph/src/store/parquet.rs,2322,expect,".expect(""row-group row count fits u64"");\n"
4.0,compute_anchor_hash,crates/spur-graph/src/validation.rs,113,expect,".expect(""SHA-256 digest always has at least eight bytes""),\n"
5.0,resolve_bare_pending_edge,crates/spur-graph/src/extract/tree_sitter.rs,1076,expect,"fallback_edge.as_ref().expect(""fallback edge"")\n"


In [30]:
-- Compare source-text hits to graph labels. Labels are evidence of syntax,
-- not type-checked targets; a missing label cannot prove absence of a call.
CREATE OR REPLACE TEMP TABLE current_text_candidates AS
WITH matched AS (
  SELECT r.hops, n.stable_symbol_id, n.qualified_name, n.file_path,
         h.line_number, h.source_line,
         CASE
           WHEN regexp_matches(h.source_line, '\.(unwrap|expect)\s*\(')
             THEN regexp_extract(h.source_line, '\.(unwrap|expect)\s*\(', 1)
           WHEN regexp_matches(h.source_line, '\b(panic|unreachable|todo|unimplemented)!\s*[\(\{\[]')
             THEN regexp_extract(h.source_line, '\b(panic|unreachable|todo|unimplemented)!\s*[\(\{\[]', 1)
         END AS sink_label,
         row_number() OVER (
           PARTITION BY h.file_path, h.line_number
           ORDER BY n.line_end - n.line_start
         ) AS smallest_symbol
  FROM current_reachable r
  JOIN current_nodes n ON n.node_id = r.node
  JOIN panic_source_hits h ON h.file_path = n.file_path
       AND h.line_number BETWEEN n.line_start AND n.line_end
  WHERE n.file_path LIKE 'crates/spur-graph/src/%'
    AND n.qualified_name NOT LIKE '%tests::%'
    AND n.symbol_kind IN ('function', 'method')
)
SELECT * EXCLUDE (smallest_symbol) FROM matched
WHERE smallest_symbol = 1 AND sink_label IS NOT NULL;

WITH labels AS (
  SELECT source_stable_id, target_label, 'resolved' AS state FROM current_edges
  WHERE relation = 'calls'
  UNION ALL
  SELECT source_stable_id, target_label, 'unresolved' AS state FROM current_unresolved
  WHERE relation = 'calls'
)
SELECT t.hops, t.qualified_name, t.file_path, t.line_number, t.sink_label,
       coalesce(string_agg(DISTINCT l.state, ', '), 'no matching label') AS graph_label_state,
       t.source_line
FROM current_text_candidates t
LEFT JOIN labels l ON l.source_stable_id = t.stable_symbol_id AND l.target_label = t.sink_label
GROUP BY ALL
ORDER BY t.hops, t.file_path, t.line_number
LIMIT 50;

hops,qualified_name,file_path,line_number,sink_label,graph_label_state,source_line
2.0,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,958,expect,unresolved,"let base = base.expect(""incremental base checked above"");\n"
2.0,write_edges,crates/spur-graph/src/store/parquet.rs,1720,expect,unresolved,".expect(""resolved edge has target"")\n"
3.0,rebuild_from_buckets,crates/spur-graph/src/store/build.rs,2530,expect,unresolved,".expect(""exhausted synthetic file NodeId space"");\n"
3.0,stable_file_id_from_path,crates/spur-graph/src/store/build.rs,2671,unwrap,unresolved,u64::from_be_bytes(hasher.finalize()[..8].try_into().unwrap())\n
4.0,impl ExternalImportRegistry::write_to_buckets,crates/spur-graph/src/store/build.rs,1460,expect,unresolved,".expect(""exhausted synthetic external NodeId space"");\n"
4.0,writer_properties,crates/spur-graph/src/store/parquet.rs,2322,expect,unresolved,".expect(""row-group row count fits u64"");\n"
4.0,compute_anchor_hash,crates/spur-graph/src/validation.rs,113,expect,unresolved,".expect(""SHA-256 digest always has at least eight bytes""),\n"
5.0,resolve_bare_pending_edge,crates/spur-graph/src/extract/tree_sitter.rs,1076,expect,unresolved,"fallback_edge.as_ref().expect(""fallback edge"")\n"
5.0,parse_raw_diff,crates/spur-graph/src/git_walk.rs,3412,unreachable,no matching label,"unreachable!(""kind=Renamed implies status_kind in [R,C]; see lines 1339-1340"")\n"


In [33]:
SET VARIABLE current_root = (SELECT node_id FROM current_nodes WHERE stable_symbol_id = '63f6898cbb3fcfdd');
SET VARIABLE macro_target = (SELECT node_id FROM current_nodes WHERE stable_symbol_id = 'ee814999c7a2927a');
WITH path AS (
  SELECT seq, node, agg_cost,
         lead(node) OVER (ORDER BY seq) AS next_node
  FROM duckrouting_dijkstra(
    $$SELECT row_number() OVER ()::BIGINT AS id, src_id AS source, dst_id AS target,
             1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
      FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/0855a502e1aabe9da17f2888253882bec044a67e8bf5b98e60324450b9e29e3d.parquet/edges.parquet')
      WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')$$,
    getvariable('current_root'), getvariable('macro_target')
  )
)
SELECT p.seq, n.qualified_name, n.file_path, p.agg_cost AS hops,
       string_agg(DISTINCT e.edge_kind || ':' || e.confidence, ', ') AS outgoing_edge_evidence
FROM path p
JOIN current_nodes n ON n.node_id = p.node
LEFT JOIN current_edges e ON e.src_id = p.node AND e.dst_id = p.next_node
  AND e.relation = 'calls' AND e.edge_kind IN ('calls', 'calls_dyn')
GROUP BY p.seq, n.qualified_name, n.file_path, p.agg_cost
ORDER BY p.seq;

seq,qualified_name,file_path,hops,outgoing_edge_evidence
1,build_with_temporal_jobs_override,crates/spur-cli/src/commands/graph.rs,0.0,calls:syntax_exact
2,run_full_walk_into,crates/spur-graph/src/git_walk.rs,1.0,calls:syntax_exact
3,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,2.0,calls:syntax_exact
4,compute_commit_walk_result,crates/spur-graph/src/git_walk.rs,3.0,calls:syntax_exact
5,file_changes_for_commit,crates/spur-graph/src/git_walk.rs,4.0,calls:syntax_exact
6,parse_raw_diff,crates/spur-graph/src/git_walk.rs,5.0,NaN


In [32]:
-- Exhaust all 256 status bytes and both gitlink branches.
-- kind_renamed follows its construction in parse_raw_diff; the panic arm
-- requires renamed after the R/C guard fails.
WITH inputs AS (
  SELECT i AS status_byte, gitlink
  FROM range(0, 256) AS bytes(i)
  CROSS JOIN (VALUES (false), (true)) AS flags(gitlink)
), modeled AS (
  SELECT *,
         status_byte IN (67, 82) AS rename_or_copy,
         NOT gitlink AND status_byte IN (67, 82) AS kind_renamed
  FROM inputs
)
SELECT count(*) AS modeled_assignments,
       count(*) FILTER (WHERE kind_renamed) AS renamed_assignments,
       count(*) FILTER (WHERE kind_renamed AND NOT rename_or_copy) AS unreachable_arm_assignments
FROM modeled;

modeled_assignments,renamed_assignments,unreachable_arm_assignments
512,2,0


In [35]:
WITH outgoing AS (
  SELECT target_label, relation FROM current_edges
  WHERE source_stable_id = 'ee814999c7a2927a'
  UNION ALL
  SELECT target_label, relation FROM current_unresolved
  WHERE source_stable_id = 'ee814999c7a2927a'
)
SELECT count(*) AS total_recorded_outgoing_edges,
       count(*) FILTER (WHERE target_label ILIKE '%unreachable%') AS labels_mentioning_unreachable
FROM outgoing;

total_recorded_outgoing_edges,labels_mentioning_unreachable
25,0


In [36]:
-- Sample the modeled clamp; the solver result below covers the full modeled domain.
WITH inputs(row_count) AS (
  VALUES (0::HUGEINT), (1::HUGEINT), (16384::HUGEINT), (16385::HUGEINT),
         (CAST('18446744073709551615' AS HUGEINT))
), clamped AS (
  SELECT row_count, greatest(1::HUGEINT, least(row_count, 16384::HUGEINT)) AS clamped_count
  FROM inputs
)
SELECT row_count, clamped_count, try_cast(clamped_count AS UBIGINT) IS NOT NULL AS fits_u64
FROM clamped ORDER BY row_count;

row_count,clamped_count,fits_u64
0,1,True
1,1,True
16384,16384,True
16385,16384,True
18446744073709551615,16384,True


### Current-snapshot results

The directed graph reaches **270 indexed symbols including the seed** within five hops. Joining those functions to the live text scan yields eight `.expect` / `.unwrap` candidates and one `unreachable!` candidate. These counts are scoped to this root, radius, source patterns, and graph; they are not a panic count or a completeness claim.

| Candidate | Experiment result |
|---|---|
| `git_walk.rs:958` | Guarded `expect`: the earlier Boolean counterexample query is unsatisfiable under its manually encoded condition. |
| `store/parquet.rs:1720` | `edge_rows` constructs resolved rows only when the target is present; checking all paths that can create or change those rows remains separate work. |
| `store/build.rs:2530` | A 64-bit solver model reaches `checked_add` failure with an existing maximum of `u64::MAX - 1` and a missing file ID. Real artifact reachability is not established. |
| `store/build.rs:2671` | The conversion takes eight bytes from a fixed-width SHA-256 digest; source inspection suggests the unwrap is guarded by that type invariant. |
| `store/build.rs:1460` | A second synthetic-node allocator has the same extreme-value condition, when it must allocate a new external node. Its real artifact reachability is not established. |
| `store/parquet.rs:2322` | A solver model of `row_count.clamp(1, 16384)` found no result outside that interval, for all non-negative modeled integers. Values in the interval fit `u64`. |
| `validation.rs:113` | This also converts eight bytes of a SHA-256 digest; the source indicates the same fixed-length invariant. |
| `extract/tree_sitter.rs:1076` | `fallback_edge` is set to `Some(...)` immediately before the `expect` on this branch, assuming the preceding call returns normally. |
| `git_walk.rs:3412` | The snapshot has 25 outgoing edges for `parse_raw_diff`, but none has a target label containing `unreachable`. Source inspection and the solver check below show that its renamed branch requires status `R` or `C`. |

For the final row, the typed Z3 request represented an 8-bit status and a Boolean gitlink flag. It encoded `kind_renamed = NOT is_gitlink AND status IN {67,82}`; the renamed branch was satisfiable, while adding `status NOT IN {67,82}` was **unsatisfiable**. The SQL cell independently enumerates all 512 combinations and finds zero violating assignments. This establishes the supplied abstraction, not compiler-derived path semantics.

The targeted existing Rust test `parse_raw_diff_handles_copy_status_like_rename` could not be run: both configured AWS builder attempts failed to resolve the required AMI. The wrapper then began a local fallback; it was interrupted in accordance with the repository's remote-build policy. No Rust test pass is claimed.

The fresh graph is a local ignored artifact. To regenerate it, rerun the build command described above, select its new content-hash directory, and update the Parquet paths in this section. Compare source OIDs again before interpreting results.

The Rust edge query at `crates/spur-graph/queries/rust/spur-edges.scm` captures normal call expressions and call-shaped tokens inside macro bodies; it does not currently have a capture for the invoked macro name itself. A prototype can record macro-invocation names and source spans as **heuristic** sink evidence, with fixtures for `panic!`, `unreachable!`, assertions, qualified or shadowed macros, and calls inside macro arguments. This alone would not detect compiler-inserted checks or expand macros.


## Coverage-gap analysis and an effectiveness experiment

The observed gap is specific: the source graph can reach a function that contains a macro invocation, but it does not retain the invoked macro name as source evidence. The experiment below compares current graph output with a prototype AST collector on a controlled fixture set. It measures **macro-invocation capture**, separately from whether an invocation resolves to a standard panic macro or can actually panic.

The prototype and fixtures stay outside the production crate. Measurements from a finite fixture set establish behavior on that set only.

### Recommendation: retain macro-invocation evidence, then classify it

The first change should add a **versioned, per-invocation evidence sidecar** in `spur-graph`. Each row should retain the owner symbol, macro path, exact source span, source OID, and declared execution conditions. Panic-risk queries can join those rows to the existing call graph. This closes the observed source-evidence gap without resolving a bare macro name to an unrelated function symbol.

The notebook prototype provides bounded evidence for this direction:

| Measurement | Current query | Prototype | Meaning |
|---|---:|---:|---|
| Direct invocation names captured in 25 fixtures | 0 / 29 | 29 / 29 | Query-level syntax capture, not panic-path recall |
| False positive invocation sites in 3 negative controls | 0 | 0 | Comments, strings, and unused macro-definition tokens |
| Fixture context-label errors | — | 0 | Only the declared fixture cases |
| Existing ordinary/body-call positive controls | captured | retained as a separate query | The prototype adds evidence; it does not replace those captures |
| Real Rust files parsed / matching snapshot | — | 60 / 60 | Source hashes checked before graph joins |
| Real invocation spans independently validated | — | 4,158 | Byte offsets and line coordinates checked |
| Exact function-owner bindings | — | 4,100 / 4,100 | No ambiguous or unmapped function sites; 58 file-scope sites retained separately |
| Named macro sink candidates in the 270-symbol neighborhood | no matching labels for the six sites | 6 | Five debug assertions and one `unreachable!`, all unresolved bare names |

The corpus scan labels 1,968 of 2,007 named sink candidates as test-context invocations. The remaining 39 contain 11 debug-assertion invocations. These are **prototype context labels**, not compiler-evaluated `cfg` results. This shows that execution context and build settings must be retained alongside capture coverage; otherwise newly visible test/debug sites can overwhelm useful results.

The earlier finite solver check already found no counterexample to the local `unreachable!` invariant. Recovering its source evidence therefore does not turn it into a demonstrated bug. The useful outcome is a complete, reviewable candidate record with a separate semantic disposition.

The evaluation uses the repository's current Rust edge-query bytes (SHA-256 recorded below) and `tree-sitter-rust 0.24.2`. Native Python parsing runs in a subprocess with `py-tree-sitter 0.25.2`, after an initial 0.26.0 corpus run disconnected the notebook kernel. Production Rust extractor integration, full indexing overhead, and compiler-backed panic recall remain unmeasured.

Parser references: [Python binding documentation](https://tree-sitter.github.io/py-tree-sitter/), [official binding releases](https://github.com/tree-sitter/py-tree-sitter/releases), and [Rust grammar 0.24.2](https://pypi.org/project/tree-sitter-rust/0.24.2/).


In [3]:
# Native parsing stays in a child process. The first 0.26.0 corpus run
# disconnected this kernel; the isolated experiment pins 0.25.2 and validates
# byte-derived coordinates. This does not change the Rust crate's dependency.
import subprocess
import sys
from pathlib import Path
_coverage_packages = Path("/private/tmp/spur-panic-coverage-python-0252")
_coverage_packages.mkdir(parents=True, exist_ok=True)
if not (_coverage_packages / "tree_sitter").exists():
    _install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
         "--disable-pip-version-check", "--only-binary=:all:",
         "--index-url", "https://pypi.org/simple", "--target", str(_coverage_packages),
         "tree-sitter==0.25.2", "tree-sitter-rust==0.24.2"],
        text=True, capture_output=True, timeout=60
    )
    print("pip_exit_code:", _install.returncode)
    if _install.returncode:
        print(_install.stderr[-2000:])
        raise RuntimeError("Research dependency installation failed")
print("isolated_package_directory:", _coverage_packages)
print("Native parser modules are not imported into the notebook kernel.")


pip_exit_code: 0
isolated_package_directory: /private/tmp/spur-panic-coverage-python-0252
Native parser modules are not imported into the notebook kernel.


In [18]:
_coverage_fixture_source = r'''
# Expected outputs are declared before the collector.
# Each tuple is (case, Rust source, expected direct macro sites, expected context).
coverage_cases = [
 ("direct", 'fn direct(){ panic!("x"); }', [("direct","panic")], "production"),
 ("qualified", 'fn qualified(){ std::panic!("x"); core::unreachable!(); }',
  [("qualified","std::panic"),("qualified","core::unreachable")], "production"),
 ("delimiters", 'fn delimiters(){ todo!{"x"}; unimplemented!["x"]; }',
  [("delimiters","todo"),("delimiters","unimplemented")], "production"),
 ("assertions", 'fn assertions(){ assert!(false); assert_eq!(1,2); assert_ne!(1,1); }',
  [("assertions","assert"),("assertions","assert_eq"),("assertions","assert_ne")], "production"),
 ("debug", 'fn debug(){ debug_assert!(false); debug_assert_eq!(1,2); debug_assert_ne!(1,1); }',
  [("debug","debug_assert"),("debug","debug_assert_eq"),("debug","debug_assert_ne")], "production"),
 ("raw_name", 'fn raw_name(){ r#panic!("x"); }', [("raw_name","panic")], "production"),
 ("spaced", 'fn spaced(){ std :: panic!("x"); }', [("spaced","std::panic")], "production"),
 ("shadowed", 'macro_rules! panic { () => { 0 } } fn shadowed(){ panic!(); }',
  [("shadowed","panic")], "production"),
 ("custom", 'fn custom(){ app::panic!("x"); }', [("custom","app::panic")], "production"),
 ("alias", 'use std::panic as explode; fn alias(){ explode!("x"); }',
  [("alias","explode")], "production"),
 ("test_module", '#[cfg(test)] mod tests { fn test_module(){ panic!("x"); } }',
  [("test_module","panic")], "test"),
 ("test_function", '#[test] fn test_function(){ panic!("x"); }',
  [("test_function","panic")], "test"),
 ("feature", '#[cfg(feature="demo")] fn feature(){ panic!("x"); }',
  [("feature","panic")], "conditional"),
 ("nested_tokens", 'fn nested_tokens(){ wrapper!(panic!("maybe")); }',
  [("nested_tokens","wrapper")], "production"),
 ("stringify", 'fn stringify(){ stringify!(panic!("inert tokens")); }',
  [("stringify","stringify")], "production"),
 ("nested_function", 'fn outer(){ fn inner(){ panic!("x"); } unreachable!(); }',
  [("inner","panic"),("outer","unreachable")], "production"),
 ("argument_call", 'fn argument_call(){ format!("{}", helper()); }',
  [("argument_call","format")], "production"),
 ("comments_strings", 'fn comments_strings(){ /* panic!("fake"); */ let s = r#"unreachable!()"#; }',
  [], "production"),
 ("definition_only", 'macro_rules! unused_macro { () => { panic!("not expanded") } }',
  [], "production"),
 ("ordinary_call", 'fn ordinary_call(){ helper(); }', [], "production"),
 ("async_test", '#[tokio::test(flavor="current_thread")] async fn async_test(){ panic!("x"); }',
  [("async_test","panic")], "test"),
 ("cfg_any", '#[cfg(any(test, feature="extra"))] fn cfg_any(){ panic!("x"); }',
  [("cfg_any","panic")], "conditional"),
 ("cfg_attr", '#[cfg_attr(feature="demo", test)] fn cfg_attr(){ panic!("x"); }',
  [("cfg_attr","panic")], "conditional"),
 ("commented_path", 'fn commented_path(){ std /* trivia */ :: panic!("x"); }',
  [("commented_path","std::panic")], "production"),
 ("absolute_path", 'fn absolute_path(){ ::core::panic!("x"); }',
  [("absolute_path","::core::panic")], "production"),
]
print("fixture_cases:", len(coverage_cases))
print("expected_direct_invocations:", sum(len(case[2]) for case in coverage_cases))
print("zero-invocation_controls:", sum(not case[2] for case in coverage_cases))
for _case, _source, _expected, _ctx in coverage_cases:
    _tree = _coverage_parser.parse(_source.encode())
    if _tree.root_node.has_error:
        raise AssertionError(f"Fixture has a parse error: {_case}")
print("all_fixture_sources_parse:", True)

'''
print("_coverage_fixture_source: saved for isolated execution")


_coverage_fixture_source: saved for isolated execution


In [19]:
_coverage_collector_source = r'''
import re
import hashlib
from collections import Counter
from bisect import bisect_right
from tree_sitter import Query, QueryCursor

_sink_names = {
    "panic", "unreachable", "todo", "unimplemented",
    "assert", "assert_eq", "assert_ne",
    "debug_assert", "debug_assert_eq", "debug_assert_ne",
}
def _syntax_text(source_bytes, node):
    return source_bytes[node.start_byte:node.end_byte].decode("utf-8")

def _normalize_macro_path(source_bytes, node):
    if node.type == "scoped_identifier":
        path = node.child_by_field_name("path")
        name = node.child_by_field_name("name")
        prefix = _normalize_macro_path(source_bytes, path) if path is not None else ""
        suffix = _normalize_macro_path(source_bytes, name) if name is not None else ""
        return prefix + "::" + suffix
    return _syntax_text(source_bytes, node).removeprefix("r#")

def _preceding_attributes(source_bytes, node):
    attributes = []
    previous = node.prev_named_sibling
    while previous is not None and previous.type == "attribute_item":
        attributes.append(_syntax_text(source_bytes, previous))
        previous = previous.prev_named_sibling
    return attributes

def collect_macro_evidence(source_text, file_path="<fixture>"):
    source_bytes = source_text.encode("utf-8")
    line_starts = [0] + [i + 1 for i, byte in enumerate(source_bytes) if byte == 10]
    tree = _coverage_parser.parse(source_bytes)
    if tree.root_node.has_error:
        raise ValueError(f"Parse error in {file_path}")
    definitions = []
    def find_definitions(node):
        if node.type == "macro_definition":
            name = node.child_by_field_name("name")
            if name is not None:
                definitions.append((node.parent.id, node.end_byte,
                                    _syntax_text(source_bytes, name).removeprefix("r#")))
            return
        for child in node.named_children:
            find_definitions(child)
    find_definitions(tree.root_node)
    evidence = []
    def visit(node, ancestors, attributes):
        # A definition's tokens have no execution caller until macro expansion.
        if node.type == "macro_definition":
            return
        attrs = attributes + _preceding_attributes(source_bytes, node)
        lineage = ancestors + [node]
        if node.type == "macro_invocation":
            macro = node.child_by_field_name("macro")
            if macro is not None:
                macro_path = _normalize_macro_path(source_bytes, macro)
                leaf = macro.child_by_field_name("name") or macro
                caller = next((a for a in reversed(ancestors) if a.type == "function_item"), None)
                caller_name_node = caller.child_by_field_name("name") if caller is not None else None
                caller_name = _syntax_text(source_bytes, caller_name_node) if caller_name_node is not None else "<file>"
                ancestor_ids = {a.id for a in ancestors}
                bare_name = macro_path.split("::")[-1]
                local_shadow = "::" not in macro_path and any(
                    scope in ancestor_ids and end <= node.start_byte and name == bare_name
                    for scope, end, name in definitions
                )
                compact_attrs = [re.sub(r"\s+", "", a) for a in attrs]
                is_test = any(
                    a == "#[cfg(test)]" or re.match(r"#\[(?:(?:tokio|async_std)::)?test(?:\(|\])", a) is not None
                    for a in compact_attrs
                )
                context = "test" if is_test else (
                    "conditional" if any("#[cfg" in a for a in compact_attrs) else "production"
                )
                if local_shadow:
                    classification = "local_macro_definition"
                elif macro_path.removeprefix("::") in {f"{root}::{name}" for root in ("core", "std") for name in _sink_names}:
                    classification = "explicit_standard_path_candidate"
                elif "::" not in macro_path and bare_name in _sink_names:
                    classification = "unresolved_bare_sink_candidate"
                elif bare_name in _sink_names:
                    classification = "other_qualified_macro"
                else:
                    classification = "other_macro_name"
                evidence.append({
                    "file_path": file_path, "caller": caller_name, "macro_path": macro_path,
                    "line": bisect_right(line_starts, node.start_byte),
                    "start_byte": node.start_byte, "end_byte": node.end_byte,
                    "name_start_byte": leaf.start_byte, "name_end_byte": leaf.end_byte,
                    "caller_start_byte": caller.start_byte if caller is not None else None,
                    "caller_end_byte": caller.end_byte if caller is not None else None,
                    "context": context, "attributes": sorted(set(attrs)),
                    "classification": classification,
                    "profile_conditional": bare_name.startswith("debug_assert"),
                    "origin": "macro_invocation_ast",
                })
        for child in node.named_children:
            visit(child, lineage, attrs)
    visit(tree.root_node, [], [])
    return tree, evidence

_query_path = Path("/Users/kevintruong/project/SpurCode/crates/spur-graph/queries/rust/spur-edges.scm")
_query_bytes = _query_path.read_bytes()
_baseline_query = Query(_coverage_language, _query_bytes.decode())
coverage_rows = []
coverage_case_metrics = []
coverage_failures = []
_ordinary_call_captured = False
_argument_call_captured = False
for case, source_text, expected, expected_context in coverage_cases:
    tree, observed = collect_macro_evidence(source_text, case)
    expected_counts = Counter(expected)
    observed_counts = Counter((r["caller"], r["macro_path"]) for r in observed)
    captures = QueryCursor(_baseline_query).captures(tree.root_node)
    baseline_names = captures.get("call.name", []) + captures.get("macro_call.name", [])
    baseline_spans = {(n.start_byte, n.end_byte) for n in baseline_names}
    baseline_detected = Counter(
        (r["caller"], r["macro_path"]) for r in observed
        if (r["name_start_byte"], r["name_end_byte"]) in baseline_spans
    )
    tp = sum((observed_counts & expected_counts).values())
    fp = sum((observed_counts - expected_counts).values())
    fn = sum((expected_counts - observed_counts).values())
    context_errors = sum(r["context"] != expected_context for r in observed)
    coverage_case_metrics.append({
        "case": case, "expected": sum(expected_counts.values()),
        "baseline_tp": sum((baseline_detected & expected_counts).values()),
        "prototype_tp": tp, "prototype_fp": fp, "prototype_fn": fn,
        "context_errors": context_errors,
    })
    if fp or fn or context_errors:
        coverage_failures.append((case, fp, fn, context_errors))
    for row in observed:
        row["case"] = case
    coverage_rows.extend(observed)
    if case == "ordinary_call":
        _ordinary_call_captured = any(_syntax_text(source_text.encode(), n) == "helper" for n in baseline_names)
    if case == "argument_call":
        _argument_call_captured = any(_syntax_text(source_text.encode(), n) == "helper" for n in baseline_names)
    print(case, "expected", sum(expected_counts.values()), "baseline_macro_sites", sum(baseline_detected.values()),
          "prototype_tp", tp, "fp", fp, "fn", fn, "context_errors", context_errors)

print("fixture_failures:", coverage_failures)
print("ordinary_call_preserved_by_baseline:", _ordinary_call_captured)
print("macro_argument_call_preserved_by_baseline:", _argument_call_captured)
print("baseline_query_sha256:", hashlib.sha256(_query_bytes).hexdigest())
assert not coverage_failures
assert _ordinary_call_captured and _argument_call_captured

'''
print("_coverage_collector_source: saved for isolated execution")


_coverage_collector_source: saved for isolated execution


In [20]:
# Run native parsing in an isolated child process and import only validated JSON.
import json
import subprocess
import sys
import tempfile
from pathlib import Path
_repo_root = Path("/Users/kevintruong/project/SpurCode")
_coverage_db = _SPUR_DUCKDB_CONNECTION
_header = ("import sys\nfrom pathlib import Path\n"
           + "sys.path.insert(0, " + repr(str(_coverage_packages)) + ")\n"
           + "import tree_sitter_rust\nfrom tree_sitter import Language, Parser\n"
           + "_coverage_language = Language(tree_sitter_rust.language())\n"
           + "_coverage_parser = Parser(_coverage_language)\n")
_scan_source = "import json\nimport importlib.metadata\n_repo = Path(\"/Users/kevintruong/project/SpurCode\")\n_live_rows, _file_rows, _parse_errors = [], [], []\nfor _file in sorted((_repo / \"crates/spur-graph/src\").rglob(\"*.rs\")):\n    _relative = _file.relative_to(_repo).as_posix()\n    _bytes = _file.read_bytes()\n    _oid = hashlib.sha1(b\"blob \" + str(len(_bytes)).encode() + b\"\\0\" + _bytes).hexdigest()\n    _file_rows.append({\"file_path\": _relative, \"source_oid\": _oid})\n    try:\n        _tree, _rows = collect_macro_evidence(_bytes.decode(\"utf-8\"), _relative)\n    except (ValueError, UnicodeDecodeError) as _error:\n        _parse_errors.append((_relative, str(_error)))\n        continue\n    for _row in _rows:\n        assert 0 <= _row[\"start_byte\"] < _row[\"end_byte\"] <= len(_bytes)\n        assert _row[\"line\"] == _bytes[:_row[\"start_byte\"]].count(bytes([10])) + 1\n        _row[\"source_oid\"] = _oid\n    _live_rows.extend(_rows)\n_result = {\n    \"runtime\": {\"tree_sitter\": importlib.metadata.version(\"tree-sitter\"),\n                \"tree_sitter_rust\": importlib.metadata.version(\"tree-sitter-rust\")},\n    \"fixture_metrics\": coverage_case_metrics,\n    \"fixture_evidence\": coverage_rows,\n    \"fixture_failures\": coverage_failures,\n    \"query_sha256\": hashlib.sha256(_query_bytes).hexdigest(),\n    \"ordinary_call_captured\": _ordinary_call_captured,\n    \"macro_argument_call_captured\": _argument_call_captured,\n    \"source_files\": _file_rows,\n    \"live_evidence\": _live_rows,\n    \"parse_errors\": _parse_errors,\n}\nPath(sys.argv[1]).write_text(json.dumps(_result), encoding=\"utf-8\")\n"
with tempfile.TemporaryDirectory(prefix="spur-macro-benchmark-", dir="/private/tmp") as _job:
    _script = Path(_job) / "benchmark.py"
    _output = Path(_job) / "result.json"
    _script.write_text(_header + _coverage_fixture_source + "\n" + _coverage_collector_source + "\n" + _scan_source)
    _process = subprocess.run([sys.executable, str(_script), str(_output)],
                              text=True, capture_output=True, timeout=60)
    print("parser_process_exit:", _process.returncode)
    if _process.returncode != 0:
        print(_process.stderr[-3000:])
        raise RuntimeError("Isolated parser did not finish; no partial results imported")
    _coverage_data = json.loads(_output.read_text())
print("parser_runtime:", _coverage_data["runtime"])
coverage_case_metrics = _coverage_data["fixture_metrics"]
coverage_rows = _coverage_data["fixture_evidence"]
live_macro_rows = _coverage_data["live_evidence"]
live_macro_parse_errors = _coverage_data["parse_errors"]
_snapshot_oids = dict(_coverage_db.execute("SELECT path, content_oid FROM current_manifests").fetchall())
live_file_rows = [(r["file_path"], r["source_oid"], _snapshot_oids.get(r["file_path"]),
                   r["source_oid"] == _snapshot_oids.get(r["file_path"]))
                  for r in _coverage_data["source_files"]]
for _row in live_macro_rows:
    _row["matches_snapshot"] = _row["source_oid"] == _snapshot_oids.get(_row["file_path"])
_coverage_db.execute("""CREATE OR REPLACE TEMP TABLE macro_fixture_metrics(
    case_name VARCHAR, expected BIGINT, baseline_tp BIGINT,
    prototype_tp BIGINT, prototype_fp BIGINT, prototype_fn BIGINT, context_errors BIGINT)""")
_coverage_db.executemany("INSERT INTO macro_fixture_metrics VALUES (?, ?, ?, ?, ?, ?, ?)", [
    (m["case"], m["expected"], m["baseline_tp"], m["prototype_tp"],
     m["prototype_fp"], m["prototype_fn"], m["context_errors"]) for m in coverage_case_metrics
])
_coverage_db.execute("""CREATE OR REPLACE TEMP TABLE macro_source_files(
    file_path VARCHAR, source_oid VARCHAR, snapshot_oid VARCHAR, matches_snapshot BOOLEAN)""")
_coverage_db.executemany("INSERT INTO macro_source_files VALUES (?, ?, ?, ?)", live_file_rows)
_coverage_db.execute("""CREATE OR REPLACE TEMP TABLE macro_live_evidence(
    file_path VARCHAR, caller VARCHAR, macro_path VARCHAR, line_number INTEGER,
    start_byte BIGINT, end_byte BIGINT, caller_start_byte BIGINT, caller_end_byte BIGINT,
    context VARCHAR, classification VARCHAR, profile_conditional BOOLEAN,
    attributes_json VARCHAR, source_oid VARCHAR, matches_snapshot BOOLEAN)""")
if live_macro_rows:
    _coverage_db.executemany("INSERT INTO macro_live_evidence VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", [
        (m["file_path"], m["caller"], m["macro_path"], m["line"],
         m["start_byte"], m["end_byte"], m["caller_start_byte"], m["caller_end_byte"],
         m["context"], m["classification"], m["profile_conditional"],
         json.dumps(m["attributes"]), m["source_oid"], m["matches_snapshot"])
        for m in live_macro_rows
    ])
print("Rust files parsed:", len(live_file_rows) - len(live_macro_parse_errors))
print("parse failures:", live_macro_parse_errors)
print("source files matching snapshot:", sum(r[3] for r in live_file_rows), "/", len(live_file_rows))
print("direct macro invocations recorded:", len(live_macro_rows))
print("The collector records syntax and context; it does not expand or resolve Rust macros.")



parser_process_exit: 0
parser_runtime: {'tree_sitter': '0.25.2', 'tree_sitter_rust': '0.24.2'}


Rust files parsed: 60
parse failures: []
source files matching snapshot: 60 / 60
direct macro invocations recorded: 4158
The collector records syntax and context; it does not expand or resolve Rust macros.


In [21]:
SELECT 'current Rust edge query' AS strategy, sum(expected) AS expected_direct_sites,
       sum(baseline_tp) AS captured_sites, sum(expected - baseline_tp) AS missed_sites,
       round(100.0 * sum(baseline_tp) / sum(expected), 1) AS fixture_capture_recall_percent
FROM macro_fixture_metrics
UNION ALL
SELECT 'prototype AST evidence', sum(expected), sum(prototype_tp), sum(prototype_fn),
       round(100.0 * sum(prototype_tp) / sum(expected), 1)
FROM macro_fixture_metrics;

strategy,expected_direct_sites,captured_sites,missed_sites,fixture_capture_recall_percent
current Rust edge query,29,0,29,0.0
prototype AST evidence,29,29,0,100.0


In [10]:
SELECT context, classification, count(*) AS direct_invocations
FROM macro_live_evidence GROUP BY ALL ORDER BY context, direct_invocations DESC;

context,classification,direct_invocations
conditional,other_macro_name,13
production,other_macro_name,1551
production,unresolved_bare_sink_candidate,39
production,local_macro_definition,6
test,unresolved_bare_sink_candidate,1968
test,other_macro_name,581


In [22]:
CREATE OR REPLACE TEMP TABLE reachable_macro_evidence AS
SELECT e.*, n.stable_symbol_id, n.qualified_name, r.hops
FROM macro_live_evidence e
JOIN current_nodes n ON n.file_path = e.file_path
  AND n.byte_range_start = e.caller_start_byte AND n.byte_range_end = e.caller_end_byte
JOIN current_reachable r ON r.node = n.node_id
WHERE e.matches_snapshot;
SELECT hops, qualified_name, file_path, line_number, macro_path, context, classification, profile_conditional
FROM reachable_macro_evidence
WHERE context <> 'test'
  AND classification IN ('explicit_standard_path_candidate', 'unresolved_bare_sink_candidate')
ORDER BY hops, file_path, line_number;

hops,qualified_name,file_path,line_number,macro_path,context,classification,profile_conditional
3.0,impl SharedParseCache::evict_before,crates/spur-graph/src/git_walk.rs,2094,debug_assert,production,unresolved_bare_sink_candidate,True
3.0,impl SharedParseCache::evict_before,crates/spur-graph/src/git_walk.rs,2095,debug_assert,production,unresolved_bare_sink_candidate,True
4.0,impl OccupancyCounter::decrement,crates/spur-graph/src/git_walk.rs,244,debug_assert,production,unresolved_bare_sink_candidate,True
5.0,parse_raw_diff,crates/spur-graph/src/git_walk.rs,3412,unreachable,production,unresolved_bare_sink_candidate,False
5.0,import_licensed_call_target,crates/spur-graph/src/store/build.rs,1922,debug_assert_eq,production,unresolved_bare_sink_candidate,True
5.0,import_licensed_call_target,crates/spur-graph/src/store/build.rs,1926,debug_assert,production,unresolved_bare_sink_candidate,True


In [25]:
# Validate the imported evidence independently of the native parser's point API.
_source_cache = {}
_coordinate_errors = []
for _row in live_macro_rows:
    _path = _row["file_path"]
    if _path not in _source_cache:
        _source_cache[_path] = (_repo_root / _path).read_bytes()
    _bytes = _source_cache[_path]
    _start, _end = _row["start_byte"], _row["end_byte"]
    _valid = 0 <= _start < _end <= len(_bytes)
    _valid = _valid and _row["line"] == _bytes[:_start].count(bytes([10])) + 1
    if _row["caller_start_byte"] is not None:
        _valid = _valid and _row["caller_start_byte"] <= _start < _end <= _row["caller_end_byte"]
    if not _valid:
        _coordinate_errors.append((_path, _start, _row["line"]))
print("validated_source_spans:", len(live_macro_rows))
print("coordinate_errors:", _coordinate_errors)
print("known_parse_raw_diff_site:", [
    (r["file_path"], r["line"]) for r in live_macro_rows
    if r["caller"] == "parse_raw_diff" and r["macro_path"] == "unreachable"
])
assert not _coordinate_errors


validated_source_spans: 4158
coordinate_errors: []
known_parse_raw_diff_site: [('crates/spur-graph/src/git_walk.rs', 3412)]


In [24]:
SELECT context, profile_conditional, count(*) AS named_sink_candidates
FROM macro_live_evidence
WHERE classification IN ('explicit_standard_path_candidate', 'unresolved_bare_sink_candidate')
GROUP BY ALL ORDER BY context, profile_conditional;

context,profile_conditional,named_sink_candidates
production,False,28
production,True,11
test,False,1967
test,True,1


In [23]:
WITH labels AS (
  SELECT source_stable_id, target_label FROM current_edges WHERE relation = 'calls'
  UNION ALL
  SELECT source_stable_id, target_label FROM current_unresolved WHERE relation = 'calls'
)
SELECT m.hops, m.qualified_name, m.file_path, m.line_number, m.macro_path, m.profile_conditional,
       count(l.target_label) AS matching_labels_in_caller
FROM reachable_macro_evidence m
LEFT JOIN labels l ON l.source_stable_id = m.stable_symbol_id
  AND l.target_label = regexp_extract(m.macro_path, '([^:]+)$', 1)
WHERE m.context <> 'test'
  AND m.classification IN ('explicit_standard_path_candidate', 'unresolved_bare_sink_candidate')
GROUP BY ALL ORDER BY m.hops, m.file_path, m.line_number;

hops,qualified_name,file_path,line_number,macro_path,profile_conditional,matching_labels_in_caller
3.0,impl SharedParseCache::evict_before,crates/spur-graph/src/git_walk.rs,2094,debug_assert,True,0
3.0,impl SharedParseCache::evict_before,crates/spur-graph/src/git_walk.rs,2095,debug_assert,True,0
4.0,impl OccupancyCounter::decrement,crates/spur-graph/src/git_walk.rs,244,debug_assert,True,0
5.0,parse_raw_diff,crates/spur-graph/src/git_walk.rs,3412,unreachable,False,0
5.0,import_licensed_call_target,crates/spur-graph/src/store/build.rs,1922,debug_assert_eq,True,0
5.0,import_licensed_call_target,crates/spur-graph/src/store/build.rs,1926,debug_assert,True,0


In [16]:
_coverage_db.execute("""CREATE OR REPLACE TEMP TABLE macro_fixture_evidence(
    case_name VARCHAR, caller VARCHAR, macro_path VARCHAR, context VARCHAR,
    classification VARCHAR, profile_conditional BOOLEAN)""")
_coverage_db.executemany("INSERT INTO macro_fixture_evidence VALUES (?, ?, ?, ?, ?, ?)", [
    (r["case"], r["caller"], r["macro_path"], r["context"], r["classification"], r["profile_conditional"])
    for r in coverage_rows
])
print("fixture evidence rows:", len(coverage_rows))


fixture evidence rows: 29


In [17]:
SELECT case_name, macro_path, context, classification, profile_conditional
FROM macro_fixture_evidence
WHERE case_name IN ('shadowed','custom','alias','nested_tokens','stringify','async_test','cfg_any','cfg_attr')
ORDER BY case_name;

case_name,macro_path,context,classification,profile_conditional
alias,explode,production,other_macro_name,False
async_test,panic,test,unresolved_bare_sink_candidate,False
cfg_any,panic,conditional,unresolved_bare_sink_candidate,False
cfg_attr,panic,conditional,unresolved_bare_sink_candidate,False
custom,app::panic,production,other_qualified_macro,False
nested_tokens,wrapper,production,other_macro_name,False
shadowed,panic,production,local_macro_definition,False
stringify,stringify,production,other_macro_name,False


### Design that closes the gap effectively

#### 1. Capture a site, not just a name

Add a dedicated Rust query capture for a `macro_invocation` and collect it in `extract/languages.rs`. Keep the existing call-expression and macro-body token captures. A macro invocation and a call-shaped token inside its arguments have different provenance and should not share `CallOrigin::MacroBody`.

Persist invocation evidence through `GraphFacts`, the artifact schema, and Parquet storage. A sidecar such as `macro_invocations.parquet` can be joined to paths by `caller_stable_symbol_id`, while avoiding an ordinary runtime `calls` edge to an arbitrarily name-matched macro/function.

| Field group | Minimum evidence | Purpose |
|---|---|---|
| Snapshot identity | graph content hash, source OID, extractor/schema version | Reject stale evidence |
| Site identity | enclosing symbol/kind, nullable caller stable ID, snapshot-scoped site ID | Keep repeated invocations and non-function sites distinct |
| Syntax | raw and normalized macro path, invocation origin | Preserve qualified names and raw identifiers |
| Location | start/end bytes plus line span | Precise diagnostics and independent validation |
| Context | raw attributes/`cfg`, test-context evidence, debug-assertion condition, execution-phase uncertainty | Select the intended build without hiding unknowns |
| Resolution | explicit-path candidate, local-definition evidence, alias/re-export resolution, unresolved | Keep uncertainty visible |
| Triage | optional sink-family policy match | Separate source capture from risk classification |

The two `debug_assert!` sites in `SharedParseCache::evict_before` illustrate why per-site rows matter: a single label at function scope cannot establish coverage of both invocations. The 58 invocations outside functions must retain their enclosing scope and execution-phase uncertainty; do not attach them to a fabricated caller. Include the sidecar in canonical hashing, manifest completeness, incremental replacement/deletion, and schema compatibility checks so it cannot silently lag the rest of the graph.

#### 2. Resolve uncertainty before increasing alert severity

- Bare `panic!` can be shadowed. The fixture collector records a visible local definition as evidence; it does not claim full Rust name resolution.
- `use std::panic as explode;` is captured syntactically, but a name-list classifier misses its standard-macro identity. Resolve aliases and re-exports in the **macro namespace**, retaining unknown results.
- A path resembling `std::panic` is a candidate until resolution establishes its meaning.
- `wrapper!(panic!(...))` and `stringify!(panic!(...))` contain similar tokens with different expansion behavior. Keep nested token evidence separate; do not synthesize runtime calls from arbitrary token text.
- Preserve `cfg` and `cfg_attr` rather than evaluating them from syntax alone. Use observed target/features/`debug_assertions` settings to mark sites active, inactive, or unknown.
- Treat source `.unwrap`/`.expect` labels similarly: unresolved names do not establish standard-library receiver types.

#### 3. Prioritize by observed benefit

| Order | Addition | Evidence for value | Remaining boundary |
|---|---|---|---|
| First | Direct invocation evidence, exact spans, context, and fixture regression coverage | Recovers the 29 fixture sites and six scoped real candidates | No macro expansion or panic feasibility |
| Next | Macro alias/re-export resolution and build-condition evaluation | Addresses the alias fixture and large test/debug candidate population | Requires reliable Rust namespace/build facts |
| Then | Targeted invariant checks with source-bound solver receipts | Existing guard/clamp examples eliminate modeled false positives | Hand-authored models cover only their encoded facts |
| Later | Optional CGA/MIR overlay keyed by compiler, target, profile, features, and source revision | Needed for dependency internals and compiler-inserted checks | Toolchain coupling and conservative path approximations |

#### 4. Measure success without conflating evidence and execution

Use separate acceptance measures:

1. **Capture correctness:** preserve all annotated direct sites, reject the negative controls, and verify exact caller/span/context. Extend fixtures for aliases, shadowing, qualified paths, nested tokens, test attributes, parse errors, and repeated invocations.
2. **Resolution coverage:** count resolved, local, and unresolved identities separately. An unresolved candidate is neither a proven panic nor a safe site.
3. **Alert usefulness:** manually adjudicate a held-out set under an explicit build configuration. Report actionable findings, invariant-guarded candidates, and unknowns separately.
4. **Graph integration:** verify round trips, incremental additions/deletions/moves, canonical hashes, source-OID checks, and coexistence with ordinary/HOF/macro-body call captures.
5. **Cost:** measure full and incremental extraction time, memory, artifact size, and query latency before setting a regression budget. This notebook does not supply performance measurements.
6. **Compiler comparison:** compare source candidates with same-build MIR evidence and targeted runtime tests. Neither a source-graph miss nor a bounded solver result establishes program-wide panic freedom.

This is an analysis and notebook prototype. The production extractor and its schema have not been changed in this turn.


### Implementation clarification: preserve scope locators and coverage domain

M2 review confirmed that physical Rust files and embedded Rust notebook cells need different source contracts. Phase one covers **physical Rust source files**; existing notebook definitions/calls remain available, while cell macro evidence is deferred. The artifact and query metadata must expose `source_scope = physical_rust_files`, so an unsupported-source empty result cannot masquerade as complete coverage.

M2 also creates exact `MacroScopeEvidence` records while validating sites. Constants, type contexts and other non-indexed scopes may have evidence-only IDs (`macro-scope-v1:…`), which must retain their source/range locators through persistence. M3 therefore adds `GraphFacts.macro_scopes` and extends `MacroInvocationEvidenceSet` with `scopes` plus typed `source_scope`. Enclosing and raw-attribute owner references must resolve to these persisted scope records; function callers still reference actual graph symbols. Preserve the existing validation entry point and check persisted scopes against supplied authoritative locations. Missing or conflicting scope data is invalid for a complete new artifact.

M4 writes a core `macro_scopes.parquet` table beside `macro_invocations.parquet`, with matching metadata, counts, completeness and incremental replacement/deletion. Both tables are required even when their complete captured sets are empty. Legacy missing evidence remains unavailable. M5 returns the relevant scope locators with filtered results and handles overlay scope replacement/removal; M6 verifies the complete chain and documents the two-table SQL join. The implementation paths required for this transport are explicitly added to the submitted plan; macro expansion, namespace resolution and actual build-condition evaluation remain outside this phase.

This clarifies the approved exact-scope/provenance requirement; it does not claim a new parser or panic-freedom result. Mutation `c1b2c0da-8dc3-4081-9dd7-b0c051e87721` records the M3–M6 contract before M3 dispatch.


In [27]:
WITH bindings AS (
  SELECT e.file_path, e.start_byte, e.end_byte, e.caller_start_byte, e.matches_snapshot,
         count(n.stable_symbol_id) AS exact_owner_matches
  FROM macro_live_evidence e
  LEFT JOIN current_nodes n ON n.file_path = e.file_path
    AND n.byte_range_start = e.caller_start_byte AND n.byte_range_end = e.caller_end_byte
    AND n.symbol_kind IN ('function', 'method')
  GROUP BY ALL
)
SELECT count(*) AS invocation_sites,
       count(*) FILTER (WHERE NOT matches_snapshot) AS source_mismatches,
       count(*) FILTER (WHERE caller_start_byte IS NULL) AS outside_functions,
       count(*) FILTER (WHERE caller_start_byte IS NOT NULL AND exact_owner_matches = 1) AS exact_function_bindings,
       count(*) FILTER (WHERE caller_start_byte IS NOT NULL AND exact_owner_matches = 0) AS unmapped_function_sites,
       count(*) FILTER (WHERE exact_owner_matches > 1) AS ambiguous_owner_sites,
       (SELECT count(*) FROM macro_live_evidence
        WHERE caller_start_byte IS NULL
          AND classification IN ('explicit_standard_path_candidate', 'unresolved_bare_sink_candidate')) AS named_sinks_outside_functions
FROM bindings;

invocation_sites,source_mismatches,outside_functions,exact_function_bindings,unmapped_function_sites,ambiguous_owner_sites,named_sinks_outside_functions
4158,0,58,4100,0,0,0


### Approved implementation and verification gates

The user approved the first phase on 2026-10-31: direct Rust macro invocation evidence, with **SOLVE PRE → TDD RED/GREEN → SOLVE POST for each task**. SPUR worker: `codex`, model `gpt-6.1-sol`, effort `xhigh`.

The implementation plan is `docs/superpowers/plans/2026-10-31-macro-invocation-evidence.md` (commit `350d976f3`). Execution plan `6888d1b0-fee6-47eb-8cc7-c8f18d8ca570`, epic `bd-156`, has sequential review gates:

| Task | Execution issue | Deliverable |
|---|---|---|
| M1 | bd-3dj | Versioned evidence, identity and provenance contract |
| M2 | bd-1rr | Direct AST capture and conservative context |
| M3 | bd-1k5 | Full/incremental artifacts and canonical hashing |
| M4 | bd-iea | Parquet table, metadata and completeness checks |
| M5 | bd-3u8 | File/caller queries and overlay/backend parity |
| M6 | bd-2u9 | End-to-end regressions and reproducible query documentation |

**Implementation verification:** M1–M6 were approved and integrated through `51f7458c6`; the independently verified graph suite recorded **1,128 passed / 0 failed / 19 existing ignores**. The CLI reader/writer schema mismatch was then repaired in `c70dd7a45`, verified by 62 focused tests and a successful full temporal workspace graph/analyst rebuild.

**Latest notebook experiment:** pinned v11 workspace snapshot `64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4` has **46,114 invocation rows / 17,551 scopes**. All **1,510 Rust source bindings** match, **12 integrity audits** report zero violations, and all **six original missing candidates** are recovered. The five-hop build slice has **263 recorded reachable symbols / 270 graph-crate macro sites / six sink-name candidates**, plus **seven unresolved method-label records**. A separate rustc 1.94.1 fixture now supplies four compiler configurations, 24 function/configuration observations and 24 passing good-input controls, including bounds/overflow MIR checks absent from macro rows. Two finite SOLVE projections pass and five corrupted controls reject. See the rebuilt-workspace section below and its durable evidence index.

Earlier review failures, source-mirror counts and prototype results retain their historical snapshots.

**Local build update:** The user then requested local builds. `SPUR_REMOTE=0 scripts/spur-cargo test -p spur-graph --test rust_macro_token_tree_query` completed successfully in the default sandbox: **4 passed, 0 failed**, wrapper exit 0; build profile 43.18 seconds, test execution 0.08 seconds. Plan mutation `ed862b95-9d08-4220-b1ed-b9c199cd6a28` updated all six task bodies to local verification while preserving their dependencies, Codex model/effort and solver/TDD gates. This is existing-regression baseline evidence, not an M1 feature result.


**M1 attempt 2 review:** Typed evidence and contract tests were produced, but GREEN stopped at a missing `GraphFacts` initializer field in `memory_eval/memory_graph.rs`. Brain accepted that same-crate mechanical scope addition, preserving prior work for attempt 3. Independent review found the fixture line values one too high: actual invocation lines **3, 4, 6**, function lines **2–5**, file lines **1–6**. Earlier PRE receipts (`sol_464b1e08c2c44f3f`, `sol_c6fbad0d1cc0486c`; rejecting controls `sol_b3b6d03c0d374e62`, `sol_03b385b54c424b49`) establish only the supplied finite constraints, not that those source annotations were correct. The retry must refresh source-bound PRE, demonstrate an assertion RED for file-scope evidence falsely marked `FunctionBodySyntax`, then obtain local GREEN and POST. M1 remains unapproved; M2–M6 remain pending.


**M1 approval:** Attempt 3 supplied a compiled assertion RED (10 passed, 1 expected failure), then GREEN (**11 contract tests passed**). Existing local regressions also passed: **4 macro-query + 11 memory-graph + 1 store test**. Fresh source-bound catalog PRE/POST receipts were reviewed: identity/ranges `sol_d8d5392abbe54013` → `sol_130b6917aed3460d`; context `sol_91d5fb4c75354c98` → `sol_77ba56afb840475c`; invalid file/function phase `sol_a1c47a86e6e14129` → `sol_6d06935d4da04ba1` (UNSAT in both). POST uses the actual GREEN fixture rows and additionally checks digest uniqueness. These are finite-fixture checks, not parser or panic-freedom proofs. Beads completion audit 96 and worker evidence 90–94 were independently verified; no active signals remain. Authored test commit `735d1dcf67435b9dc0837ecaab7fb13a0ffe0f9e`, fix `2035c09f83b239b66058ea92a9f530fb8da94a08`; normalized worker tree `72ca468c1` is identical. M1 approval unlocks M2 under the same local build, Codex model/effort and PRE/TDD/POST gates.


**M2 attempt 1 review:** Local RED demonstrated six missing-collector cases (including 0 vs 42 sites) and a missing direct query capture. The required GREEN suite passed **16/16** (6 collector, 5 query, 5 HOF). Fresh catalog checks cover the 42-site finite contract and expected rejecting controls; selected PRE/POST receipts and actual logs were independently inspected. The brain also ran the extractor on the real **60-file spur-graph corpus and recovered 4,158 sites** with valid physical source OIDs/byte ranges and caller references. This matches the prototype’s source-site count, but artifact persistence is still pending.

A corrected local notebook fixture exposed the boundary bug: one Rust cell emits a row for `review.ipynb#cell:rust-cell`, whose OID and span refer to virtual cell bytes, not an addressable physical graph file. The source-addressability regression fails. Mutation `80f4bad2-ab90-458a-ad5c-153916ef59d9` requests a physical-source-owner guard plus RED/GREEN and refreshed PRE/POST, preserving ordinary notebook definitions/calls. The combined debug corpus-and-verification driver took 110.865812333 seconds, including extra file reads and hashing; it does not measure isolated extraction speed or production latency. Scope locator transport into durable artifacts is explicitly flagged for M3 preparation.


**M2 approval:** The new notebook assertion failed before the guard, then passed after gating by actual `NodeKind::File`. Local GREEN is **17/17 capture/query/HOF**, **11/11 M1 contracts**, and **18/18 notebook tests** (the latter uses `TMPDIR=/private/tmp` for the existing canonical-path harness issue). The physical 42-site observation remains byte-identical. Fresh source-bound PRE/POST positive receipts `sol_0c9c76d19900490a` → `sol_47e6bbbd095f4c2e` pass; Cell-owner and nonzero-cell controls `sol_f581cc4bc7ba42cd`/`sol_9f5572432b474c4d` and `sol_2dc04d7c94e74453`/`sol_ef95396171b54a4d` reject. Actual logs, source bindings, scoped diff, committed tree and Beads completion132/worker audits125–130 were independently checked. Author test `6a1eb31fdcecfdc4746088ec1a7da8d6c75178bc`, fix `59225ba135c9674e848c46ff20a80f839b2c8fe2`; normalized tree `a6c6c6eef` is identical. Approval-time overlap warnings were inspected: M1 is an ancestor, its only shared changed file retains the intended module addition with the M1 API intact; the warning label was cleared. M3 now retains exact scope locator records and explicit physical-Rust source coverage as specified in the design clarification.


**M3 startup correction:** Attempt 1 stopped before implementation because it selected the old source planning issue `bd-shq`; the actual assigned execution issue is `bd-1k5`, which was correctly in progress. Audit/signal calls properly rejected the unassigned template. No files, tests or solver runs were produced in that attempt. Mutation `b87390e6-7aa7-4463-80d3-80bb5f49acac` makes the execution issue explicit at the start of M3–M6 prompts and removes automatic stale source-plan context paths. M3 was requested to retry against `bd-1k5`; it keeps the same scope transport contract, local builds and solver/TDD gates. No source-template activation or delegation rebind is needed.


**M3 attempt 2 review and acceptance ownership:** The transport implementation has actual RED assertions and passing local **36 macro tests + 155 store tests**, dependent/all-target checks and formatting. Finite catalog PRE/POST covers source/scope/caller references, exact scope locations, full/incremental/removed row-set equality, and deliberate stale-row/dangling/missing-scope controls; selected persisted receipts were reloaded successfully. The task was **not approved**: the integration suite had 19 pass / 6 fail / 1 ignored and an exploratory extractor run had 45 pass / 7 fail / 3 ignored.

Independent tests on approved M2 `a6c6c6eef` reproduce the same seven extractor failures and source-hash failure. Diagnostics identify the latter: after a cache write, `.spur/graph-index.pointer.json` joins the source manifests even though `src/lib.rs` is unchanged. The six corpus golden comparisons also extract their own `expected_graph_index.json` output as input. Mutation `a99f1c78-a651-4f7a-bddf-b69d8f263e77` assigns stable corpus input/output separation and intentional schema snapshots to M3; all disk/Parquet/cache integration and narrow exclusion of graph-generated metadata from Git/FS source enumeration to M4. The five Full-vs-Incremental failures depend on M4 preserving evidence through Parquet. No check was waived: M4 must pass the complete integration suite, and M6 still runs the full final regression. M3 retry preserves its code and must pass its revised core/golden gates before approval.


**M3 approval:** Attempt 3 supplied genuine corpus-separation RED assertions, then the full extractor suite passed **54 tests / 3 ignored twice with `SPUR_GRAPH_BLESS` unset**. Local **36 macro tests + 155 store tests**, dependent test checks, all-target checks, formatting and diff checks passed. Exact golden outputs are excluded from temporary input corpora while legitimate JSON inputs remain; six snapshots were intentionally refreshed. Composition now returns contextual errors instead of adding an `expect` panic candidate. Fresh corpus catalog PRE has 6 passing fixtures + 1 rejecting control; POST has 6 passing fixtures + 2 rejecting controls (golden leakage and source-input loss). Selected original transport receipts and all fresh POST receipts were reloaded; their scope remains the encoded finite facts. Authored RED `d4736cb0b3eb649ae1cf4a23e7d7c6727af5b571`, fix `40a020e08f7f8d9b2d1aaa2b9b022463ef53d81c`; normalized tree `5eefd08f70739d8c9c27698dd272d63604f299bd` is identical. Beads completion202 / worker evidence199 were reviewed. Approval-time overlap warnings were checked against both upstream ancestors and cleared as expected sequential changes.

M4 delegation `84ae243a56ac4dc6` is dispatched on that approved tree. Its handoff requires `macro_invocations.parquet` **and** `macro_scopes.parquet`, metadata and deterministic integrity binding, explicit legacy-unavailable versus complete-empty evidence, and the complete `integration_git_incremental` suite. The prior **19 pass / 6 fail / 1 ignored** ledger is pending M4 acceptance, including generated graph-pointer exclusion from Git and filesystem source enumeration. The feature remains unmerged; no panic-execution or panic-freedom claim follows from macro syntax evidence.


**M4 approval:** Beads completion318 / worker evidence316,314,311,310, the exact nine-file diff, authored commits and local logs were reviewed. Genuine RED: storage **0 pass / 8 fail** and full incremental **19 pass / 9 fail / 1 ignored**. GREEN: **52 acceptance tests** (incremental28, storage9, roundtrip9, schema4, dispatch2), **155 store tests + 1 discovery test**, all-target check, formatting and diff checks pass. Existing **1 integration + 3 store ignores** remain; no failed check was waived. This fixes all six original M3 disk/cache failures.

`macro_invocations.parquet` and `macro_scopes.parquet` are core tables. Explicit manifest metadata/counts preserve legacy unavailable versus complete empty, and both zero-row tables are required for complete empty evidence. Typed fields retain source/site/caller/scope/path/span/context; nested JSON retains raw attributes and resolution payloads. A deterministic evidence-set BLAKE3 checksum binds both tables and their evidence/snapshot/source-domain metadata. Missing/truncated tables, stale provenance, bad counts/versions and same-count valid-version tampering are rejected. Stored validation uses snapshot manifests/scopes/graph callers without rereading newer source bytes; publication validates core evidence before pointers. One shared root-relative policy excludes graph-owned outputs from Git and filesystem inputs while preserving legitimate JSON, unrelated `.spur` sources and nested fixtures.

PRE has **1 passing + 5 rejecting** fixtures; POST has **9 passing + 12 rejecting** projections. Selected persisted requests were reloaded, including counts `sol_cfc0221fa443415b`, sources `sol_8a75e65861484170`, scope/attribute references `sol_9086026845d04b89`, callers `sol_d52b578b7d144467`, metadata `sol_b05ed9b6eb6d4915`, and checksum tamper `sol_964301fba4e04ea5` (UNSAT). Both source-mode positives `sol_1632d5a5436b4caf` / `sol_6325ab453b9e48a6` and actual RED leakage controls `sol_07be048593fb4210` / `sol_e3ad6b59994146da` were checked. These are finite observed-contract checks, not parser/cryptographic/atomicity/panic-freedom proofs. Runtime tests establish the tested IO/path behavior.

Authored RED `133e73603873bfc6c373e5027cb54cebb36260df`, implementation `2c2bf5ade8ff68cb566770cfaffd6b0af5b62081`; normalized tree `f3c56ecd8d0e656b09407ec10118d08b4d713e6e` is identical. Approval-time overlap warnings were reviewed against all three upstream ancestors and cleared. M5 delegation `6202dbac723345ec` is dispatched with handoff mutation `36dbf02f-b383-45b0-902f-f9a44f5dcc9c`; it must preserve integrity while adding filtered queries and exact overlay/backend parity. Full-feature experiments and integration remain pending M5–M6.


**M5 attempt 1 review:** The four-file implementation adds exact file/caller/source filters, row and scope provenance, snapshot bindings, and explicit Complete/Partial/Unavailable/UnsupportedSource states. Memory queries validate once; Parquet freezes both compressed evidence tables, validates the full shared checksum and references once, then prunes filtered reads without evidence disk IO. Overlays replace rows/scopes and retain separate base/delta source bindings. Local **40/40 tests** (queries12, macro Parquet9, overlay12, parity7), all-target compile, formatting and diff checks pass. Beads completion446 and worker evidence439–443, actual diff, authored commits and selected persisted PRE/POST receipts were reviewed. These finite receipts cover their filtered/overlay fixtures, not all query inputs.

Approval is held for two independently reproduced failures on normalized commit `773c154a7e61b4500279f9e06246c10bf2c21dff` (tree-identical to authored `5c0bce858bfe2a51f7018cb2d4039da881860a67`):

1. A nonidentity overlay accepts `source_oid` plus a caller with no explicit `file_path`, after inferring the caller's path. The same original filter is rejected by the memory client because `source_oid` requires `file_path`. Validate shape before routing to preserve backend parity.
2. A valid source-manifest table missing the zero-site `empty.rs` row is accepted while `manifest.json` still declares two rows. Macro queries return `Complete` and silently drop that source binding. Validate supporting-source completeness before certifying coverage. Both checksummed macro tables remain intact in this case; the older full reader also lacks this source-count check, so the new query fix must not be presented as fixing all legacy readers.

The independent local wrapper run produced **0 passed / 2 failed**, exit101; final log `/private/tmp/m5-brain-review-final-red.log`, preserved harness `/private/tmp/m5-brain-review-regressions.rs` (SHA256 `d6bf737bd1d76f61eadc59f7f3243c2d48f7212a16254e0422d48e0a75a393e2`). Requested focused rework within the existing four files: fresh PRE, actual regression RED, separate test/fix commits, GREEN and POST for the new input-shape/source-completeness facts. Prior receipts may be retained only for unchanged exact facts. M5 retry delegation `6f4342f441e041c6` is running; M6 remains pending. No implementation has been merged into active main.


**M5 approval:** Attempt 2 preserves the reviewed query implementation and fixes both failures. The original input shape is checked before overlay caller-file inference and before coverage availability. Focused Parquet macro queries compare decoded source-manifest rows with the declared count before certifying any filter, including a source with zero macro sites. The legacy full-reader supporting-count behavior remains outside this focused fix.

Both new regressions produced actual assertion RED before authored test commit `53f43fbf45f2aaaf933786ae2db689485784dede`. GREEN is **42/42**: queries14, macro Parquet9, overlay12, parity7, plus all-target compilation, formatting and diff checks. Tests preserve valid caller-only and explicit-file/source filters across **six client configurations**, reject the invalid original filter consistently, and reject missing source rows under global/matching/zero-site file filters. The previously validated immutable context remains stable after disk mutation. Beads completion459 and worker evidence453–457, exact full/rework diffs, authored/normalized trees and artifact SHA256 values were checked.

All **12 fresh receipts** were reloaded/pinned: PRE2 pass/3 expected rejections; POST4 pass/3 expected rejections. Actual source identity/count `sol_23f136c991694d2f`, omission control `sol_a8a86727af054630`, coverage outcomes `sol_8bc09ba3ade84705` / RED rejection `sol_e23db51503f34df2`, shape outcomes `sol_38bc4fab5eb74ab1` / `sol_7dd7b550a0a74ede`, and actual invalid-acceptance rejection `sol_72a8c3950b394a37` match the runtime facts. The two shape projections jointly cover 18 actual cases within the catalog bound. These are finite contracts; PRE’s leakage control is not a general runtime guarantee for same-count source-table substitutions.

Authored fix `4c6e93fc25beab30e83a6860d82fb98a8ebb2ffe`; normalized tree `e6205548455fd3de9047abc250146d90d4ed2e1a` is identical. The rework touches three files only; the storage helper is unchanged from attempt 1. Approval-time overlap warnings were checked against all four upstream ancestors and cleared. Handoff mutation `2d511fed-133a-4196-8d56-3968f3953b12` carries the final API, remaining limits and execution identities to M6. Final delegation `ba4957a3d9aa41a7` is running with local builds and the same Codex model/effort. It must pass the complete regression and end-to-end gates, exercise multi-row-group query correctness, and report bounded extraction/storage/first-query/subsequent-query costs without inventing performance thresholds. The implementation remains unmerged pending that review.


**M6 attempt 1 review:** The three-file test/docs patch is reviewable but acceptance remains red. Held-out end-to-end tests **3/3** pass, including exact three-site caller evidence, repeated debug candidates, scope/attribute provenance, unsupported notebook cells, full/incremental/delete/rename/move/overlay parity and real row-group boundaries. SQL executes with result sets **1 manifest / 18 invocation rows / 3 caller sites / 22 attribute-owner rows**, with exact IDs checked against the runtime oracle. Historical pre-query scaffold RED is correctly historical; wrong initial test-oracle and stale build-cache setup runs were not counted as production RED. All 11 source-bound M6 PRE/POST receipts were reloaded/pinned: PRE2 pass/3 expected rejections; POST2 pass/4 expected rejections. Actual three-file diff, authored/normalized trees and all 16 referenced artifact SHA256 values match the handoff.

Mandatory full `--no-fail-fast` result: **1,121 passed / 6 failed / 19 existing ignored**. Complete extractor **54 passed / 3 ignored** and incremental integration **28 passed / 1 ignored** pass freshly. Dependent `--tests` and `--keep-going` checks both fail with missing M4 manifest members in an analyst fixture. These results are not full-feature acceptance.

The final-gate failures are assigned to four bounded repairs under mutation `6ceecfdf-a59f-4c29-82a8-d3d171c4fc58`:

1. Four macOS failures occur while tests try to create invalid UTF-8 filenames. Retain rejection/retry/fail-closed assertions through the real strict `PathBuf` collector and existing runtime error injection, without filesystem-dependent invalid names or skips.
2. The Markdown assertion expects a file binding from July. Production commit `2709b4187a7cf2f88a6d61a32731986ba4e9f9ff` in August deliberately keeps `Links` unresolved. Preserve that established policy and test reference destination text plus anti-phantom binding, without introducing a new resolver.
3. Restore the actual structural/temporal parity requirement for the missing `inline::Mode::Fast` enum variant; investigate capture provenance/projection and preserve every structural stable ID rather than excluding variants.
4. Mechanically initialize the analyst fixture’s new optional evidence metadata/count fields to unavailable.

The two scope-drift signals were consumed by this explicit scope update, not waived. Retry 2 preserves the original tests/docs, requires fresh repair PRE/TDD/POST and all final regression/dependent/format gates, and updates the final evidence matrix only from observed results. Previous authored test `2bb8c9423d966a28ee35b77825bbee25fce2a6d7`, docs `09b907c791fc5b070111f167c5243c456ebe7361`; normalized `dbc83f89e75c57ccd19011f6668659245d147e21` is identical. Full records are in `bd-2u9` audits585–595 and root review; retry delegation `5f0a6040d3f64f6f` is allocated. No merge or closure occurred.

The passing bounded debug observation covers six physical Rust files plus one notebook, 18 sites and 22 scopes. Full extraction took **0.113751375 s**, artifact construction/validation **0.074807 s**, incremental rebuild **0.110345875 s**; first filtered query with context validation **0.003337250 s**, subsequent query/equality check **0.000667583 s**. Full macro tables occupy **9,745 + 4,548 bytes**, all artifact files **32,510 bytes**. This is one observation under the default three-test concurrency, excludes compilation/setup/extra verification IO from extraction, and supplies no speed threshold or production benchmark. Retry must refresh the final cost/SQL export after code repairs.


**M6 approval and integration:** Attempt 2 completed all four bounded repairs. The temporal projection fix separates query-local pattern namespaces, restoring `Fast` and matching structural stable IDs; other changes are test harness/policy alignment and three legacy optional fixture fields. Worker serial1128/0/19, dependent `--tests`, graph all-targets, both-crate formatting, SQL oracle and all 19 current repair + 11 retained macro receipts were checked. Brain then independently ran the exact final source with **default parallelism** and ordinary loopback mock permission: **1128 passed / 0 failed / 19 existing ignored**, 87 target summaries, exit0. Log `/private/tmp/m6-brain-default-full-loopback.log`, SHA256 `fba8833e6827538685ec9a987cee8929df8ba83468fa6608c89dff306db75350`. Earlier schedule-sensitive failures remain observed history with unknown cause; no timeout/fallback fix or test waiver is claimed.

All six review gates were approved. The dedicated plan integration tree matches reviewed M6 `02607f5479de23e2a05dc876bb9104167fd131f5` and authored `d9c6321559513d287ff0638a53ca54ef0280ddcf`. Applied only reviewed deltas to active main: M1 `1cd740e85`, M2 `44bfced71`, M3 `caae9e630`, M4 `6328f3c52`, M5 `eb85ed874`, M6 `51f7458c6`. Documentation conflicts preserve prior issue IDs and M3/M4 gate ownership. All 62 non-document feature files match the tested tree, and the five unrelated dirty user files are byte-identical and unstaged. The six authored TDD histories are retained under local `spur/macro-evidence/m1-authored` through `m6-authored` branches.

The final executed notebook section records the real 63-file selected-source artifact, exact six-site caller join, zero integrity violations, source SHA256/blob checks, conservative context counts and separate measured costs. Snapshot `37ef5600b79894ce8d016e8c6eb35dbfc5b62f051516e3754861702d1da2f014`; origin code commit `51f7458c6d898a17544046e4273128dd8759f400`. Root finite post-integration verification `sol_3334f66c731e402e` passes; wrong caller/source, missing site and duplicate controls reject (`sol_6cc5218dda1d4dc1`, `sol_9a35f31ed4c84345`, `sol_1f23cab3b6984534`, `sol_485cf5209fb8495b`). All five were reloaded/pinned. This closes the concrete direct-syntax witness gap while retaining the physical-file, alias/cfg/expansion and execution limits described below.


### Implemented macro evidence: final source experiment

All six tasks are approved and integrated on `main` through `51f7458c6` after an independent complete default-parallel test run: **1,128 passed / 0 failed / 19 existing ignored**. Dependent compilation, all-target checks, formatting, executed SQL and source-bound finite solver receipts were reviewed. Earlier worker parallel failures remain historical observations; no timeout or fallback policy was changed.

The following cells inspect a separate validated artifact for the **63 current physical Rust files under `crates/spur-graph/src`**. A byte-identical mirror preserves repository-relative paths and caller identities; its source manifest is recorded with the origin commit and hashes. It covers this selected source set, not the entire workspace or embedded notebook Rust. The serving graph and earlier prototype measurements above are separate snapshots. Source changes since the earlier 60-file prototype make raw total counts unsuitable as a before/after recall metric.

The effectiveness check revisits the six concrete direct macro candidates missing from the original call-graph slice. Typed invocation evidence, exact source OIDs/ranges, real callers and evidence-only scopes are inspected together. This remains syntax evidence, with alias/build/expansion and panic execution unresolved.

In [28]:
CREATE OR REPLACE TEMP VIEW implemented_macro_sites AS
SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/macro-invocations-final-20261031-51f7458c6.parquet/macro_invocations.parquet');
CREATE OR REPLACE TEMP VIEW implemented_macro_scopes AS
SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/macro-invocations-final-20261031-51f7458c6.parquet/macro_scopes.parquet');
CREATE OR REPLACE TEMP VIEW implemented_graph_symbols AS
SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/macro-invocations-final-20261031-51f7458c6.parquet/nodes.parquet');
CREATE OR REPLACE TEMP VIEW implemented_source_manifest AS
SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/macro-invocations-final-20261031-51f7458c6.parquet/file_manifests.parquet');

SELECT graph_content_hash,
       macro_evidence.source_scope AS source_scope,
       macro_evidence.complete AS evidence_complete,
       schema_version AS graph_schema,
       macro_evidence.schema_version AS evidence_schema,
       (SELECT count(*) FROM implemented_source_manifest) AS source_files,
       (SELECT count(*) FROM implemented_macro_sites) AS invocation_sites,
       (SELECT count(*) FROM implemented_macro_scopes) AS scope_records,
       (SELECT count(*) FROM implemented_macro_sites WHERE caller_stable_symbol_id IS NULL) AS sites_without_function_caller
FROM read_json_auto('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/macro-invocations-final-20261031-51f7458c6.parquet/manifest.json');

graph_content_hash,source_scope,evidence_complete,graph_schema,evidence_schema,source_files,invocation_sites,scope_records,sites_without_function_caller
37ef5600b79894ce8d016e8c6eb35dbfc5b62f051516e3754861702d1da2f014,physical_rust_files,True,spur-graph-schema-v11,spur-macro-invocation-v1,63,4256,1636,58


In [30]:
-- Full source/site keys remain in the views; prefixes below are display-only.
SELECT replace(m.file_path, 'crates/spur-graph/src/', '') AS file,
       c.qualified_name AS caller, m.normalized_macro_path AS macro_path,
       m.line_start, m.line_end, m.test_context, m.execution_phase,
       m.debug_assertions_candidate AS debug_candidate,
       left(m.source_oid, 12) AS source_blob, left(m.site_id, 12) AS site_key
FROM implemented_macro_sites m
JOIN implemented_graph_symbols c
  ON c.stable_symbol_id = m.caller_stable_symbol_id AND c.file_path = m.file_path
WHERE (
  (m.file_path = 'crates/spur-graph/src/git_walk.rs'
   AND c.qualified_name IN ('impl OccupancyCounter::decrement',
       'impl SharedParseCache::evict_before', 'parse_raw_diff'))
  OR
  (m.file_path = 'crates/spur-graph/src/store/build.rs'
   AND c.qualified_name = 'import_licensed_call_target')
)
AND (m.normalized_macro_path LIKE 'debug_assert%'
     OR m.normalized_macro_path = 'unreachable')
ORDER BY m.file_path, m.line_start, m.site_id;

file,caller,macro_path,line_start,line_end,test_context,execution_phase,debug_candidate,source_blob,site_key
git_walk.rs,impl OccupancyCounter::decrement,debug_assert,244,244,unspecified,function_body_syntax,True,96681b490e68,376a54c55803
git_walk.rs,impl SharedParseCache::evict_before,debug_assert,2095,2095,unspecified,function_body_syntax,True,96681b490e68,832881b51acb
git_walk.rs,impl SharedParseCache::evict_before,debug_assert,2096,2096,unspecified,function_body_syntax,True,96681b490e68,ea67e46911d1
git_walk.rs,parse_raw_diff,unreachable,3413,3413,unspecified,function_body_syntax,False,96681b490e68,1d2ac2833cf6
store/build.rs,import_licensed_call_target,debug_assert_eq,2033,2033,unspecified,function_body_syntax,True,0dd2c588e822,4adf531cd4dc
store/build.rs,import_licensed_call_target,debug_assert,2037,2040,unspecified,function_body_syntax,True,0dd2c588e822,928e07c557e4


In [31]:
WITH attribute_owners AS (
  SELECT m.file_path, m.source_oid,
         json_extract_string(a.value, '$.owner.stable_symbol_id') AS owner_id,
         json_extract_string(a.value, '$.owner.kind') AS owner_kind
  FROM implemented_macro_sites m, json_each(m.attributes) a
)
SELECT 'duplicate site keys' AS check_name, count(*) AS violations
FROM (SELECT site_id FROM implemented_macro_sites GROUP BY site_id HAVING count(*) > 1)
UNION ALL
SELECT 'duplicate scope keys', count(*)
FROM (SELECT scope_id FROM implemented_macro_scopes GROUP BY scope_id HAVING count(*) > 1)
UNION ALL
SELECT 'invocation source mismatch', count(*)
FROM implemented_macro_sites m
LEFT JOIN implemented_source_manifest f ON f.path = m.file_path AND f.content_oid = m.source_oid
WHERE f.path IS NULL
UNION ALL
SELECT 'scope source mismatch', count(*)
FROM implemented_macro_scopes s
LEFT JOIN implemented_source_manifest f ON f.path = s.file_path AND f.content_oid = s.source_oid
WHERE f.path IS NULL
UNION ALL
SELECT 'missing enclosing scope', count(*)
FROM implemented_macro_sites m
LEFT JOIN implemented_macro_scopes s
 ON s.scope_id = m.enclosing_scope_id AND s.kind = m.enclosing_scope_kind
 AND s.file_path = m.file_path AND s.source_oid = m.source_oid
WHERE s.scope_id IS NULL
UNION ALL
SELECT 'missing attribute owner', count(*)
FROM attribute_owners a LEFT JOIN implemented_macro_scopes s
 ON s.scope_id = a.owner_id AND s.kind = a.owner_kind
 AND s.file_path = a.file_path AND s.source_oid = a.source_oid
WHERE s.scope_id IS NULL
UNION ALL
SELECT 'invalid graph function caller', count(*)
FROM implemented_macro_sites m LEFT JOIN implemented_graph_symbols c
 ON c.stable_symbol_id = m.caller_stable_symbol_id AND c.file_path = m.file_path
WHERE m.caller_stable_symbol_id IS NOT NULL
 AND (c.stable_symbol_id IS NULL OR c.symbol_kind NOT IN ('function', 'method'))
UNION ALL
SELECT 'invented or missing function caller', count(*)
FROM implemented_macro_sites
WHERE (caller_stable_symbol_id IS NOT NULL AND enclosing_scope_kind <> 'function')
 OR (execution_phase = 'function_body_syntax' AND caller_stable_symbol_id IS NULL)
ORDER BY check_name;

check_name,violations
duplicate scope keys,0
duplicate site keys,0
invalid graph function caller,0
invented or missing function caller,0
invocation source mismatch,0
missing attribute owner,0
missing enclosing scope,0
scope source mismatch,0


In [32]:
SELECT test_context, execution_phase,
       count(*) AS invocation_sites,
       count(*) FILTER (WHERE debug_assertions_candidate) AS debug_candidates,
       count(*) FILTER (WHERE sink_family_candidate IS NOT NULL) AS name_based_sink_candidates
FROM implemented_macro_sites
GROUP BY test_context, execution_phase
ORDER BY test_context, execution_phase;

test_context,execution_phase,invocation_sites,debug_candidates,name_based_sink_candidates
conditional,function_body_syntax,2568,1,1979
conditional,item_syntax,5,0,0
unspecified,const_context_syntax,44,0,0
unspecified,function_body_syntax,1630,11,39
unspecified,item_syntax,9,0,0


In [33]:
WITH measurement AS (
  SELECT * FROM read_json_auto('/Users/kevintruong/project/SpurCode/.spur/graph/experiments/macro-invocations-final-20261031-51f7458c6.parquet/review-summary.json')
)
SELECT '1. selected-source extraction' AS operation, round(extract_s, 6) AS seconds FROM measurement
UNION ALL SELECT '2. artifact construction and validation', round(artifact_s, 6) FROM measurement
UNION ALL SELECT '3. Parquet write and stored validation', round(write_s, 6) FROM measurement
UNION ALL SELECT '4. first filtered query and full context validation', round(first_filtered_query_s, 6) FROM measurement
UNION ALL SELECT '5. repeated filtered query and equality check', round(subsequent_filtered_query_s, 6) FROM measurement
ORDER BY operation;

operation,seconds
1. selected-source extraction,105.690547
2. artifact construction and validation,78.882536
3. Parquet write and stored validation,0.487259
4. first filtered query and full context validation,0.276619
5. repeated filtered query and equality check,0.005686


In [34]:
from pathlib import Path
import hashlib
import json

artifact_dir = Path("/Users/kevintruong/project/SpurCode/.spur/graph/experiments/macro-invocations-final-20261031-51f7458c6.parquet")
origin = json.loads((artifact_dir / "origin-source-manifest.json").read_text())
summary = json.loads((artifact_dir / "review-summary.json").read_text())
manifest_oids = {item["path"]: item["content_oid"] for item in summary["source_manifests"]}
assert len(origin["paths"]) == len(manifest_oids) == 63
for relative_path in origin["paths"]:
    data = (Path(origin["origin_root"]) / relative_path).read_bytes()
    assert hashlib.sha256(data).hexdigest() == origin["sha256"][relative_path], relative_path
    blob_oid = hashlib.sha1(b"blob " + str(len(data)).encode() + b"\0" + data).hexdigest()
    assert blob_oid == manifest_oids[relative_path], relative_path
{
    "physical_source_files_verified": len(manifest_oids),
    "origin_code_commit": origin["head"],
    "source_sha256_and_artifact_blob_oids_match": True,
    "validated_artifact_roundtrip": summary["validated_roundtrip"],
    "original_gap_candidates_recovered": summary["original_gap_candidate_count"],
    "macro_table_bytes": summary["sizes_bytes"]["macro_invocations.parquet"] + summary["sizes_bytes"]["macro_scopes.parquet"],
}

{'physical_source_files_verified': 63,
 'origin_code_commit': '51f7458c6d898a17544046e4273128dd8759f400',
 'source_sha256_and_artifact_blob_oids_match': True,
 'validated_artifact_roundtrip': True,
 'original_gap_candidates_recovered': 6,
 'macro_table_bytes': 335674}

### Effectiveness and remaining gap

The implemented pipeline closes the measured **6/6 direct-syntax witness gap**: five debug-assert candidates and the `unreachable!` candidate are now retrievable by their real caller and exact physical source location. The two `SharedParseCache::evict_before` sites retain distinct keys. A seventh similarly named candidate in those two files belongs to a test outside the original caller slice and is excluded from that six-site claim.

The selected current source snapshot contains **4,256 invocation rows, 1,636 scope records and 63 verified source files**. Of these sites, **58 have no function caller**: 44 use const-context syntax and 14 item syntax. Their scope provenance is retained instead of inventing a function owner. All eight SQL identity/reference checks return **zero violations**, and the Rust full read/write/query roundtrip validated table metadata, counts, shared checksum and stored provenance. The Python cell independently matches every origin file’s SHA256 and artifact blob OID.

Context remains conservative: 2,573 sites are conditional and 1,683 unspecified. Name-based sink candidates total 2,018 (1,979 conditional, 39 unspecified). **Unspecified is not a production classification; a candidate name is not a proven standard macro identity or a panic execution.** This run does not evaluate cfg, aliases, expansion, dependency/compiler-inserted panics or reachability.

Fresh finite catalog verification of the six exact sites, four graph callers and two physical source identities uses 60 variables:
- Positive actual set: `sol_3334f66c731e402e` — SAT/pass.
- Invented caller: `sol_6cc5218dda1d4dc1`; stale source: `sol_9a35f31ed4c84345`; omitted site: `sol_1f23cab3b6984534`; duplicate identity: `sol_485cf5209fb8495b` — expected UNSAT/rejections.

These receipts check the supplied finite identities, references and count. Runtime/source checks establish the measured producer behavior; they do not turn the model into a Rust execution proof.

The real-source debug observation separates **105.691 s extraction**, **78.883 s artifact construction/validation**, **0.487 s Parquet write**, **0.277 s first filtered query/context validation**, and **0.00569 s repeated query/equality check**. The two macro tables occupy **335,674 bytes**. Compilation and the later notebook/source verification are excluded from those phase timers. This single cold source recapture is a bounded observation, not a production benchmark or a comparison with the earlier mixed 110.866-second driver. A release measurement and profile should precede any performance change.

The next semantic experiment should pin a Rust build configuration and compare compiler-derived panic/call evidence with this syntax inventory, preserving unknown mappings and expansion provenance. CGA/MIR has not been run here.

Artifact: `/Users/kevintruong/project/SpurCode/.spur/graph/experiments/macro-invocations-final-20261031-51f7458c6.parquet`

Snapshot: `37ef5600b79894ce8d016e8c6eb35dbfc5b62f051516e3754861702d1da2f014`; origin code commit `51f7458c6d898a17544046e4273128dd8759f400`. The directory retains the producer source/log, origin manifest and solver requests/receipts beside the validated tables. Earlier research cells are historical snapshots; the source changed from 60 to 63 files, so 4,158 versus 4,256 is not a paired recall metric.

**Archive availability update:** this source-mirror cache directory was removed during the later workspace graph rebuild. The saved results above remain historical evidence. The rebuilt-workspace section below uses snapshot `64c5b7a1…` and exports its evidence outside the graph cache.


## Rebuilt workspace experiment — direct macro evidence and panic triage

This section uses the completed **v11 workspace graph**, snapshot `64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4`, after the schema-guard repair in `c70dd7a45`. Its manifest declares **46,114 direct macro rows and 17,551 scope records**. File manifests bind the actual workspace bytes; the indexed commit alone does not describe uncommitted working-tree changes.

The earlier 63-file mirror is an archived selected-source experiment. Its saved outputs remain evidence of that run, but its cache directory was removed during the later graph rebuild. Counts from that mirror and this full workspace have different denominators.

We will measure: (1) exact recovery of the six original candidates; (2) source, scope and caller integrity; (3) direct macro candidates reachable within the historical five-hop call-graph budget; and (4) remaining method-call, cfg, expansion and compiler-check gaps. A macro name is a candidate, an edge is recorded graph evidence, and a path to a caller does not prove the invocation executes or panics.

All fresh relations use the `pg11_` prefix and one pinned artifact path. SQL schema discovery comes first. Old cells keep their historical snapshots.

In [36]:
-- Pin every input to one immutable graph directory; do not reuse archived relations.
CREATE OR REPLACE TEMP VIEW pg11_macro_sites_raw AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4.parquet/macro_invocations.parquet');
CREATE OR REPLACE TEMP VIEW pg11_macro_scopes_raw AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4.parquet/macro_scopes.parquet');
CREATE OR REPLACE TEMP VIEW pg11_nodes_raw AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4.parquet/nodes.parquet');
CREATE OR REPLACE TEMP VIEW pg11_edges_raw AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4.parquet/edges.parquet');
CREATE OR REPLACE TEMP VIEW pg11_unresolved_raw AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4.parquet/edges_unresolved.parquet');
CREATE OR REPLACE TEMP VIEW pg11_source_manifest_raw AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4.parquet/file_manifests.parquet');
CREATE OR REPLACE TEMP VIEW pg11_files_raw AS SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4.parquet/files.parquet');
CREATE OR REPLACE TEMP VIEW pg11_manifest_raw AS SELECT * FROM read_json_auto('/Users/kevintruong/project/SpurCode/.spur/graph/64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4.parquet/manifest.json');

SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_name IN ('pg11_macro_sites_raw', 'pg11_macro_scopes_raw', 'pg11_nodes_raw', 'pg11_edges_raw', 'pg11_unresolved_raw', 'pg11_source_manifest_raw', 'pg11_files_raw', 'pg11_manifest_raw')
ORDER BY table_name, ordinal_position;

table_name,column_name,data_type
pg11_edges_raw,source_stable_id,VARCHAR
pg11_edges_raw,target_stable_id,VARCHAR
pg11_edges_raw,src_id,BIGINT
pg11_edges_raw,dst_id,BIGINT
pg11_edges_raw,target_label,VARCHAR
pg11_edges_raw,relation,VARCHAR
pg11_edges_raw,confidence,VARCHAR
pg11_edges_raw,confidence_score,FLOAT
pg11_edges_raw,edge_kind,VARCHAR
pg11_edges_raw,bind_method,VARCHAR


In [37]:
import json
from pathlib import Path
import duckdb

pg11_artifact = Path('/Users/kevintruong/project/SpurCode/.spur/graph/64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4.parquet')
pg11_meta = json.loads((pg11_artifact / 'manifest.json').read_text())
assert pg11_meta['graph_content_hash'] == '64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4'
assert pg11_meta['schema_version'] == 'spur-graph-schema-v11'
assert pg11_meta['complete'] and pg11_meta['macro_evidence']['complete']

# fetchall preserves the schema beyond the SQL table preview's 20-row display.
pg11_schema = duckdb.sql("""
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_name IN ('pg11_macro_sites_raw', 'pg11_macro_scopes_raw',
 'pg11_nodes_raw', 'pg11_edges_raw', 'pg11_unresolved_raw',
 'pg11_source_manifest_raw', 'pg11_files_raw', 'pg11_manifest_raw')
ORDER BY table_name, ordinal_position
""").fetchall()
print(json.dumps({name: {col: typ for rel, col, typ in pg11_schema if rel == name}
                  for name in sorted({rel for rel, _, _ in pg11_schema})}, indent=2))

{
  "pg11_edges_raw": {
    "source_stable_id": "VARCHAR",
    "target_stable_id": "VARCHAR",
    "src_id": "BIGINT",
    "dst_id": "BIGINT",
    "target_label": "VARCHAR",
    "relation": "VARCHAR",
    "confidence": "VARCHAR",
    "confidence_score": "FLOAT",
    "edge_kind": "VARCHAR",
    "bind_method": "VARCHAR",
    "import_path": "VARCHAR",
    "receiver_text": "VARCHAR",
    "scope_text": "VARCHAR"
  },
  "pg11_files_raw": {
    "stable_file_id": "VARCHAR",
    "node_id": "BIGINT",
    "file_path": "VARCHAR"
  },
  "pg11_macro_scopes_raw": {
    "file_path": "VARCHAR",
    "source_oid": "VARCHAR",
    "byte_range_start": "BIGINT",
    "byte_range_end": "BIGINT",
    "line_start": "BIGINT",
    "line_end": "BIGINT",
    "scope_id": "VARCHAR",
    "kind": "VARCHAR"
  },
  "pg11_macro_sites_raw": {
    "file_path": "VARCHAR",
    "source_oid": "VARCHAR",
    "byte_range_start": "BIGINT",
    "byte_range_end": "BIGINT",
    "line_start": "BIGINT",
    "line_end": "BIGINT",
    "sit

In [38]:
-- Freeze this run in the notebook connection; later cells cannot silently follow CURRENT.
CREATE OR REPLACE TEMP TABLE pg11_macro_sites AS SELECT * FROM pg11_macro_sites_raw;
CREATE OR REPLACE TEMP TABLE pg11_macro_scopes AS SELECT * FROM pg11_macro_scopes_raw;
CREATE OR REPLACE TEMP TABLE pg11_nodes AS SELECT * FROM pg11_nodes_raw;
CREATE OR REPLACE TEMP TABLE pg11_edges AS SELECT * FROM pg11_edges_raw;
CREATE OR REPLACE TEMP TABLE pg11_unresolved AS SELECT * FROM pg11_unresolved_raw;
CREATE OR REPLACE TEMP TABLE pg11_source_manifest AS SELECT * FROM pg11_source_manifest_raw;
CREATE OR REPLACE TEMP TABLE pg11_files AS SELECT * FROM pg11_files_raw;
CREATE OR REPLACE TEMP TABLE pg11_manifest AS SELECT * FROM pg11_manifest_raw;

SELECT m.graph_content_hash, m.schema_version,
       m.macro_evidence.complete AS evidence_declared_complete,
       m.macro_evidence.source_scope AS source_scope,
       (SELECT count(*) FROM pg11_source_manifest) AS graph_files,
       (SELECT count(DISTINCT file_path) FROM pg11_macro_sites) AS files_with_macro_sites,
       (SELECT count(*) FROM pg11_macro_sites) AS invocation_sites,
       (SELECT count(*) FROM pg11_macro_scopes) AS scope_records,
       (SELECT count(*) FROM pg11_macro_sites WHERE caller_stable_symbol_id IS NULL) AS sites_without_function_caller,
       (SELECT count(*) FROM pg11_macro_sites) = m.row_counts.macro_invocations
         AND (SELECT count(*) FROM pg11_macro_scopes) = m.row_counts.macro_scopes
         AS macro_manifest_counts_match
FROM pg11_manifest m;

graph_content_hash,schema_version,evidence_declared_complete,source_scope,graph_files,files_with_macro_sites,invocation_sites,scope_records,sites_without_function_caller,macro_manifest_counts_match
64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4,spur-graph-schema-v11,True,physical_rust_files,3579,1265,46114,17551,209,True


In [46]:
CREATE OR REPLACE TEMP TABLE pg11_integrity AS
WITH attribute_owners AS (
  SELECT m.file_path, m.source_oid,
         json_extract_string(a.value, '$.owner.stable_symbol_id') AS owner_id,
         json_extract_string(a.value, '$.owner.kind') AS owner_kind
  FROM pg11_macro_sites m, json_each(m.attributes) a
)
SELECT 'duplicate site keys' AS check_name, count(*) AS violations
FROM (SELECT site_id FROM pg11_macro_sites GROUP BY site_id HAVING count(*) > 1)
UNION ALL
SELECT 'duplicate scope keys', count(*)
FROM (SELECT scope_id FROM pg11_macro_scopes GROUP BY scope_id HAVING count(*) > 1)
UNION ALL
SELECT 'invocation source mismatch', count(*)
FROM pg11_macro_sites m
LEFT JOIN pg11_source_manifest f ON f.path = m.file_path AND f.content_oid = m.source_oid
WHERE f.path IS NULL
UNION ALL
SELECT 'scope source mismatch', count(*)
FROM pg11_macro_scopes s
LEFT JOIN pg11_source_manifest f ON f.path = s.file_path AND f.content_oid = s.source_oid
WHERE f.path IS NULL
UNION ALL
SELECT 'missing enclosing scope', count(*)
FROM pg11_macro_sites m
LEFT JOIN pg11_macro_scopes s
 ON s.scope_id = m.enclosing_scope_id AND s.kind = m.enclosing_scope_kind
 AND s.file_path = m.file_path AND s.source_oid = m.source_oid
WHERE s.scope_id IS NULL
UNION ALL
SELECT 'missing attribute owner', count(*)
FROM attribute_owners a LEFT JOIN pg11_macro_scopes s
 ON s.scope_id = a.owner_id AND s.kind = a.owner_kind
 AND s.file_path = a.file_path AND s.source_oid = a.source_oid
WHERE s.scope_id IS NULL
UNION ALL
SELECT 'invalid graph function caller', count(*)
FROM pg11_macro_sites m LEFT JOIN pg11_nodes c
 ON c.stable_symbol_id = m.caller_stable_symbol_id AND c.file_path = m.file_path
WHERE m.caller_stable_symbol_id IS NOT NULL
 AND (c.stable_symbol_id IS NULL OR c.symbol_kind NOT IN ('function', 'method'))
UNION ALL
SELECT 'invented or missing function caller', count(*)
FROM pg11_macro_sites
WHERE (caller_stable_symbol_id IS NOT NULL AND enclosing_scope_kind <> 'function')
 OR (execution_phase = 'function_body_syntax' AND caller_stable_symbol_id IS NULL)
UNION ALL
SELECT 'invalid invocation span', count(*)
FROM pg11_macro_sites
WHERE byte_range_start < 0 OR byte_range_end <= byte_range_start
 OR line_start < 1 OR line_end < line_start
UNION ALL
SELECT 'site outside enclosing scope span', count(*)
FROM pg11_macro_sites m JOIN pg11_macro_scopes s
 ON s.scope_id = m.enclosing_scope_id AND s.file_path = m.file_path AND s.source_oid = m.source_oid
WHERE m.byte_range_start < s.byte_range_start OR m.byte_range_end > s.byte_range_end
UNION ALL
SELECT 'caller differs from enclosing function scope', count(*)
FROM pg11_macro_sites
WHERE caller_stable_symbol_id IS NOT NULL
 AND enclosing_scope_kind = 'function'
 AND caller_stable_symbol_id <> enclosing_scope_id
UNION ALL
SELECT 'site outside function caller span', count(*)
FROM pg11_macro_sites m JOIN pg11_nodes c
 ON c.stable_symbol_id = m.caller_stable_symbol_id AND c.file_path = m.file_path
WHERE m.byte_range_start < c.byte_range_start OR m.byte_range_end > c.byte_range_end
ORDER BY check_name;
SELECT * FROM pg11_integrity ORDER BY check_name;

check_name,violations
caller differs from enclosing function scope,0
duplicate scope keys,0
duplicate site keys,0
invalid graph function caller,0
invalid invocation span,0
invented or missing function caller,0
invocation source mismatch,0
missing attribute owner,0
missing enclosing scope,0
scope source mismatch,0


In [40]:
CREATE OR REPLACE TEMP TABLE pg11_witnesses AS
SELECT m.*, c.qualified_name AS caller_name
FROM pg11_macro_sites m
JOIN pg11_nodes c ON c.stable_symbol_id = m.caller_stable_symbol_id
                  AND c.file_path = m.file_path
WHERE (
  (m.file_path = 'crates/spur-graph/src/git_walk.rs'
   AND c.qualified_name IN ('impl OccupancyCounter::decrement',
       'impl SharedParseCache::evict_before', 'parse_raw_diff'))
  OR
  (m.file_path = 'crates/spur-graph/src/store/build.rs'
   AND c.qualified_name = 'import_licensed_call_target')
)
AND (m.normalized_macro_path LIKE 'debug_assert%'
     OR m.normalized_macro_path = 'unreachable');

SELECT replace(file_path, 'crates/spur-graph/src/', '') AS file,
       caller_name, normalized_macro_path AS macro_path, line_start, line_end,
       test_context, execution_phase, debug_assertions_candidate,
       left(source_oid, 12) AS source_blob, left(site_id, 12) AS site_key
FROM pg11_witnesses
ORDER BY file_path, line_start, site_id;

file,caller_name,macro_path,line_start,line_end,test_context,execution_phase,debug_assertions_candidate,source_blob,site_key
git_walk.rs,impl OccupancyCounter::decrement,debug_assert,244,244,unspecified,function_body_syntax,True,96681b490e68,376a54c55803
git_walk.rs,impl SharedParseCache::evict_before,debug_assert,2095,2095,unspecified,function_body_syntax,True,96681b490e68,832881b51acb
git_walk.rs,impl SharedParseCache::evict_before,debug_assert,2096,2096,unspecified,function_body_syntax,True,96681b490e68,ea67e46911d1
git_walk.rs,parse_raw_diff,unreachable,3413,3413,unspecified,function_body_syntax,False,96681b490e68,1d2ac2833cf6
store/build.rs,import_licensed_call_target,debug_assert_eq,2033,2033,unspecified,function_body_syntax,True,0dd2c588e822,4adf531cd4dc
store/build.rs,import_licensed_call_target,debug_assert,2037,2040,unspecified,function_body_syntax,True,0dd2c588e822,928e07c557e4


In [41]:
-- Replay the earlier five-hop budget from the same entry-point meaning.
-- These are recorded resolved graph targets, not type-checked Rust execution paths.
CREATE OR REPLACE TEMP TABLE pg11_root AS
SELECT node_id, stable_symbol_id, qualified_name, file_path
FROM pg11_nodes
WHERE file_path = 'crates/spur-cli/src/commands/graph.rs'
  AND qualified_name = 'build_with_temporal_jobs_override';
SELECT CASE WHEN count(*) = 1 THEN true ELSE error('entry point must resolve uniquely') END
FROM pg11_root;

CREATE OR REPLACE TEMP TABLE pg11_call_edges AS
SELECT DISTINCT src_id, dst_id
FROM pg11_edges
WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')
  AND src_id IS NOT NULL AND dst_id IS NOT NULL;

CREATE OR REPLACE TEMP TABLE pg11_reachable AS
WITH RECURSIVE walk(node_id, hops) AS (
  SELECT node_id, 0 FROM pg11_root
  UNION
  SELECT e.dst_id, w.hops + 1
  FROM walk w JOIN pg11_call_edges e ON e.src_id = w.node_id
  WHERE w.hops < 5
)
SELECT node_id, min(hops) AS hops FROM walk GROUP BY node_id;

CREATE OR REPLACE TEMP TABLE pg11_reached_nodes AS
SELECT r.hops, n.* FROM pg11_reachable r JOIN pg11_nodes n USING (node_id);

CREATE OR REPLACE TEMP TABLE pg11_reached_macro_sites AS
SELECT r.hops, r.qualified_name AS caller_name, m.*
FROM pg11_reached_nodes r JOIN pg11_macro_sites m
 ON m.caller_stable_symbol_id = r.stable_symbol_id AND m.file_path = r.file_path;

SELECT r.hops, count(*) AS recorded_reachable_symbols,
       count(*) FILTER (WHERE r.file_path LIKE 'crates/spur-graph/src/%') AS spur_graph_symbols,
       (SELECT count(*) FROM pg11_reached_macro_sites m
        WHERE m.hops = r.hops AND m.file_path LIKE 'crates/spur-graph/src/%')
        AS spur_graph_macro_sites,
       (SELECT count(*) FROM pg11_reached_macro_sites m
        WHERE m.hops = r.hops AND m.file_path LIKE 'crates/spur-graph/src/%'
          AND m.sink_family_candidate IS NOT NULL)
        AS spur_graph_name_based_sink_candidates
FROM pg11_reached_nodes r
GROUP BY r.hops ORDER BY r.hops;

hops,recorded_reachable_symbols,spur_graph_symbols,spur_graph_macro_sites,spur_graph_name_based_sink_candidates
0,1,0,0,0
1,26,10,44,0
2,50,41,51,0
3,59,59,68,2
4,51,51,51,1
5,76,75,56,3


In [42]:
CREATE OR REPLACE TEMP TABLE pg11_reached_sink_candidates AS
SELECT * FROM pg11_reached_macro_sites
WHERE file_path LIKE 'crates/spur-graph/src/%' AND sink_family_candidate IS NOT NULL;

CREATE OR REPLACE TEMP TABLE pg11_witness_reach AS
WITH labels AS (
  SELECT source_stable_id, target_label FROM pg11_edges WHERE relation = 'calls'
  UNION ALL
  SELECT source_stable_id, target_label FROM pg11_unresolved WHERE relation = 'calls'
)
SELECT w.site_id, w.caller_stable_symbol_id, w.caller_name, w.file_path,
       w.source_oid, w.normalized_macro_path, w.line_start, w.line_end,
       w.test_context, w.execution_phase, w.resolution,
       w.debug_assertions_candidate, r.hops,
       count(l.target_label) AS matching_macro_call_label_records
FROM pg11_witnesses w
LEFT JOIN pg11_reached_nodes r
 ON r.stable_symbol_id = w.caller_stable_symbol_id AND r.file_path = w.file_path
LEFT JOIN labels l ON l.source_stable_id = w.caller_stable_symbol_id
                 AND l.target_label = w.normalized_macro_path
GROUP BY ALL;

SELECT hops, caller_name, normalized_macro_path AS macro_path,
       line_start, debug_assertions_candidate AS debug_candidate,
       test_context, resolution, matching_macro_call_label_records
FROM pg11_witness_reach ORDER BY hops, file_path, line_start;

hops,caller_name,macro_path,line_start,debug_candidate,test_context,resolution,matching_macro_call_label_records
3,impl SharedParseCache::evict_before,debug_assert,2095,True,unspecified,"{""kind"":""unresolved""}",0
3,impl SharedParseCache::evict_before,debug_assert,2096,True,unspecified,"{""kind"":""unresolved""}",0
4,impl OccupancyCounter::decrement,debug_assert,244,True,unspecified,"{""kind"":""unresolved""}",0
5,parse_raw_diff,unreachable,3413,False,unspecified,"{""kind"":""unresolved""}",0
5,import_licensed_call_target,debug_assert_eq,2033,True,unspecified,"{""kind"":""unresolved""}",0
5,import_licensed_call_target,debug_assert,2037,True,unspecified,"{""kind"":""unresolved""}",0


In [43]:
CREATE OR REPLACE TEMP TABLE pg11_method_frontier AS
WITH labels AS (
  SELECT source_stable_id, target_label, confidence, bind_method,
         receiver_text, edge_kind, 'resolved target recorded' AS state
  FROM pg11_edges WHERE relation = 'calls'
  UNION ALL
  SELECT source_stable_id, target_label, confidence, bind_method,
         receiver_text, edge_kind, 'unresolved target label' AS state
  FROM pg11_unresolved WHERE relation = 'calls'
)
SELECT r.hops, r.stable_symbol_id AS caller_id, r.qualified_name AS caller_name,
       r.file_path, l.target_label, l.state, l.confidence, l.bind_method,
       l.receiver_text, l.edge_kind
FROM pg11_reached_nodes r JOIN labels l ON l.source_stable_id = r.stable_symbol_id
WHERE r.file_path LIKE 'crates/spur-graph/src/%'
  AND l.target_label IN ('unwrap', 'expect');

-- Edge records can collapse multiple invocations; these are not call-site counts.
SELECT state, target_label, count(*) AS edge_records,
       count(DISTINCT caller_id) AS distinct_callers,
       min(hops) AS nearest_hops, max(hops) AS farthest_hops
FROM pg11_method_frontier
GROUP BY state, target_label ORDER BY state, target_label;

state,target_label,edge_records,distinct_callers,nearest_hops,farthest_hops
unresolved target label,expect,6,6,2,5
unresolved target label,unwrap,1,1,3,3


In [44]:
CREATE OR REPLACE TEMP TABLE pg11_context_counts AS
WITH corpus AS (
 SELECT 'workspace' AS scope, * FROM pg11_macro_sites
 UNION ALL
 SELECT 'spur-graph/src' AS scope, * FROM pg11_macro_sites
 WHERE file_path LIKE 'crates/spur-graph/src/%'
 UNION ALL
 SELECT 'five-hop spur-graph/src' AS scope, m.* EXCLUDE (hops, caller_name)
 FROM pg11_reached_macro_sites m
 WHERE file_path LIKE 'crates/spur-graph/src/%'
)
SELECT scope, test_context,
       CASE WHEN debug_assertions_candidate THEN 'debug-assert name candidate'
            WHEN sink_family_candidate IS NOT NULL THEN 'other sink-name candidate'
            ELSE 'other macro syntax' END AS evidence_class,
       count(*) AS invocation_sites,
       count(DISTINCT file_path) AS files,
       count(*) FILTER (WHERE caller_stable_symbol_id IS NULL) AS without_function_caller
FROM corpus GROUP BY scope, test_context, evidence_class;

SELECT * FROM pg11_context_counts
ORDER BY scope, evidence_class, test_context;

scope,test_context,evidence_class,invocation_sites,files,without_function_caller
five-hop spur-graph/src,unspecified,debug-assert name candidate,5,2,0
five-hop spur-graph/src,unspecified,other macro syntax,264,14,0
five-hop spur-graph/src,unspecified,other sink-name candidate,1,1,0
spur-graph/src,conditional,debug-assert name candidate,1,1,0
spur-graph/src,unspecified,debug-assert name candidate,11,6,0
spur-graph/src,conditional,other macro syntax,594,29,5
spur-graph/src,unspecified,other macro syntax,1644,51,53
spur-graph/src,conditional,other sink-name candidate,1978,33,0
spur-graph/src,unspecified,other sink-name candidate,28,7,0
workspace,conditional,debug-assert name candidate,2,2,0


In [45]:
import hashlib
import json
from pathlib import Path

pg11_repo = Path('/Users/kevintruong/project/SpurCode').resolve()
pg11_live_sources = []
pg11_source_bytes = {}
for path, expected_oid in duckdb.sql("""
 SELECT path, content_oid FROM pg11_source_manifest
 WHERE lower(path) LIKE '%.rs' ORDER BY path
""").fetchall():
    physical = pg11_repo / path
    try:
        data = physical.read_bytes()
        actual_oid = hashlib.sha1(b'blob ' + str(len(data)).encode() + b'\0' + data).hexdigest()
        state = 'match' if actual_oid == expected_oid else 'changed'
        pg11_source_bytes[path] = data
        pg11_live_sources.append((path, expected_oid, actual_oid, len(data), state))
    except OSError:
        pg11_live_sources.append((path, expected_oid, None, None, 'unavailable'))

duckdb.sql("""CREATE OR REPLACE TEMP TABLE pg11_live_sources(
 file_path VARCHAR, indexed_oid VARCHAR, live_oid VARCHAR, byte_length BIGINT, state VARCHAR)""")
duckdb.executemany("INSERT INTO pg11_live_sources VALUES (?, ?, ?, ?, ?)", pg11_live_sources)

pg11_witness_source_checks = []
for row in duckdb.sql("""
 SELECT site_id, caller_stable_symbol_id, caller_name, file_path, source_oid,
        byte_range_start, byte_range_end, line_start, line_end,
        normalized_macro_path
 FROM pg11_witnesses ORDER BY file_path, line_start
""").fetchall():
    site, caller, name, path, oid, start, end, first, last, macro = row
    data = pg11_source_bytes[path]
    actual_oid = hashlib.sha1(b'blob ' + str(len(data)).encode() + b'\0' + data).hexdigest()
    assert actual_oid == oid
    assert first == data[:start].count(b'\n') + 1
    assert last == data[:end - 1].count(b'\n') + 1
    pg11_witness_source_checks.append(dict(
        site_id=site, caller_id=caller, caller_name=name, file_path=path,
        source_oid=oid, byte_start=start, byte_end=end,
        line_start=first, line_end=last, macro=macro,
        exact_source=data[start:end].decode('utf-8')))

assert len(pg11_witness_source_checks) == 6
assert len({r['site_id'] for r in pg11_witness_source_checks}) == 6
print(json.dumps({
 'physical_rust_source_bindings': duckdb.sql(
    "SELECT state, count(*) FROM pg11_live_sources GROUP BY state ORDER BY state").fetchall(),
 'witness_source_and_coordinate_checks': pg11_witness_source_checks,
 'source_read_reason': 'Exact raw byte/OID/span verification; per-symbol MCP graph lookup is unavailable.'
}, indent=2))

{
  "physical_rust_source_bindings": [
    [
      "match",
      1510
    ]
  ],
  "witness_source_and_coordinate_checks": [
    {
      "site_id": "376a54c55803b2d6f334c1b3012db1306fd9925531c46ec524b1bd325610a342",
      "caller_id": "4164ec18cc1ebc3d",
      "caller_name": "impl OccupancyCounter::decrement",
      "file_path": "crates/spur-graph/src/git_walk.rs",
      "source_oid": "96681b490e68b05eed4d958894cdf4b7fa7f870d",
      "byte_start": 7715,
      "byte_end": 7782,
      "line_start": 244,
      "line_end": 244,
      "macro": "debug_assert",
      "exact_source": "debug_assert!(previous >= count, \"worker-pool occupancy underflow\")"
    },
    {
      "site_id": "832881b51acbe4a2ec2c4e5cbad15fcf9a49a42f8f866cec88c809cc6ea908f8",
      "caller_id": "bdf5193c1b04ff4f",
      "caller_name": "impl SharedParseCache::evict_before",
      "file_path": "crates/spur-graph/src/git_walk.rs",
      "source_oid": "96681b490e68b05eed4d958894cdf4b7fa7f870d",
      "byte_start": 77465,


### Controlled compiler experiment: checks beyond macro syntax

A separate six-case fixture tests ordinary assertions, debug assertions, an explicit panic, slice indexing, `u8` addition, and `Option::unwrap`. It is **not part of the workspace snapshot**. We will hold optimization at zero and the panic strategy at unwind, then vary debug assertions and overflow checks independently across four configurations. Good-input controls and deliberately failing inputs will be observed in isolated child processes.

The [rustc output options](https://doc.rust-lang.org/rustc/command-line-arguments.html#--emit-specifies-the-types-of-output-files-to-generate) document `--emit=mir`; the [codegen options](https://doc.rust-lang.org/rustc/codegen-options/) document the two switches. The harness uses [`catch_unwind`](https://doc.rust-lang.org/stable/std/panic/fn.catch_unwind.html), so the explicit unwind strategy is part of the experiment. MIR text observations are pinned to the measured compiler version and fixture; they are not a general MIR parser or whole-program panic proof. No CGA run is claimed.

In [47]:
-- Existing shellfs extension: record the exact compiler without installing tools.
SELECT line FROM read_csv(
  $$/Users/kevintruong/.cargo/bin/rustup run 1.94.1 rustc --version --verbose |$$,
  columns = {'line': 'VARCHAR'}, header = false, delim = '\t');

line
rustc 1.94.1 (e408947bf 2026-03-25)
binary: rustc
commit-hash: e408947bfd200af42db322daf0fadfe7e26d3bd1
commit-date: 2026-03-25
host: aarch64-apple-darwin
release: 1.94.1
LLVM version: 21.1.8


In [49]:
import os
import subprocess
import tempfile
import time

pg11_evidence_dir = pg11_repo / 'docs/research/artifacts/2026-10-31-panic-graph-64c5b7a163b7'
pg11_evidence_dir.mkdir(parents=True, exist_ok=True)
pg11_fixture_root = Path(tempfile.mkdtemp(prefix='spur-panic-fixture-64c5b7a1-', dir='/private/tmp'))
pg11_fixture_files = {"src/lib.rs":"#[inline(never)]\npub fn direct_assert(value: u8) { assert!(value > 0); }\n#[inline(never)]\npub fn debug_assertion(value: u8) { debug_assert!(value > 0); }\n#[inline(never)]\npub fn explicit_panic(enabled: bool) { if enabled { panic!(\"controlled explicit panic\"); } }\n#[inline(never)]\npub fn index_slice(values: &[u8], index: usize) -> u8 { values[index] }\n#[inline(never)]\npub fn add_byte(value: u8) -> u8 { value + 1 }\n#[inline(never)]\npub fn unwrap_option(value: Option<u8>) -> u8 { value.unwrap() }\n\npub fn entry(value: u8, index: usize, enabled: bool, optional: Option<u8>, values: &[u8]) -> u8 {\n    direct_assert(value);\n    debug_assertion(value);\n    explicit_panic(enabled);\n    index_slice(values, index) ^ add_byte(value) ^ unwrap_option(optional)\n}\n","src/main.rs":"use panic_fixture::*;\nfn caught(action: impl FnOnce() + std::panic::UnwindSafe) -> bool {\n    std::panic::catch_unwind(action).is_err()\n}\nfn main() {\n    std::panic::set_hook(Box::new(|_| {}));\n    let safe_ok = !caught(|| {\n        direct_assert(1);\n        debug_assertion(1);\n        explicit_panic(false);\n        let _ = index_slice(&[1], 0);\n        let _ = add_byte(0);\n        let _ = unwrap_option(Some(1));\n    });\n    println!(r#\"{{\"direct_assert\":{},\"debug_assertion\":{},\"explicit_panic\":{},\"index_slice\":{},\"add_byte\":{},\"unwrap_option\":{},\"safe_controls_all_ok\":{}}}\"#,\n        caught(|| direct_assert(0)),\n        caught(|| debug_assertion(0)),\n        caught(|| explicit_panic(true)),\n        caught(|| { let _ = index_slice(&[], 0); }),\n        caught(|| { let _ = add_byte(255); }),\n        caught(|| { let _ = unwrap_option(None); }),\n        safe_ok);\n}\n","Cargo.toml":"[package]\nname = \"panic_fixture\"\nversion = \"0.0.0\"\nedition = \"2021\"\n\n[workspace]\n\n[lib]\npath = \"src/lib.rs\"\n\n[[bin]]\nname = \"panic-probe\"\npath = \"src/main.rs\"\n\n[profile.dev]\npanic = \"unwind\"\nopt-level = 0\ndebug = 0\n","rust-toolchain.toml":"[toolchain]\nchannel = \"1.94.1\"\n"}
for name, contents in pg11_fixture_files.items():
    path = pg11_fixture_root / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents)
pg11_fixture_env = dict(os.environ)
for key in ('SPUR_GRAPH_ARTIFACT_DIR', 'SPUR_CODE_GRAPH_INDEX',
            'SPUR_GRAPH_SKIP_ANALYST', 'SPUR_ANALYST_DB',
            'SPUR_GRAPH_WITH_TEMPORAL', 'SPUR_GRAPH_TEMPORAL_JOBS',
            'CARGO_ENCODED_RUSTFLAGS'):
    pg11_fixture_env.pop(key, None)
pg11_fixture_env['SPUR_REMOTE'] = '0'
pg11_fixture_commands = []
def pg11_fixture_run(args, env=None):
    started = time.monotonic()
    result = subprocess.run(args, cwd=pg11_fixture_root,
                            env=env or pg11_fixture_env,
                            capture_output=True, text=True, timeout=60)
    pg11_fixture_commands.append(dict(
        argv=[str(a) for a in args], exit_code=result.returncode,
        seconds=time.monotonic() - started))
    if result.returncode:
        raise RuntimeError(f"fixture command failed: {args}\n{result.stderr[-4000:]}")
    return result

pg11_fixture_run(['git', 'init', '--quiet'])
pg11_fixture_run(['git', 'config', 'user.name', 'SPUR notebook experiment'])
pg11_fixture_run(['git', 'config', 'user.email', 'notebook@local.invalid'])
pg11_fixture_run(['git', 'add', '--', *pg11_fixture_files.keys()])
pg11_fixture_run(['git', 'commit', '--quiet', '-m',
                  'test(panic-fixture): E1 record controlled panic cases'])
pg11_compiler_version = pg11_fixture_run([
 '/Users/kevintruong/.cargo/bin/rustup', 'run', '1.94.1',
 'rustc', '--version', '--verbose']).stdout

pg11_fixture_graph_run = pg11_fixture_run([
 '/Users/kevintruong/.cargo/bin/spur', 'graph', 'build',
 '--root', str(pg11_fixture_root), '--no-analyst',
 '--no-section-embeddings', '--no-code-symbol-embeddings'])
pg11_fixture_artifact = (pg11_fixture_root / '.spur/graph/CURRENT').resolve()
pg11_fixture_manifest = json.loads((pg11_fixture_artifact / 'manifest.json').read_text())
assert pg11_fixture_manifest['schema_version'] == 'spur-graph-schema-v11'
assert json.loads((pg11_repo / '.spur/graph/CURRENT/manifest.json').read_text())['graph_content_hash'] == pg11_meta['graph_content_hash']

for name, file in [('sites', 'macro_invocations'), ('nodes', 'nodes')]:
    location = str(pg11_fixture_artifact / (file + '.parquet')).replace("'", "''")
    duckdb.sql(f"CREATE OR REPLACE TEMP TABLE pg11_fixture_{name} AS SELECT * FROM read_parquet('{location}')")
    schema = duckdb.sql(f"DESCRIBE pg11_fixture_{name}").fetchall()
    assert schema, 'fixture schema discovery must succeed'
pg11_fixture_macros = duckdb.sql("""
 SELECT n.entity_name AS case_name, m.normalized_macro_path AS macro_path,
        m.site_id, m.source_oid, m.line_start, m.line_end
 FROM pg11_fixture_sites m JOIN pg11_fixture_nodes n
  ON n.stable_symbol_id = m.caller_stable_symbol_id AND n.file_path = m.file_path
 WHERE m.file_path = 'src/lib.rs'
 ORDER BY case_name, m.line_start
""").fetchall()
assert len(pg11_fixture_macros) == 3
(pg11_evidence_dir / 'fixture-source.json').write_text(json.dumps({
 'files': pg11_fixture_files,
 'sha256': {name: hashlib.sha256(text.encode()).hexdigest()
            for name, text in pg11_fixture_files.items()}
}, indent=2) + '\n')
(pg11_evidence_dir / 'rustc-version.txt').write_text(pg11_compiler_version)
print(json.dumps({
 'fixture_root': str(pg11_fixture_root),
 'fixture_graph_hash': pg11_fixture_manifest['graph_content_hash'],
 'library_macro_sites': pg11_fixture_macros,
 'workspace_graph_unchanged': True,
 'compiler': pg11_compiler_version
}, indent=2))

{
  "fixture_root": "/private/tmp/spur-panic-fixture-64c5b7a1-krtrqe78",
  "fixture_graph_hash": "907b772421db3874d4581cdedf9fdbba8af3480253c9079566d8665895f3fc90",
  "library_macro_sites": [
    [
      "debug_assertion",
      "debug_assert",
      "24886b9f59648f2482ee68dc301784742d9589a13c2585b50188b9bb1756d936",
      "b538e4c2521bccdf459fc696c5f95ca515bef3cc",
      4,
      4
    ],
    [
      "direct_assert",
      "assert",
      "129fb42bc386b2a4504a9597f4c576b2c0bbc8fcdf0b601398842c929badd689",
      "b538e4c2521bccdf459fc696c5f95ca515bef3cc",
      2,
      2
    ],
    [
      "explicit_panic",
      "panic",
      "e32832bccbc7635db5076690dc5dc35ea678446f0b4a2015a3f5da48fcb3e112",
      "b538e4c2521bccdf459fc696c5f95ca515bef3cc",
      6,
      6
    ]
  ],
  "workspace_graph_unchanged": true,
  "compiler": "rustc 1.94.1 (e408947bf 2026-03-25)\nbinary: rustc\ncommit-hash: e408947bfd200af42db322daf0fadfe7e26d3bd1\ncommit-date: 2026-03-25\nhost: aarch64-apple-darwin\nrelea

In [50]:
pg11_mir_profiles = []
pg11_mir_texts = {}
pg11_wrapper = pg11_repo / 'scripts/spur-cargo'

for debug_assertions in (False, True):
    for overflow_checks in (False, True):
        label = f'd{int(debug_assertions)}_o{int(overflow_checks)}'
        flags = (f"-C debug-assertions={'yes' if debug_assertions else 'no'} "
                 f"-C overflow-checks={'yes' if overflow_checks else 'no'} "
                 "-C panic=unwind -C opt-level=0")
        build_env = dict(pg11_fixture_env)
        build_env['RUSTFLAGS'] = flags
        build_env['RUSTUP_TOOLCHAIN'] = '1.94.1'
        target = pg11_fixture_root / 'target' / label
        mir_path = pg11_evidence_dir / (label + '.mir')
        pg11_fixture_run([
            str(pg11_wrapper), '--workdir', str(pg11_fixture_root),
            'rustc', '--offline', '--lib', '--target-dir', str(target),
            '--', '--emit=mir=' + str(mir_path)], env=build_env)
        probe = pg11_fixture_run([
            str(pg11_wrapper), '--workdir', str(pg11_fixture_root),
            'run', '--offline', '--quiet', '--bin', 'panic-probe',
            '--target-dir', str(target)], env=build_env)
        observed = json.loads(probe.stdout)
        pg11_mir_texts[label] = mir_path.read_text()
        pg11_mir_profiles.append(dict(
            profile=label, debug_assertions=debug_assertions,
            overflow_checks=overflow_checks, panic_strategy='unwind',
            opt_level=0, rustflags=flags, runtime=observed,
            mir_file=mir_path.name, mir_sha256=hashlib.sha256(mir_path.read_bytes()).hexdigest(),
            mir_bytes=mir_path.stat().st_size))
        assert observed['safe_controls_all_ok'], 'all six good-input controls must finish'

(pg11_evidence_dir / 'compiler-profiles.json').write_text(
    json.dumps(pg11_mir_profiles, indent=2) + '\n')
(pg11_evidence_dir / 'fixture-commands.json').write_text(
    json.dumps(pg11_fixture_commands, indent=2) + '\n')
print(json.dumps(pg11_mir_profiles, indent=2))

[
  {
    "profile": "d0_o0",
    "debug_assertions": false,
    "overflow_checks": false,
    "panic_strategy": "unwind",
    "opt_level": 0,
    "rustflags": "-C debug-assertions=no -C overflow-checks=no -C panic=unwind -C opt-level=0",
    "runtime": {
      "direct_assert": true,
      "debug_assertion": false,
      "explicit_panic": true,
      "index_slice": true,
      "add_byte": false,
      "unwrap_option": true,
      "safe_controls_all_ok": true
    },
    "mir_file": "d0_o0.mir",
    "mir_sha256": "6b8206305852384f2acafaf9349cb38896d854cae05c43d0e264eeca1901caaf",
    "mir_bytes": 3346
  },
  {
    "profile": "d0_o1",
    "debug_assertions": false,
    "overflow_checks": true,
    "panic_strategy": "unwind",
    "opt_level": 0,
    "rustflags": "-C debug-assertions=no -C overflow-checks=yes -C panic=unwind -C opt-level=0",
    "runtime": {
      "direct_assert": true,
      "debug_assertion": false,
      "explicit_panic": true,
      "index_slice": true,
      "add_byte"

In [51]:
# Inspect the measured compiler's human-readable MIR before defining the fixture-only observations.
print(pg11_mir_texts['d1_o1'])

// WARNING: This output format is intended for human consumers only
// and is subject to change without notice. Knock yourself out.
// HINT: See also -Z dump-mir for MIR at specific points during compilation.
fn direct_assert(_1: u8) -> () {
    debug value => _1;
    let mut _0: ();
    let mut _2: bool;
    let mut _3: !;

    bb0: {
        _2 = Gt(copy _1, const 0_u8);
        switchInt(move _2) -> [0: bb2, otherwise: bb1];
    }

    bb1: {
        return;
    }

    bb2: {
        _3 = core::panicking::panic(const "assertion failed: value > 0") -> unwind continue;
    }
}

alloc1 (size: 27, align: 1) {
    0x00 │ 61 73 73 65 72 74 69 6f 6e 20 66 61 69 6c 65 64 │ assertion failed
    0x10 │ 3a 20 76 61 6c 75 65 20 3e 20 30                │ : value > 0
}

fn debug_assertion(_1: u8) -> () {
    debug value => _1;
    let mut _0: ();
    let mut _2: bool;
    let mut _3: !;

    bb0: {
        _2 = Gt(copy _1, const 0_u8);
        switchInt(move _2) -> [0: bb1, otherwise: bb2];
    }

In [57]:
import re

pg11_fixture_cases = ('direct_assert', 'debug_assertion', 'explicit_panic',
                      'index_slice', 'add_byte', 'unwrap_option')
for relation, file in [('edges', 'edges'), ('unresolved', 'edges_unresolved')]:
    location = str(pg11_fixture_artifact / (file + '.parquet')).replace("'", "''")
    duckdb.sql(f"CREATE OR REPLACE TEMP TABLE pg11_fixture_{relation} AS SELECT * FROM read_parquet('{location}')")
    assert duckdb.sql(f"DESCRIBE pg11_fixture_{relation}").fetchall()

pg11_fixture_unwrap_records = dict(duckdb.sql("""
 WITH labels AS (
  SELECT source_stable_id, target_label FROM pg11_fixture_edges WHERE relation = 'calls'
  UNION ALL
  SELECT source_stable_id, target_label FROM pg11_fixture_unresolved WHERE relation = 'calls'
 )
 SELECT n.entity_name, count(*)
 FROM labels l JOIN pg11_fixture_nodes n ON n.stable_symbol_id = l.source_stable_id
 WHERE n.file_path = 'src/lib.rs' AND l.target_label = 'unwrap'
 GROUP BY n.entity_name
""").fetchall())
pg11_compiler_observations = []
for profile in pg11_mir_profiles:
    text = pg11_mir_texts[profile['profile']]
    headings = list(re.finditer(r'(?m)^fn ([A-Za-z0-9_]+)\(', text))
    blocks = {m.group(1): text[m.start():headings[i+1].start() if i+1 < len(headings) else len(text)]
              for i, m in enumerate(headings)}
    assert all(case in blocks for case in pg11_fixture_cases)
    for case in pg11_fixture_cases:
        block = blocks[case]
        lines = [line.strip() for line in block.splitlines()]
        panic_calls = [line for line in lines
                       if re.search(r'=\s*(core::panicking::panic|panic_fmt|begin_panic|assert_failed)\(', line)]
        bounds = [line for line in lines if line.startswith('assert(') and 'index out of bounds' in line]
        overflow = [line for line in lines if line.startswith('assert(') and 'overflow' in line]
        unwrap = [line for line in lines if re.search(r'=\s*Option::<u8>::unwrap\(', line)]
        markers = []
        if panic_calls: markers.append('panic call')
        if bounds: markers.append('bounds Assert')
        if overflow: markers.append('overflow Assert')
        if unwrap: markers.append('typed Option::unwrap call')
        pg11_compiler_observations.append(dict(
            profile=profile['profile'], debug_assertions=profile['debug_assertions'],
            overflow_checks=profile['overflow_checks'], case_name=case,
            graph_macro_sites=sum(row[0] == case for row in pg11_fixture_macros),
            graph_unwrap_label_records=pg11_fixture_unwrap_records.get(case, 0),
            mir_panic_calls=len(panic_calls), mir_bounds_asserts=len(bounds),
            mir_overflow_asserts=len(overflow), mir_typed_unwrap_calls=len(unwrap),
            bad_input_panicked=profile['runtime'][case],
            good_controls_passed=profile['runtime']['safe_controls_all_ok'],
            mir_evidence=', '.join(markers) if markers else 'none in emitted function MIR',
            matched_mir_lines=panic_calls + bounds + overflow + unwrap))
duckdb.sql("""CREATE OR REPLACE TEMP TABLE pg11_compiler_observations(
 profile VARCHAR, debug_assertions BOOLEAN, overflow_checks BOOLEAN, case_name VARCHAR,
 graph_macro_sites BIGINT, graph_unwrap_label_records BIGINT, mir_panic_calls BIGINT,
 mir_bounds_asserts BIGINT, mir_overflow_asserts BIGINT, mir_typed_unwrap_calls BIGINT,
 bad_input_panicked BOOLEAN, good_controls_passed BOOLEAN, mir_evidence VARCHAR)""")
pg11_observation_fields = ('profile', 'debug_assertions', 'overflow_checks', 'case_name',
 'graph_macro_sites', 'graph_unwrap_label_records', 'mir_panic_calls', 'mir_bounds_asserts',
 'mir_overflow_asserts', 'mir_typed_unwrap_calls', 'bad_input_panicked',
 'good_controls_passed', 'mir_evidence')
duckdb.executemany("INSERT INTO pg11_compiler_observations VALUES (" + ','.join('?' for _ in pg11_observation_fields) + ")",
                  [tuple(row[key] for key in pg11_observation_fields) for row in pg11_compiler_observations])
(pg11_evidence_dir / 'compiler-observations.json').write_text(
 json.dumps(pg11_compiler_observations, indent=2) + '\n')
print(json.dumps({
 'compiler_profiles': len(pg11_mir_profiles),
 'function_profile_observations': len(pg11_compiler_observations),
 'profiles_with_all_six_good_inputs_ok': sum(p['runtime']['safe_controls_all_ok'] for p in pg11_mir_profiles),
 'individual_good_input_cases': len(pg11_fixture_cases) * sum(p['runtime']['safe_controls_all_ok'] for p in pg11_mir_profiles),
 'fixture_library_macro_sites': len(pg11_fixture_macros),
 'implicit_check_families_observed': sorted({label for row in pg11_compiler_observations
     for key, label in [('mir_bounds_asserts', 'bounds Assert'), ('mir_overflow_asserts', 'overflow Assert')]
     if row[key] > 0}),
 'parser_scope': 'This six-case fixture and measured rustc MIR text only.'
}, indent=2))

{
  "compiler_profiles": 4,
  "function_profile_observations": 24,
  "profiles_with_all_six_good_inputs_ok": 4,
  "individual_good_input_cases": 24,
  "fixture_library_macro_sites": 3,
  "implicit_check_families_observed": [
    "bounds Assert",
    "overflow Assert"
  ],
  "parser_scope": "This six-case fixture and measured rustc MIR text only."
}


In [58]:
-- Boolean columns mean "the deliberately failing input panicked" in that configuration.
-- All six good-input controls completed in every configuration.
SELECT case_name,
       max(graph_macro_sites) AS graph_macro_sites,
       max(graph_unwrap_label_records) AS graph_unwrap_label_records,
       string_agg(DISTINCT mir_evidence, ' | ' ORDER BY mir_evidence) AS measured_mir_evidence,
       bool_or(bad_input_panicked) FILTER (WHERE profile = 'd0_o0') AS d0_o0_panicked,
       bool_or(bad_input_panicked) FILTER (WHERE profile = 'd0_o1') AS d0_o1_panicked,
       bool_or(bad_input_panicked) FILTER (WHERE profile = 'd1_o0') AS d1_o0_panicked,
       bool_or(bad_input_panicked) FILTER (WHERE profile = 'd1_o1') AS d1_o1_panicked
FROM pg11_compiler_observations
GROUP BY case_name ORDER BY case_name;

case_name,graph_macro_sites,graph_unwrap_label_records,measured_mir_evidence,d0_o0_panicked,d0_o1_panicked,d1_o0_panicked,d1_o1_panicked
add_byte,0,0,none in emitted function MIR | overflow Assert,False,True,False,True
debug_assertion,1,0,none in emitted function MIR | panic call,False,False,True,True
direct_assert,1,0,panic call,True,True,True,True
explicit_panic,1,0,panic call,True,True,True,True
index_slice,0,0,bounds Assert,True,True,True,True
unwrap_option,0,1,typed Option::unwrap call,True,True,True,True


In [54]:
-- Cross-check the recursive CTE with the already-loaded routing extension.
-- duckrouting prepares its edge query on another connection, so use the pinned Parquet input.
SET VARIABLE pg11_route_root = (SELECT node_id FROM pg11_root);
SET VARIABLE pg11_route_target = (
 SELECT node_id FROM pg11_nodes
 WHERE file_path = 'crates/spur-graph/src/git_walk.rs' AND qualified_name = 'parse_raw_diff'
);
CREATE OR REPLACE TEMP TABLE pg11_raw_diff_path AS
SELECT seq, node, agg_cost AS hops
FROM duckrouting_dijkstra(
 $$SELECT row_number() OVER ()::BIGINT AS id, src_id AS source, dst_id AS target,
          1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
   FROM read_parquet('/Users/kevintruong/project/SpurCode/.spur/graph/64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4.parquet/edges.parquet')
   WHERE relation = 'calls' AND edge_kind IN ('calls', 'calls_dyn')
     AND src_id IS NOT NULL AND dst_id IS NOT NULL$$,
 getvariable('pg11_route_root'), getvariable('pg11_route_target')
);

WITH path AS (
 SELECT *, lead(node) OVER (ORDER BY seq) AS next_node FROM pg11_raw_diff_path
)
SELECT p.seq, p.hops, n.qualified_name, n.file_path,
       string_agg(DISTINCT e.edge_kind || ':' || e.confidence || ':' ||
                  coalesce(e.bind_method, 'unspecified'), ', ') AS outgoing_graph_evidence
FROM path p JOIN pg11_nodes n ON n.node_id = p.node
LEFT JOIN pg11_edges e ON e.src_id = p.node AND e.dst_id = p.next_node
 AND e.relation = 'calls' AND e.edge_kind IN ('calls', 'calls_dyn')
GROUP BY p.seq, p.hops, n.qualified_name, n.file_path ORDER BY p.seq;

seq,hops,qualified_name,file_path,outgoing_graph_evidence
1,0.0,build_with_temporal_jobs_override,crates/spur-cli/src/commands/graph.rs,calls:syntax_exact:import_licensed
2,1.0,run_full_walk_into,crates/spur-graph/src/git_walk.rs,calls:syntax_exact:singleton
3,2.0,run_full_walk_into_with_stats,crates/spur-graph/src/git_walk.rs,calls:syntax_exact:singleton
4,3.0,compute_commit_walk_result,crates/spur-graph/src/git_walk.rs,calls:syntax_exact:singleton
5,4.0,file_changes_for_commit,crates/spur-graph/src/git_walk.rs,calls:syntax_exact:singleton
6,5.0,parse_raw_diff,crates/spur-graph/src/git_walk.rs,NaN


In [55]:
import copy

def pg11_records(query):
    cursor = duckdb.execute(query)
    keys = [column[0] for column in cursor.description]
    return [dict(zip(keys, values)) for values in cursor.fetchall()]

pg11_catalog_observations = dict(
 snapshot=pg11_meta['graph_content_hash'],
 sites=pg11_records("""
  SELECT site_id AS site, caller_stable_symbol_id AS caller,
         enclosing_scope_id AS scope, file_path, source_oid,
         file_path || '@' || source_oid AS source_key
  FROM pg11_witnesses ORDER BY site_id
 """),
 callers=pg11_records("""
  SELECT n.stable_symbol_id AS caller, s.scope_id AS scope,
         n.file_path || '@' || s.source_oid AS source_key
  FROM pg11_nodes n JOIN pg11_macro_scopes s
   ON s.scope_id = n.stable_symbol_id AND s.kind = 'function'
    AND s.file_path = n.file_path
  WHERE n.stable_symbol_id IN (SELECT caller_stable_symbol_id FROM pg11_witnesses)
  ORDER BY n.stable_symbol_id
 """),
 sources=pg11_records("""
  SELECT path || '@' || content_oid AS source_key
  FROM pg11_source_manifest
  WHERE path IN (SELECT DISTINCT file_path FROM pg11_witnesses)
  ORDER BY path
 """))
base = [dict(row, snapshot=pg11_catalog_observations['snapshot'])
        for row in pg11_catalog_observations['sites']]
variants = {'observed': copy.deepcopy(base),
            'missing_site': copy.deepcopy(base[:-1]),
            'duplicate_site': copy.deepcopy(base + [base[0]]),
            'wrong_existing_caller': copy.deepcopy(base),
            'stale_source': copy.deepcopy(base),
            'mixed_snapshot': copy.deepcopy(base)}
alternate = next(row['caller'] for row in base
                 if row['source_key'] == base[0]['source_key']
                    and row['caller'] != base[0]['caller'])
variants['wrong_existing_caller'][0]['caller'] = alternate
variants['stale_source'][0]['source_oid'] = '0' * 40
variants['stale_source'][0]['source_key'] = base[0]['file_path'] + '@' + '0' * 40
variants['mixed_snapshot'][0]['snapshot'] = '37ef5600b79894ce8d016e8c6eb35dbfc5b62f051516e3754861702d1da2f014'

duckdb.sql("""CREATE OR REPLACE TEMP TABLE pg11_control_rows(
 control VARCHAR, snapshot VARCHAR, site_id VARCHAR, caller_id VARCHAR,
 scope_id VARCHAR, file_path VARCHAR, source_oid VARCHAR)""")
duckdb.executemany("INSERT INTO pg11_control_rows VALUES (?, ?, ?, ?, ?, ?, ?)",
 [(control, row['snapshot'], row['site'], row['caller'], row['scope'],
   row['file_path'], row['source_oid'])
  for control, rows in variants.items() for row in rows])

duckdb.sql("""
 CREATE OR REPLACE TEMP TABLE pg11_control_matrix AS
 WITH audited AS (
  SELECT v.control, count(*) AS rows,
         count(*) - count(DISTINCT v.site_id) AS duplicate_site_keys,
         count(*) FILTER (WHERE c.stable_symbol_id IS NULL OR c.symbol_kind NOT IN ('function', 'method'))
           AS invalid_caller_references,
         count(*) FILTER (WHERE v.caller_id <> v.scope_id) AS caller_scope_conflicts,
         count(*) FILTER (WHERE f.path IS NULL) AS invalid_source_references,
         count(*) FILTER (WHERE v.snapshot <> (SELECT graph_content_hash FROM pg11_manifest))
           AS mixed_snapshot_rows
  FROM pg11_control_rows v
  LEFT JOIN pg11_nodes c ON c.stable_symbol_id = v.caller_id AND c.file_path = v.file_path
  LEFT JOIN pg11_source_manifest f ON f.path = v.file_path AND f.content_oid = v.source_oid
  GROUP BY v.control
 )
 SELECT *, CASE WHEN rows = 6 AND duplicate_site_keys = 0
                     AND invalid_caller_references = 0 AND caller_scope_conflicts = 0
                     AND invalid_source_references = 0 AND mixed_snapshot_rows = 0
                THEN 'accept' ELSE 'reject' END AS decision
 FROM audited
""")
assert duckdb.sql("SELECT count(*) FROM pg11_control_matrix WHERE decision='accept' AND control='observed'").fetchone()[0] == 1
assert duckdb.sql("SELECT count(*) FROM pg11_control_matrix WHERE decision='reject' AND control<>'observed'").fetchone()[0] == 5
print(json.dumps(pg11_catalog_observations, indent=2))

{
  "snapshot": "64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4",
  "sites": [
    {
      "site": "1d2ac2833cf6dc1d3b7fa3601fc28d7184148fb00f29a97f0f44e996bc8354e9",
      "caller": "4036cf39bb8ba8b5",
      "scope": "4036cf39bb8ba8b5",
      "file_path": "crates/spur-graph/src/git_walk.rs",
      "source_oid": "96681b490e68b05eed4d958894cdf4b7fa7f870d",
      "source_key": "crates/spur-graph/src/git_walk.rs@96681b490e68b05eed4d958894cdf4b7fa7f870d"
    },
    {
      "site": "376a54c55803b2d6f334c1b3012db1306fd9925531c46ec524b1bd325610a342",
      "caller": "4164ec18cc1ebc3d",
      "scope": "4164ec18cc1ebc3d",
      "file_path": "crates/spur-graph/src/git_walk.rs",
      "source_oid": "96681b490e68b05eed4d958894cdf4b7fa7f870d",
      "source_key": "crates/spur-graph/src/git_walk.rs@96681b490e68b05eed4d958894cdf4b7fa7f870d"
    },
    {
      "site": "4adf531cd4dcbee501afe4567e913d5d23415c2c5a384334714c83069de995a3",
      "caller": "0e644eabb44926f9",
      "scope"

In [56]:
-- Corrupted copies stay inside this experiment; the source graph tables are unchanged.
SELECT * FROM pg11_control_matrix ORDER BY control;

control,rows,duplicate_site_keys,invalid_caller_references,caller_scope_conflicts,invalid_source_references,mixed_snapshot_rows,decision
duplicate_site,7,1,0,0,0,0,reject
missing_site,5,0,0,0,0,0,reject
mixed_snapshot,6,0,0,0,0,1,reject
observed,6,0,0,0,0,0,accept
stale_source,6,0,0,0,1,0,reject
wrong_existing_caller,6,0,0,1,0,0,reject


In [60]:
# Durable evidence is stored outside .spur/graph, whose contents are rebuildable cache.
pg11_solver_dossier = json.loads((pg11_evidence_dir / 'solver-receipts.json').read_text())
assert pg11_solver_dossier['observations'] == pg11_catalog_observations
assert len(pg11_solver_dossier['runs']) == 7
assert sum(r['family_result']['status'] == 'sat' for r in pg11_solver_dossier['runs']) == 2
assert sum(r['family_result']['status'] == 'unsat' for r in pg11_solver_dossier['runs']) == 5
assert all(r['reloaded']['pinned'] and r['reloaded']['variable_count'] <= 64
           for r in pg11_solver_dossier['runs'])
assert duckdb.sql("SELECT count(*), sum(violations) FROM pg11_integrity").fetchone() == (12, 0)
assert duckdb.sql("SELECT count(*), count(DISTINCT site_id) FROM pg11_witnesses").fetchone() == (6, 6)
assert duckdb.sql("SELECT count(*) FROM pg11_witness_reach WHERE hops IS NULL OR hops > 5").fetchone()[0] == 0
pg11_cte_distance = duckdb.sql("""
 SELECT r.hops FROM pg11_reached_nodes r
 WHERE r.file_path='crates/spur-graph/src/git_walk.rs' AND r.qualified_name='parse_raw_diff'
""").fetchone()[0]
pg11_routing_distance = duckdb.sql("SELECT max(hops) FROM pg11_raw_diff_path").fetchone()[0]
assert pg11_cte_distance == pg11_routing_distance == 5
assert len(pg11_compiler_observations) == 24
assert all(p['runtime']['safe_controls_all_ok'] for p in pg11_mir_profiles)
for p in pg11_mir_profiles:
    assert hashlib.sha256((pg11_evidence_dir / p['mir_file']).read_bytes()).hexdigest() == p['mir_sha256']
assert json.loads((pg11_repo / '.spur/graph/CURRENT/manifest.json').read_text())['graph_content_hash'] == pg11_meta['graph_content_hash']

def pg11_write_json(name, value):
    (pg11_evidence_dir / name).write_text(json.dumps(value, indent=2) + '\n')

pg11_summary = dict(
 schema_version=1, graph_content_hash=pg11_meta['graph_content_hash'],
 graph_schema=pg11_meta['schema_version'],
 macro_evidence=pg11_meta['macro_evidence'],
 graph_files=duckdb.sql("SELECT count(*) FROM pg11_source_manifest").fetchone()[0],
 macro_invocation_rows=duckdb.sql("SELECT count(*) FROM pg11_macro_sites").fetchone()[0],
 macro_scope_rows=duckdb.sql("SELECT count(*) FROM pg11_macro_scopes").fetchone()[0],
 macro_bearing_files=duckdb.sql("SELECT count(DISTINCT file_path) FROM pg11_macro_sites").fetchone()[0],
 rust_source_checks=pg11_records("SELECT state, count(*) AS files FROM pg11_live_sources GROUP BY state"),
 sites_without_function_caller=duckdb.sql("SELECT count(*) FROM pg11_macro_sites WHERE caller_stable_symbol_id IS NULL").fetchone()[0],
 nonfunction_site_phases=pg11_records("""
  SELECT execution_phase, count(*) AS sites,
         count(*) FILTER (WHERE sink_family_candidate IS NOT NULL) AS sink_name_candidates
  FROM pg11_macro_sites WHERE caller_stable_symbol_id IS NULL
  GROUP BY execution_phase ORDER BY execution_phase
 """),
 root=pg11_records("SELECT * FROM pg11_root")[0],
 maximum_observed_hops=5,
 recorded_reachable_symbols=duckdb.sql("SELECT count(*) FROM pg11_reachable").fetchone()[0],
 spur_graph_reachable_symbols=duckdb.sql("SELECT count(*) FROM pg11_reached_nodes WHERE file_path LIKE 'crates/spur-graph/src/%'").fetchone()[0],
 reached_spur_graph_macro_sites=duckdb.sql("SELECT count(*) FROM pg11_reached_macro_sites WHERE file_path LIKE 'crates/spur-graph/src/%'").fetchone()[0],
 reached_spur_graph_sink_name_candidates=duckdb.sql("SELECT count(*) FROM pg11_reached_sink_candidates").fetchone()[0],
 recovered_original_witnesses=6,
 unresolved_method_label_records=duckdb.sql("SELECT count(*) FROM pg11_method_frontier").fetchone()[0],
 distinct_method_frontier_callers=duckdb.sql("SELECT count(DISTINCT caller_id) FROM pg11_method_frontier").fetchone()[0],
 raw_diff_cte_distance=pg11_cte_distance,
 raw_diff_routing_distance=pg11_routing_distance,
 compiler=dict(version=pg11_compiler_version, profiles=4, observed_function_profiles=24,
               good_input_controls=24, fixture_library_macro_sites=3,
               fixture_unwrap_label_records=pg11_fixture_unwrap_records,
               scope='isolated six-case fixture; no whole-workspace MIR or CGA analysis'),
 caveats=[
  'Candidate names and recorded graph paths do not prove macro identity, cfg activation, or panic execution.',
  'Method label edge records are not call-site counts or type-checked targets.',
  'The five-hop budget does not establish whole-program reachability or absence of other panics.',
  'SQL audits cover declared finite identity/reference/span properties, not the full Rust artifact validator.',
  'Compiler observations use human-readable MIR of the pinned fixture/compiler, not a general MIR parser.',
  'Solver receipts validate supplied finite facts; they do not inspect Rust or prove panic freedom.'])
pg11_write_json('experiment-summary.json', pg11_summary)
pg11_write_json('workspace-manifest.json', pg11_meta)
pg11_write_json('workspace-witnesses.json', dict(
 source_checks=pg11_witness_source_checks,
 reachability=pg11_records("SELECT * FROM pg11_witness_reach ORDER BY site_id")))
pg11_write_json('method-frontier.json', pg11_records(
 "SELECT * FROM pg11_method_frontier ORDER BY hops, file_path, caller_name, target_label"))
pg11_write_json('context-counts.json', pg11_records(
 "SELECT * FROM pg11_context_counts ORDER BY scope, evidence_class, test_context"))
pg11_write_json('integrity-controls.json', dict(
 audits=pg11_records("SELECT * FROM pg11_integrity ORDER BY check_name"),
 negative_controls=pg11_records("SELECT * FROM pg11_control_matrix ORDER BY control")))
pg11_write_json('routing-path.json', pg11_records("""
 SELECT p.seq, p.node, p.hops, n.stable_symbol_id, n.qualified_name, n.file_path
 FROM pg11_raw_diff_path p JOIN pg11_nodes n ON n.node_id=p.node ORDER BY p.seq
"""))
pg11_write_json('fixture-graph.json', dict(
 manifest=pg11_fixture_manifest,
 library_macro_sites=pg11_records("""
  SELECT n.entity_name AS case_name, m.*
  FROM pg11_fixture_sites m JOIN pg11_fixture_nodes n
   ON n.stable_symbol_id=m.caller_stable_symbol_id AND n.file_path=m.file_path
  WHERE m.file_path='src/lib.rs' ORDER BY n.entity_name
 """),
 method_label_records=pg11_fixture_unwrap_records))

# Preserve the exact audited source corpus compactly, without copying source bodies.
compact = dict(
 columns=['file_path', 'indexed_blob_oid', 'byte_length', 'live_bytes_matched'],
 rows=[[path, expected, size, state == 'match']
       for path, expected, actual, size, state in pg11_live_sources],
 audit='SHA1 of Git blob header plus current bytes; full byte and line checks for the six witnesses are separate.')
(pg11_evidence_dir / 'rust-source-bindings.json').write_text(json.dumps(compact, separators=(',', ':')) + '\n')
pg11_inputs = []
for name in ('manifest.json', 'macro_invocations.parquet', 'macro_scopes.parquet',
             'nodes.parquet', 'edges.parquet', 'edges_unresolved.parquet',
             'file_manifests.parquet', 'files.parquet'):
    path = pg11_artifact / name
    pg11_inputs.append(dict(file=name, bytes=path.stat().st_size,
                           sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
pg11_write_json('workspace-inputs.json', dict(
 graph_content_hash=pg11_meta['graph_content_hash'], artifact=str(pg11_artifact), files=pg11_inputs,
 checksum_note='SHA256 records input bytes; macro shared BLAKE3 validation is not reimplemented by these SQL audits.'))

pg11_export_inventory = [dict(file=p.name, bytes=p.stat().st_size,
                             sha256=hashlib.sha256(p.read_bytes()).hexdigest())
                        for p in sorted(pg11_evidence_dir.iterdir())
                        if p.is_file() and p.name != 'evidence-index.json']
pg11_write_json('evidence-index.json', dict(
 graph_content_hash=pg11_meta['graph_content_hash'], files=pg11_export_inventory))
print(json.dumps(dict(
 workspace=pg11_summary,
 exported_files=len(pg11_export_inventory) + 1,
 evidence_directory=str(pg11_evidence_dir),
 evidence_bytes_excluding_index=sum(row['bytes'] for row in pg11_export_inventory)
), indent=2))

{
  "workspace": {
    "schema_version": 1,
    "graph_content_hash": "64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4",
    "graph_schema": "spur-graph-schema-v11",
    "macro_evidence": {
      "complete": true,
      "schema_version": "spur-macro-invocation-v1",
      "extractor_version": "2026-10-31-physical-rust-macro-evidence-v1",
      "source_scope": "physical_rust_files",
      "checksum_blake3": "a940b331261b2bf60d2b4d87bf33205b1fd98027fafe7c79eee4e8c2a4b19e9c"
    },
    "graph_files": 3579,
    "macro_invocation_rows": 46114,
    "macro_scope_rows": 17551,
    "macro_bearing_files": 1265,
    "rust_source_checks": [
      {
        "state": "match",
        "files": 1510
      }
    ],
    "sites_without_function_caller": 209,
    "nonfunction_site_phases": [
      {
        "execution_phase": "const_context_syntax",
        "sites": 123,
        "sink_name_candidates": 1
      },
      {
        "execution_phase": "item_syntax",
        "sites": 86,
      

### Findings and the next effective gap to close

The new graph closes the measured **6/6 direct-syntax witness gap**. The workspace has **46,114 invocation rows / 17,551 scopes**, and all **1,510 physical Rust source files** match their recorded blob IDs. All **12 SQL identity, source, ownership and span audits** report zero violations. The **209 sites without a function caller** retain const/item scope: 123 const-context sites and 86 item sites.

From `build_with_temporal_jobs_override`, the historical five-hop budget now records **263 reachable symbols**, including 236 in `spur-graph/src`. Those graph-crate callers contain **270 macro sites** and the six original sink-name candidates. All six still have zero matching ordinary-call labels: the improvement comes from the dedicated macro evidence. The recursive CTE and duckrouting agree that `parse_raw_diff` is five recorded calls away. Its path edges are marked `syntax_exact`; this is graph evidence, not compiler-checked path feasibility. Older 270-symbol results used a different snapshot and should not be used as a recall denominator.

The bounded slice also contains **seven unresolved method-label records in seven callers**: six `expect` and one `unwrap`. These records can collapse multiple invocations and do not establish receiver types. The six macro candidates likewise have unresolved macro identity and unspecified test context; five are debug-assert name candidates. **Unspecified does not mean production.**

The controlled fixture adds concrete evidence for the next layer. Its six deliberately selected library cases have three macro sites and one unwrap label. With rustc **1.94.1**, four independent debug/overflow configurations produce **24 function/configuration observations**. All **24 good-input controls** complete. On the selected failing inputs:
- Debug assertions follow the debug-assertions switch; arithmetic overflow follows the overflow-checks switch.
- Slice indexing has a compiler bounds `Assert` despite having no macro site.
- Checked addition has a compiler overflow `Assert` despite having no macro site.
- `Option::unwrap` is a typed MIR call and panics for `None`, but its dependency body is outside this emitted library MIR.
- Ordinary assertions and the explicit panic remain visible in all four configurations.

This is a controlled fixture result, **not a workspace panic count or recall estimate**. A flat search for panic calls in local MIR also misses terminators and dependency bodies.

| Remaining gap | Measured reason to prioritize it | Effective next experiment / acceptance |
|---|---|---|
| Method call-site multiplicity and callee types | Seven unresolved `expect`/`unwrap` label records | Preserve each invocation's byte span and receiver/type evidence; distinguish standard Option/Result calls from similarly named methods and retain unknown targets. |
| Build configuration and macro expansion | Five unresolved debug-name candidates; fixture behavior changes with flags | Bind evidence to toolchain, target, features/cfg and compiler flags; retain invocation-to-expansion/source mappings. Repeat the four-configuration controls. |
| Compiler-inserted checks and dependency calls | Fixture bounds/overflow checks have no macro rows; unwrap body is outside local MIR | Add build-specific MIR Assert/callee evidence alongside syntax evidence. Hold out indexing, overflow, cfg and cross-crate cases before applying it to a real build slice. |
| Value-flow feasibility | The same emitted checks coexist with 24 successful good-input cases | Model source-bound guard/input invariants on selected paths. Record assumptions and counterexamples; candidate absence or a bounded graph path cannot establish panic freedom. |

The metadata controls accept the observed six-site set and reject omission, duplication, stale source, mixed snapshot, and assignment to a different existing function. Catalog SOLVE verifies the same finite obligations in two fixed projections below its 64-variable limit: source/snapshot identity (**52 variables**, `sol_d0ea89640dda4390`) and caller/scope association (**50**, `sol_cdc60250074d4159`). Five corrupted controls return UNSAT. These checks validate supplied metadata, not Rust execution.

**Re-run:** after the managed DuckDB setup, execute this rebuilt-workspace section in order. It is pinned to graph `64c5b7a1…`; archived cells above retain removed cache paths. The compiler fixture is recreated outside the workspace, uses the local cargo wrapper offline, and leaves the main graph unchanged. The evidence ledger contains recorded MCP solver receipts; running this notebook does not rerun Z3.

[Durable evidence index](docs/research/artifacts/2026-10-31-panic-graph-64c5b7a163b7/evidence-index.json) · [Summary](docs/research/artifacts/2026-10-31-panic-graph-64c5b7a163b7/experiment-summary.json) · [Compiler observations](docs/research/artifacts/2026-10-31-panic-graph-64c5b7a163b7/compiler-observations.json) · [Exact solver requests/receipts](docs/research/artifacts/2026-10-31-panic-graph-64c5b7a163b7/solver-receipts.json)

**2026-11-01 update:** P1/P2/P3 are implemented and accepted. The final notebook section records five real pinned CGA builds, 45 executable controls and 26 corruption/substitution rejections, with a standalone durable replay. The source-bound workspace frontier now distinguishes three exact callers, two closure sites and two macro-token cases. Whole-workspace compiler panic analysis remains unrun. [Completed pilot and evidence](docs/research/artifacts/2026-11-01-panic-evidence-pilot/README.md).

### @spec Panic evidence pilot: source sites and compiler configurations

**Status:** implementation authorized by the user's “approved for next step” following the findings above. Design task `bd-3je`, source epic `bd-1hg`. This is an experimental sidecar API in `spur-graph`; the published v11 graph remains the notebook baseline.

**Problem and acceptance:** seven stored unresolved method labels cannot establish how many invocations occurred or whether their callees are standard Option/Result methods. The four existing compiler configurations show checks absent from direct syntax. Close these experimentally with lossless direct method sites and actual compiler instance/Assert evidence, each pinned to exact source and a declared build. Preserve unknowns and candidate-path semantics.

**Architecture:**
1. A new public `panic_evidence` module collects direct physical Rust method-call syntax from exact bytes checked against an existing `GraphIndexArtifact` source manifest. It walks the Rust AST, independently of deduplicated Calls edges; it never rewrites graph edges, artifacts, CURRENT, or schema v11.
2. A separate CGA adapter accepts the pinned export format plus an explicit capture envelope. It preserves raw compiler instance identities, all edge kinds, raw locations, and unresolved associations. It uses a build fingerprint and export checksum before associating compiler locations with syntax sites.
3. A small example command and held-out fixture exercise capture/import through the local cargo wrapper. Root runs the acceptance cells through Notebook MCP, compares the existing syntax baseline, and records durable commands, hashes, receipts and limitations.

**Source contract:** `PanicEvidenceSource { file_path, bytes }`; `collect_method_call_sites(graph, sources) -> Result<MethodCallSiteEvidenceSet>`. The set contains schema ID, graph hash, exact supplied-file bindings and deterministically ordered rows. Coverage means **supplied physical Rust files**, never the whole workspace or compiler reachability. Each row retains a domain-separated stable site ID, source Git blob OID, whole-call half-open byte range, method and receiver byte ranges/text, one-based inclusive line range, and lexical phase. Only a matching actual graph function/method in that file can be a caller; nested closure, const/item, malformed or ambiguous ownership remains explicitly qualified/unknown. No synthetic function IDs. Keep every repeated, chained, generic and qualified direct method invocation. Macro token trees, strings/comments and Rust notebook cells are outside this direct-syntax scope. No type certainty is derived from receiver text or names.

**Compiler/capture contract:** pin CGA revision `2fc63108f0b674fe7a178191ffa15647a18879f1` and its `nightly-2026-10-18` toolchain for this pilot. The envelope records schema, exact source/manifest/Cargo.lock/config bindings, graph hash, rustc verbose identity, host and target, exporter revision/binary digest, package/target selection, feature/default-feature choices, cfg and compiler flags, Cargo arguments, relevant allowlisted environment, capture command and raw-export SHA256. Canonicalization sorts set-like keys/features/cfg and retains order-sensitive arguments/flags. All fields contributing to semantics contribute to the build ID. Missing/inconsistent identities and stale inputs are rejected. This is a bounded fixture capture contract, not certification of arbitrary ambient Cargo environments.

CGA JSON nodes and edge references are validated before use. Preserve `Called`, `VTableCalled`, `FnPtrCast`, `BuiltinAssert` and `AsyncPanic` separately; vtable/pointer evidence is possible reachability. Preserve compiler canonical instance names and their generic arguments; CGA does not export a separate receiver-type field. `BuiltinAssert` has no reason field here, so do not label its reason from a guess. Keep external/unknown locations without manufacturing local source or callers. Preserve raw source coordinates and a documented, tested conversion to UTF-8 byte ranges; the Rust RangeInclusive serializer alone does not establish whether the compiler endpoint is inclusive.

**Association contract:** source/config identity must match the expected snapshot/build before association. Match on the bound path and compiler byte location, with explicit exact/unique-containing/ambiguous/unmapped outcomes; never associate by a bare method name. Multiple instantiated callees or uses at a source site remain separate compiler observations. Compiler absence is not a safe-method verdict. CGA currently omits macro expansion maps; retain that capability as unavailable instead of inventing an expansion mapping. A source method named unwrap/expect stays an unknown candidate until a real compiler observation supplies its canonical callee. Caller metadata is validated against ownership, not just any existing symbol.

**Held-out acceptance corpus:** repeated unwrap/expect calls in one function, chained and generic methods, a safe custom method with the same name, Option and Result calls, an out-of-line local dependency, cfg/feature-gated code, macro-generated code, UTF-8 source before a site, slice indexing and checked addition. Repeat independent debug-assertions/overflow-checks configurations at opt-level 0 with panic=unwind, plus explicit cfg-feature contrasts. Record optimizer effects; emitted observations are not dynamic execution proofs. Good/bad runtime controls run in child processes using the same declared flags where supported.

**Negative controls:** duplicate/omitted source site, stale source OID, mixed graph hash, wrong existing caller, altered target/toolchain/feature/flags, changed raw export, dangling compiler IDs, ambiguous source locations and unknown exporter revision. Positive and corrupted identity/reference/cardinality facts use catalog SOLVE PRE/POST in fixed finite projections; solver verdicts concern supplied facts, not Rust execution.

**Delivery:** three serial, reviewed SPUR Codex tasks (gpt-6.1-sol/xhigh): P1 syntax collector and validation; P2 build envelope and pinned CGA adapter; P3 executable fixture/example and held-out acceptance. Each task records catalog navigation, persisted PRE/POST receipts, a real assertion RED, GREEN logs, authored test/fix commits and unified diff. Root owns the notebook single-writer acceptance and final integration. Never bare cargo: `SPUR_REMOTE=0 scripts/spur-cargo`. No whole-workspace CGA build, changed workspace toolchain, default graph-schema bump, or whole-program panic-freedom claim.

**Primary references:** [CGA](https://github.com/OpenDevicePartnership/cga/tree/2fc63108f0b674fe7a178191ffa15647a18879f1), [exported data](https://github.com/OpenDevicePartnership/cga/blob/2fc63108f0b674fe7a178191ffa15647a18879f1/cga_data/src/callgraph.rs), [location assembly](https://github.com/OpenDevicePartnership/cga/blob/2fc63108f0b674fe7a178191ffa15647a18879f1/cga_analysis/src/callgraph/data.rs).


### Pilot compiler backend: capability check

The three-task sidecar plan is running as `770b6376-2b19-46bf-9fba-4004ed3e76b3` (P1 `bd-3kv`, P2 `bd-147`, P3 `bd-1l9`). This cell checks only whether the real compiler exporter runs on an isolated four-function source; it is separate from P3's held-out acceptance.

CGA is pinned to `2fc63108f0b674fe7a178191ffa15647a18879f1`, with rustc `1.101.0-nightly` (`8006091eb71ae7d9e9bccb66c6122c9d68af4466`, nightly-2026-10-18). Its unmodified macOS build failed at linking (exit 101). The temporary checkout has a **build-only rpath delimiter patch**, from `-Wl,-rpath=…` to `-Wl,-rpath,…`, SHA256 `72f3d4cd64439d988deba0ee1e0f449df8290e6f1d1e0ae83ba82d97b1592aca`. The corrected build passed. Exporter logic is unchanged, but this is a patched producer; capture identity must retain the patch and binary digests.

The upstream launcher overwrites `RUSTFLAGS`; the probe therefore declares debug/overflow/panic/optimization settings in named Cargo profiles and saves verbose effective compiler commands. Compiler instance names are typed observations. No separate receiver type, assertion reason, or expansion mapping is supplied by this exporter. The compiler upgrade is specific to this pilot; earlier stable-rustc observations remain pinned to their own toolchain.

In [61]:
import os, json, hashlib, subprocess, time
from pathlib import Path
from collections import Counter

pilot_probe = Path("/private/tmp/spur-cga-pilot-probe-20261031")
pilot_probe.mkdir(exist_ok=True)
(pilot_probe / "src").mkdir(exist_ok=True)
pilot_sources = {
    "Cargo.toml": '[package]\nname = "spur-panic-cga-probe"\nversion = "0.0.0"\nedition = "2021"\n\n[workspace]\n\n[profile.d0_o0]\ninherits = "dev"\nopt-level = 0\ndebug-assertions = false\noverflow-checks = false\npanic = "unwind"\n\n[profile.d1_o1]\ninherits = "dev"\nopt-level = 0\ndebug-assertions = true\noverflow-checks = true\npanic = "unwind"\n',
    "src/lib.rs": '#![no_std]\n\n#[unsafe(no_mangle)]\npub fn option_unwrap(value: Option<u8>) -> u8 { let _ = "雪🙂"; value.unwrap() }\n\n#[unsafe(no_mangle)]\npub fn slice_index(values: &[u8], index: usize) -> u8 { values[index] }\n\n#[unsafe(no_mangle)]\npub fn add(value: u8) -> u8 { value + 1 }\n\n#[unsafe(no_mangle)]\npub fn debug_assertion(value: u8) { debug_assert!(value > 0); }\n',
    "rust-toolchain.toml": '[toolchain]\nchannel = "nightly-2026-10-18"\nprofile = "minimal"\n',
}
for rel, content in pilot_sources.items():
    path = pilot_probe / rel
    if path.exists() and path.read_text() != content:
        raise RuntimeError("Probe source differs: " + rel)
    path.write_text(content)
pilot_producer = json.loads(Path("/private/tmp/spur-cga-pilot-producer.json").read_text())
for meta in pilot_producer["binaries"].values():
    assert hashlib.sha256(Path(meta["path"]).read_bytes()).hexdigest() == meta["sha256"]
pilot_env = {k: v for k, v in os.environ.items() if k in {"HOME", "PATH", "TMPDIR", "RUSTUP_HOME", "CARGO_HOME", "LANG", "LC_CTYPE", "SSL_CERT_FILE", "SSL_CERT_DIR"}}
pilot_env.update({
    "PATH": str(Path(pilot_producer["binaries"]["cargo-cga"]["path"]).parent) + os.pathsep + pilot_env["PATH"],
    "SPUR_REMOTE": "0", "SPUR_SCCACHE_S3": "0", "RUSTC_WRAPPER": "",
    "RUSTUP_TOOLCHAIN": "nightly-2026-10-18", "CARGO_INCREMENTAL": "0",
})
pilot_current = Path("/Users/kevintruong/project/SpurCode/.spur/graph/CURRENT")
pilot_graph_before = {"path": str(pilot_current.resolve()), "manifest_sha256": hashlib.sha256((pilot_current / "manifest.json").read_bytes()).hexdigest()}
pilot_cmd = [
    "/Users/kevintruong/project/SpurCode/scripts/spur-cargo", "--workdir", str(pilot_probe),
    "cga", "extract", "-T", "json", "-o", str(pilot_probe / "d1_o1.callgraph.json"),
    "--build-std", "core", "--target", "aarch64-apple-darwin",
    "--profile", "d1_o1", "--target-dir", str(pilot_probe / "target" / "d1_o1"), "--verbose",
]
pilot_started = time.monotonic()
with (pilot_probe / "capture-d1_o1.log").open("wb") as log:
    pilot_proc = subprocess.run(pilot_cmd, env=pilot_env, stdout=log, stderr=subprocess.STDOUT, timeout=600)
pilot_elapsed = time.monotonic() - pilot_started
pilot_log = (pilot_probe / "capture-d1_o1.log").read_bytes()
pilot_record = {
    "kind": "capability_probe", "profile": "d1_o1", "command": pilot_cmd,
    "exit_code": pilot_proc.returncode, "elapsed_seconds": pilot_elapsed,
    "producer": pilot_producer, "log_sha256": hashlib.sha256(pilot_log).hexdigest(),
    "sources": {rel: {"sha256": hashlib.sha256(content.encode()).hexdigest(), "git_blob_oid": hashlib.sha1(b"blob " + str(len(content.encode())).encode() + bytes([0]) + content.encode()).hexdigest()} for rel, content in pilot_sources.items()},
    "graph_before": pilot_graph_before,
}
pilot_record["graph_after"] = {"path": str(pilot_current.resolve()), "manifest_sha256": hashlib.sha256((pilot_current / "manifest.json").read_bytes()).hexdigest()}
assert pilot_record["graph_before"] == pilot_record["graph_after"], "Published graph changed during probe"
if pilot_proc.returncode:
    (pilot_probe / "capability-failed.json").write_text(json.dumps(pilot_record, indent=2))
    print(json.dumps(pilot_record, indent=2))
    print(pilot_log.decode(errors="replace")[-6000:])
    raise RuntimeError("CGA capability probe failed; preserved command and log")
pilot_raw = (pilot_probe / "d1_o1.callgraph.json").read_bytes()
pilot_cg = json.loads(pilot_raw)
pilot_record.update({
    "raw_sha256": hashlib.sha256(pilot_raw).hexdigest(), "raw_bytes": len(pilot_raw),
    "node_count": len(pilot_cg["nodes"]), "root_count": len(pilot_cg["roots"]),
    "edge_count": sum(len(v) for v in pilot_cg["edges"].values()),
})
pilot_local = []
for node_id, node in enumerate(pilot_cg["nodes"]):
    if "spur_panic_cga_probe" in node["name"]:
        for edge in pilot_cg["edges"].get(str(node_id), []):
            kind = edge["kind"]
            target = next(iter(kind.values())) if isinstance(kind, dict) else None
            pilot_local.append({"caller": node["name"], "kind": next(iter(kind)) if isinstance(kind, dict) else kind, "target": pilot_cg["nodes"][target]["name"] if target is not None else None, "location": edge["location"]})
pilot_record["local_observations"] = pilot_local
(pilot_probe / "capability.json").write_text(json.dumps(pilot_record, ensure_ascii=False, indent=2))
print(json.dumps(pilot_record, ensure_ascii=False, indent=2))


{
  "kind": "capability_probe",
  "profile": "d1_o1",
  "command": [
    "/Users/kevintruong/project/SpurCode/scripts/spur-cargo",
    "--workdir",
    "/private/tmp/spur-cga-pilot-probe-20261031",
    "cga",
    "extract",
    "-T",
    "json",
    "-o",
    "/private/tmp/spur-cga-pilot-probe-20261031/d1_o1.callgraph.json",
    "--build-std",
    "core",
    "--target",
    "aarch64-apple-darwin",
    "--profile",
    "d1_o1",
    "--target-dir",
    "/private/tmp/spur-cga-pilot-probe-20261031/target/d1_o1",
    "--verbose"
  ],
  "exit_code": 0,
  "elapsed_seconds": 70.0224073749996,
  "producer": {
    "upstream_revision": "2fc63108f0b674fe7a178191ffa15647a18879f1",
    "patch_kind": "build-only linker rpath argument delimiter; exporter logic unchanged",
    "patch_path": "/private/tmp/spur-cga-pilot-macos-rpath.patch",
    "patch_sha256": "72f3d4cd64439d988deba0ee1e0f449df8290e6f1d1e0ae83ba82d97b1592aca",
    "build_rs_original_sha256": "420f931f4114d471c2ac7579d3ed05a1adc90619f99f

In [62]:
# Compare real emitted compiler evidence; this remains a capability probe.
import shutil
pilot_low_cmd = [part.replace("d1_o1", "d0_o0") for part in pilot_cmd]
pilot_low_started = time.monotonic()
with (pilot_probe / "capture-d0_o0.log").open("wb") as log:
    pilot_low_proc = subprocess.run(pilot_low_cmd, env=pilot_env, stdout=log, stderr=subprocess.STDOUT, timeout=600)
pilot_low_elapsed = time.monotonic() - pilot_low_started
if pilot_low_proc.returncode:
    print((pilot_probe / "capture-d0_o0.log").read_text(errors="replace")[-6000:])
    raise RuntimeError("Low-check compiler probe failed")
pilot_low_raw = (pilot_probe / "d0_o0.callgraph.json").read_bytes()
pilot_low_cg = json.loads(pilot_low_raw)

def pilot_local_edges(cg, function):
    return [edge for node_id, node in enumerate(cg["nodes"])
            if node["name"] == "spur_panic_cga_probe::" + function
            for edge in cg["edges"].get(str(node_id), [])]

def pilot_called_targets(cg, function):
    return [cg["nodes"][edge["kind"]["Called"]]["name"]
            for edge in pilot_local_edges(cg, function)
            if isinstance(edge["kind"], dict) and "Called" in edge["kind"]]

pilot_matrix = []
for label, cg in [("d0_o0", pilot_low_cg), ("d1_o1", pilot_cg)]:
    entry = {
        "profile": label,
        "roots": [cg["nodes"][n]["name"] for n in cg["roots"]],
        "index_builtin_asserts": sum(e["kind"] == "BuiltinAssert" for e in pilot_local_edges(cg, "slice_index")),
        "addition_builtin_asserts": sum(e["kind"] == "BuiltinAssert" for e in pilot_local_edges(cg, "add")),
        "debug_called_targets": pilot_called_targets(cg, "debug_assertion"),
        "unwrap_called_targets": pilot_called_targets(cg, "option_unwrap"),
    }
    assert entry["index_builtin_asserts"] == 1
    assert entry["addition_builtin_asserts"] == int(label == "d1_o1")
    assert ("core::panicking::panic" in entry["debug_called_targets"]) == (label == "d1_o1")
    assert entry["unwrap_called_targets"] == ["core::option::Option::<u8>::unwrap"]
    pilot_matrix.append(entry)
pilot_unwrap_edge = next(e for e in pilot_local_edges(pilot_cg, "option_unwrap")
                        if isinstance(e["kind"], dict) and "Called" in e["kind"])
pilot_span = pilot_unwrap_edge["location"]["span"]
pilot_source = pilot_sources["src/lib.rs"]
pilot_lines = pilot_source.splitlines(keepends=True)
def pilot_byte_offset(position):
    line = position["line"] - 1
    column = position["column"] - 1
    return sum(len(text.encode()) for text in pilot_lines[:line]) + len(pilot_lines[line][:column].encode())
pilot_range = [pilot_byte_offset(pilot_span["start"]), pilot_byte_offset(pilot_span["end"])]
pilot_slice = pilot_source.encode()[pilot_range[0]:pilot_range[1]].decode()
assert pilot_slice == "value.unwrap()"
assert all((pilot_probe / rel).read_text() == content for rel, content in pilot_sources.items())
pilot_coordinates = {
    "raw_span": pilot_span, "byte_range": pilot_range, "slice": pilot_slice,
    "column_unit": "one-based Unicode characters", "end": "exclusive compiler span.hi",
    "utf8_prefix": "雪🙂",
    "byte_column_minus_character_column": len(pilot_lines[3][:pilot_span["start"]["column"] - 1].encode()) - (pilot_span["start"]["column"] - 1),
}
assert pilot_coordinates["byte_column_minus_character_column"] == 5
pilot_effective_commands = {
    label: [line for line in (pilot_probe / ("capture-" + label + ".log")).read_text().splitlines()
            if "--crate-name spur_panic_cga_probe" in line]
    for label in ["d0_o0", "d1_o1"]
}
assert all(len(lines) == 1 for lines in pilot_effective_commands.values())
pilot_current_after = {"path": str(pilot_current.resolve()), "manifest_sha256": hashlib.sha256((pilot_current / "manifest.json").read_bytes()).hexdigest()}
assert pilot_current_after == pilot_graph_before
pilot_capability = {
    "scope": "two-profile, four-root capability probe; P1/P2/P3 implementation acceptance pending",
    "matrix": pilot_matrix, "coordinates": pilot_coordinates,
    "effective_compiler_commands": pilot_effective_commands,
    "commands": {"d0_o0": pilot_low_cmd, "d1_o1": pilot_cmd},
    "elapsed_seconds": {"d0_o0": pilot_low_elapsed, "d1_o1": pilot_elapsed},
    "producer": pilot_producer, "sources": pilot_record["sources"],
    "graph_before": pilot_graph_before, "graph_after": pilot_current_after,
    "limitations": [
        "BuiltinAssert reasons are absent in the upstream export.",
        "Debug assertion location points to the core macro definition, without expansion mapping.",
        "Compiler edge presence does not prove input-specific execution or panic freedom.",
        "The two-profile capability probe is separate from held-out implementation acceptance."
    ],
}
pilot_artifacts = Path("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-10-31-panic-evidence-pilot-setup")
pilot_artifacts.mkdir(parents=True, exist_ok=True)
for label in ["d0_o0", "d1_o1"]:
    for filename in [label + ".callgraph.json", "capture-" + label + ".log"]:
        shutil.copyfile(pilot_probe / filename, pilot_artifacts / filename)
for path in [Path("/private/tmp/spur-cga-pilot-macos-rpath.patch"),
             Path("/private/tmp/spur-cga-pilot-build.log"),
             Path("/private/tmp/spur-cga-pilot-build-patched.log")]:
    shutil.copyfile(path, pilot_artifacts / path.name)
(pilot_artifacts / "producer.json").write_text(json.dumps(pilot_producer, indent=2))
(pilot_artifacts / "probe-sources.json").write_text(json.dumps(pilot_sources, ensure_ascii=False, indent=2))
(pilot_artifacts / "capability.json").write_text(json.dumps(pilot_capability, ensure_ascii=False, indent=2))
pilot_index = {
    "scope": pilot_capability["scope"],
    "files": {path.name: {"bytes": path.stat().st_size, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
              for path in sorted(pilot_artifacts.iterdir()) if path.is_file() and path.name != "evidence-index.json"}
}
(pilot_artifacts / "evidence-index.json").write_text(json.dumps(pilot_index, indent=2))
print(json.dumps({"matrix": pilot_matrix, "coordinates": pilot_coordinates, "elapsed_seconds": pilot_capability["elapsed_seconds"], "artifact_dir": str(pilot_artifacts), "graph_unchanged": True}, ensure_ascii=False, indent=2))


{
  "matrix": [
    {
      "profile": "d0_o0",
      "roots": [
        "spur_panic_cga_probe::debug_assertion",
        "spur_panic_cga_probe::slice_index",
        "spur_panic_cga_probe::option_unwrap",
        "spur_panic_cga_probe::add"
      ],
      "index_builtin_asserts": 1,
      "addition_builtin_asserts": 0,
      "debug_called_targets": [],
      "unwrap_called_targets": [
        "core::option::Option::<u8>::unwrap"
      ]
    },
    {
      "profile": "d1_o1",
      "roots": [
        "spur_panic_cga_probe::debug_assertion",
        "spur_panic_cga_probe::slice_index",
        "spur_panic_cga_probe::add",
        "spur_panic_cga_probe::option_unwrap"
      ],
      "index_builtin_asserts": 1,
      "addition_builtin_asserts": 1,
      "debug_called_targets": [
        "core::panicking::panic"
      ],
      "unwrap_called_targets": [
        "core::option::Option::<u8>::unwrap"
      ]
    }
  ],
  "coordinates": {
    "raw_span": {
      "start": {
        "line": 4,
 

In [63]:
# Archive raw diagnostics losslessly without committing whitespace-only log lines.
# The decoded SHA256 remains the original command-log digest.
pilot_archives = {}
for pilot_log_path in sorted(pilot_artifacts.glob("*.log")):
    pilot_log_bytes = pilot_log_path.read_bytes()
    pilot_envelope = {
        "encoding": "utf8",
        "original_name": pilot_log_path.name,
        "original_byte_length": len(pilot_log_bytes),
        "original_sha256": hashlib.sha256(pilot_log_bytes).hexdigest(),
        "text": pilot_log_bytes.decode("utf-8"),
    }
    pilot_archive_path = pilot_log_path.with_suffix(".log.json")
    pilot_archive_path.write_text(json.dumps(pilot_envelope, ensure_ascii=False, indent=2))
    pilot_reloaded = json.loads(pilot_archive_path.read_text())
    assert pilot_reloaded["text"].encode("utf-8") == pilot_log_bytes
    pilot_archives[pilot_log_path.name] = {
        "archive": pilot_archive_path.name,
        "decoded_sha256": pilot_envelope["original_sha256"],
    }
    pilot_log_path.unlink()  # Only this experiment's copied log; raw /private/tmp log is retained.
for pilot_archive_path in sorted(pilot_artifacts.glob("*.log.json")):
    pilot_reloaded = json.loads(pilot_archive_path.read_text())
    assert hashlib.sha256(pilot_reloaded["text"].encode()).hexdigest() == pilot_reloaded["original_sha256"]
    pilot_archives[pilot_reloaded["original_name"]] = {
        "archive": pilot_archive_path.name,
        "decoded_sha256": pilot_reloaded["original_sha256"],
    }
pilot_index = {
    "scope": pilot_capability["scope"],
    "raw_log_archives": pilot_archives,
    "files": {path.name: {"bytes": path.stat().st_size, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
              for path in sorted(pilot_artifacts.iterdir()) if path.is_file() and path.name != "evidence-index.json"}
}
(pilot_artifacts / "evidence-index.json").write_text(json.dumps(pilot_index, indent=2))
print(json.dumps({"payloads": len(pilot_index["files"]), "lossless_log_archives": pilot_archives}, indent=2))


{
  "payloads": 10,
  "lossless_log_archives": {
    "capture-d0_o0.log": {
      "archive": "capture-d0_o0.log.json",
      "decoded_sha256": "54be1316697d2ef4e1ec9db54abfddd1f01506d318ba1b93f93085f2fd90c7f8"
    },
    "capture-d1_o1.log": {
      "archive": "capture-d1_o1.log.json",
      "decoded_sha256": "54073e0ca172be8427587503cbd49d3530eb690d67c6d093f5a1d0b747cc65f1"
    },
    "spur-cga-pilot-build-patched.log": {
      "archive": "spur-cga-pilot-build-patched.log.json",
      "decoded_sha256": "15eaa0381f71facc76e1edd078bab672688ed5fb4a04e984f47dfea589774cd7"
    },
    "spur-cga-pilot-build.log": {
      "archive": "spur-cga-pilot-build.log.json",
      "decoded_sha256": "0066fb0b2527169baa7431de0428f39b086f6f0a28a20c4bd4f532e5f9633bdf"
    }
  }
}


### Compiler backend capability result

The real, patched pinned CGA exporter ran successfully through Notebook MCP in **70.02s** with checks enabled and **44.09s** with both debug/overflow checks disabled. Each export retained the four declared roots.

| Observation in the isolated source | d0_o0 | d1_o1 |
|---|---:|---:|
| Slice-index `BuiltinAssert` | 1 | 1 |
| Addition `BuiltinAssert` | 0 | 1 |
| Debug-assert call to `core::panicking::panic` | 0 | 1 |
| Typed call to `core::option::Option::<u8>::unwrap` | 1 | 1 |

The exported unwrap span `(line 4, columns 63..77)` converts to **bytes 100..114**, exactly `value.unwrap()`. The preceding `雪🙂` creates a five-byte difference between character and byte columns, so this checks a real UTF-8 case and the exclusive endpoint. The verbose compiler invocations are retained with the raw exports and named profile manifests.

The debug assertion's emitted location is the **core macro definition**, which demonstrates the missing expansion-to-invocation mapping in this exporter. The typed unwrap callee is compiler evidence; the source spelling alone remains unknown. `BuiltinAssert` has no exported reason field. These emitted graph observations do not establish input-specific execution or panic freedom.

The published v11 graph pointer and manifest checksum matched before and after both runs. The two-profile capability check establishes a usable backend for P2/P3. This initial capability check preceded P1/P2/P3. **The completed pilot is recorded in the final notebook section**: five real configurations, held-out custom/dependency/cfg/macro controls, and a checksummed archive replay. [Accepted pilot evidence](docs/research/artifacts/2026-11-01-panic-evidence-pilot/README.md).

[Capability evidence](docs/research/artifacts/2026-10-31-panic-evidence-pilot-setup/capability.json) · [Raw-export and log index](docs/research/artifacts/2026-10-31-panic-evidence-pilot-setup/evidence-index.json) · [Producer/patch identity](docs/research/artifacts/2026-10-31-panic-evidence-pilot-setup/producer.json)

## Build-bound panic evidence pilot: invocation coverage

The original graph frontier contains **seven unresolved method-label records**, not seven individual calls. This experiment copies only the five source-bound Rust files behind those records into a separate temporary snapshot and measures exact invocation sites with the P1 collector. The original v11 graph remains the comparison baseline; the subset has its own graph hash. Method names remain candidates until compiler evidence identifies their targets.

The companion held-out compiler experiment uses the pinned, patched CGA producer and independent debug/overflow configurations. It tests source and capture integrity, typed standard versus custom methods, generic instances, cfg, dependency and macro locations. Its results describe those captured builds and inputs.

In [64]:
from pathlib import Path
import hashlib, json, tempfile

pilot_repo = Path('/Users/kevintruong/project/SpurCode')
pilot_old_artifacts = pilot_repo / 'docs/research/artifacts/2026-10-31-panic-graph-64c5b7a163b7'
pilot_frontier = json.loads((pilot_old_artifacts / 'method-frontier.json').read_text())
pilot_baseline = json.loads((pilot_old_artifacts / 'rust-source-bindings.json').read_text())
pilot_bound_rows = [dict(zip(pilot_baseline['columns'], row)) for row in pilot_baseline['rows']]
pilot_binding_by_path = {row['file_path']: row for row in pilot_bound_rows}
pilot_paths = sorted({row['file_path'] for row in pilot_frontier})
assert len(pilot_frontier) == 7 and len(pilot_paths) == 5
pilot_subset_root = Path(tempfile.mkdtemp(prefix='panic-method-subset-', dir='/private/tmp')).resolve()
pilot_subset_bindings = []
for rel in pilot_paths:
    source = (pilot_repo / rel).read_bytes()
    oid = hashlib.sha1(b'blob ' + str(len(source)).encode() + b'\0' + source).hexdigest()
    expected = pilot_binding_by_path[rel]
    assert oid == expected['indexed_blob_oid'] and len(source) == expected['byte_length'], rel
    dest = pilot_subset_root / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(source)
    pilot_subset_bindings.append({'file_path': rel, 'source_oid': oid, 'byte_length': len(source), 'sha256': hashlib.sha256(source).hexdigest()})
pilot_current = pilot_repo / '.spur/graph/CURRENT'
pilot_graph_path = pilot_current.resolve()
pilot_graph_manifest_sha = hashlib.sha256((pilot_graph_path / 'manifest.json').read_bytes()).hexdigest()
assert pilot_graph_manifest_sha == '1b9275fc691c61f990e278c8d3bcb2748dd03068a2dabd17b31750e2b8f50f56'
print(json.dumps({'baseline_graph': pilot_graph_path.name, 'manifest_sha256': pilot_graph_manifest_sha, 'frontier_records': len(pilot_frontier), 'subset_files': len(pilot_paths), 'subset_source_bytes': sum(row['byte_length'] for row in pilot_subset_bindings), 'all_source_bindings_match': True}, indent=2))

{
  "baseline_graph": "64c5b7a163b7c832c537236915db12b7cb7c6591e8e45bd072ea0404c826d8c4.parquet",
  "manifest_sha256": "1b9275fc691c61f990e278c8d3bcb2748dd03068a2dabd17b31750e2b8f50f56",
  "frontier_records": 7,
  "subset_files": 5,
  "subset_source_bytes": 799848,
  "all_source_bindings_match": true
}


In [66]:
import subprocess, time

pilot_runner_meta = json.loads(Path('/private/tmp/panic-evidence-root/runner.json').read_text())
pilot_runner = Path(pilot_runner_meta['path'])
assert hashlib.sha256(pilot_runner.read_bytes()).hexdigest() == pilot_runner_meta['sha256']
pilot_subset_request = {'command': 'snapshot', 'root': str(pilot_subset_root), 'source_paths': pilot_paths}
pilot_subset_request_file = Path('/private/tmp/panic-evidence-root/subset-request.json')
pilot_subset_request_file.write_text(json.dumps(pilot_subset_request))
pilot_subset_started = time.monotonic()
with pilot_subset_request_file.open('rb') as request_stream, open('/private/tmp/panic-evidence-root/subset-stdout.json', 'wb') as output_stream, open('/private/tmp/panic-evidence-root/subset-stderr.log', 'wb') as error_stream:
    pilot_subset_process = subprocess.Popen(['/usr/bin/time', '-l', str(pilot_runner)], stdin=request_stream, stdout=output_stream, stderr=error_stream)
print(json.dumps({'pid': pilot_subset_process.pid, 'state': 'running', 'previous_attempt': '90-second timeout; no result certified', 'limit_seconds': 600, 'stdout': '/private/tmp/panic-evidence-root/subset-stdout.json'}))

{"pid": 60363, "state": "running", "previous_attempt": "90-second timeout; no result certified", "limit_seconds": 600, "stdout": "/private/tmp/panic-evidence-root/subset-stdout.json"}


In [67]:
pilot_subset_status = pilot_subset_process.poll()
pilot_subset_elapsed = time.monotonic() - pilot_subset_started
if pilot_subset_status is None:
    print(json.dumps({'state': 'running', 'elapsed_seconds': round(pilot_subset_elapsed, 2), 'result_certified': False}))
else:
    assert pilot_subset_status == 0, Path('/private/tmp/panic-evidence-root/subset-stderr.log').read_text()
    pilot_subset = json.loads(Path('/private/tmp/panic-evidence-root/subset-stdout.json').read_text())
    print(json.dumps({'state': 'complete', 'elapsed_at_poll': round(pilot_subset_elapsed, 2), 'graph_keys': list(pilot_subset['graph']), 'method_evidence_keys': list(pilot_subset['method_sites']), 'first_site': pilot_subset['method_sites']['sites'][0]}, indent=2))

{
  "state": "complete",
  "elapsed_at_poll": 104.41,
  "graph_keys": [
    "header",
    "manifest_version",
    "graph_content_hash",
    "file_manifests",
    "files",
    "symbols",
    "edges",
    "macro_invocations",
    "tombstones"
  ],
  "method_evidence_keys": [
    "schema_version",
    "extractor_version",
    "graph_content_hash",
    "source_scope",
    "sources",
    "sites"
  ],
  "first_site": {
    "site_id": "2974296f0752d4a386a58b1c7f4e73dce88cc6deeaecfba11c32edeea52cbf85",
    "file_path": "crates/spur-graph/src/extract/tree_sitter.rs",
    "source_oid": "e785ddb9fee1e3dbdeee27a4f1bc177cb57956c1",
    "byte_range": [
      2466,
      2526
    ],
    "method_byte_range": [
      2491,
      2498
    ],
    "receiver_byte_range": [
      2466,
      2490
    ],
    "line_range": [
      79,
      79
    ],
    "call_text": "Query::new(language, \"\").context(\"compile empty tags query\")",
    "method_name": "context",
    "receiver_text": "Query::new(language, \"\

In [70]:
import re

pilot_subset_status = pilot_subset_process.poll()
assert pilot_subset_status == 0, Path('/private/tmp/panic-evidence-root/subset-stderr.log').read_text()
pilot_subset = json.loads(Path('/private/tmp/panic-evidence-root/subset-stdout.json').read_text())
pilot_method_sites = pilot_subset['method_sites']['sites']
assert len(pilot_subset['method_sites']['sources']) == len(pilot_paths)
assert all(site['compiler_target'] == 'unknown' for site in pilot_method_sites)
pilot_frontier_comparison = []
for record in pilot_frontier:
    symbols = [s for s in pilot_subset['graph']['symbols'] if s['stable_symbol_id'] == record['caller_id']]
    assert len(symbols) == 1 and symbols[0]['file_path'] == record['file_path']
    symbol = symbols[0]
    start, end = symbol['byte_range']
    sites = [s for s in pilot_method_sites if s['file_path'] == record['file_path'] and s['method_name'] == record['target_label'] and start <= s['byte_range'][0] and s['byte_range'][1] <= end]
    direct = [s for s in sites if s['caller_stable_symbol_id'] == record['caller_id']]
    closure = [s for s in sites if s['lexical_phase'] == 'closure_body_syntax']
    macros = []
    for macro in pilot_subset['graph']['macro_invocations']['rows']:
        a, b = macro['byte_range']
        if macro['file_path'] == record['file_path'] and start <= a and b <= end:
            text = (pilot_repo / record['file_path']).read_bytes()[a:b].decode()
            if re.search(r'\.\s*' + re.escape(record['target_label']) + r'\s*\(', text):
                macros.append({'macro_site': macro, 'token_text': text, 'interpretation': 'method-like tokens; no P1 method invocation or expansion claim'})
    state = 'exact function caller' if direct else ('closure syntax; outer function is lexical context only' if closure else ('macro token evidence only; method collector excludes token trees' if macros else 'no matching invocation evidence'))
    pilot_frontier_comparison.append({'caller': record['caller_name'], 'caller_id': record['caller_id'], 'file_path': record['file_path'], 'method': record['target_label'], 'original_label_records': 1, 'exact_function_sites': len(direct), 'closure_sites': len(closure), 'source_sites_in_function_range': len(sites), 'macro_token_witnesses': len(macros), 'state': state, 'sites': sites, 'macro_evidence': macros, 'compiler_target': 'Unknown'})
pilot_subset_timing = Path('/private/tmp/panic-evidence-root/subset-stderr.log').read_text()
pilot_subset_real_seconds = float(re.search(r'([0-9.]+) real', pilot_subset_timing).group(1))
pilot_subset_result = {'baseline_graph': pilot_graph_path.name, 'baseline_manifest_sha256': pilot_graph_manifest_sha, 'subset_root': str(pilot_subset_root), 'subset_graph_hash': pilot_subset['graph']['graph_content_hash'], 'source_bindings': pilot_subset_bindings, 'runner': pilot_runner_meta, 'elapsed_seconds': pilot_subset_real_seconds, 'subset_site_count': len(pilot_method_sites), 'comparison': pilot_frontier_comparison, 'timing': pilot_subset_timing}
Path('/private/tmp/panic-evidence-root/subset-result.json').write_text(json.dumps(pilot_subset_result, ensure_ascii=False, indent=2) + '\n')
print(json.dumps({'source_subset_sites_all_methods': len(pilot_method_sites), 'frontier_label_records': len(pilot_frontier_comparison), 'frontier_exact_function_sites': sum(r['exact_function_sites'] for r in pilot_frontier_comparison), 'frontier_closure_sites': sum(r['closure_sites'] for r in pilot_frontier_comparison), 'frontier_macro_token_witnesses': sum(r['macro_token_witnesses'] for r in pilot_frontier_comparison), 'all_compiler_targets': 'Unknown', 'elapsed_seconds_debug_snapshot': pilot_subset_real_seconds}, indent=2))
for row in pilot_frontier_comparison:
    print(row['caller'] + ': ' + row['state'])

{
  "source_subset_sites_all_methods": 6023,
  "frontier_label_records": 7,
  "frontier_exact_function_sites": 3,
  "frontier_closure_sites": 2,
  "frontier_macro_token_witnesses": 2,
  "all_compiler_targets": "Unknown",
  "elapsed_seconds_debug_snapshot": 103.27
}
run_full_walk_into_with_stats: exact function caller
write_edges: macro token evidence only; method collector excludes token trees
rebuild_from_buckets: closure syntax; outer function is lexical context only
stable_file_id_from_path: macro token evidence only; method collector excludes token trees
impl ExternalImportRegistry::write_to_buckets: closure syntax; outer function is lexical context only
compute_anchor_hash: exact function caller
resolve_bare_pending_edge: exact function caller


### What the original seven records actually cover

The source-bound subset contains 6,023 method invocations across all methods and contexts. Within the seven selected caller ranges, the new evidence distinguishes:

| Legacy caller | Method label | Measured evidence |
|---|---|---|
| `run_full_walk_into_with_stats` | `expect` | 1 exact function-owned invocation |
| `compute_anchor_hash` | `expect` | 1 exact function-owned invocation |
| `resolve_bare_pending_edge` | `expect` | 1 exact function-owned invocation |
| `rebuild_from_buckets` | `expect` | 1 closure invocation; outer function is lexical context |
| `ExternalImportRegistry::write_to_buckets` | `expect` | 1 closure invocation; outer function is lexical context |
| `write_edges` | `expect` | Method-like tokens within a macro; excluded from P1 invocation rows |
| `stable_file_id_from_path` | `unwrap` | Method-like tokens within `format!`; excluded from P1 invocation rows |

Thus **five individual AST call sites are recovered**, with **three exact outer-function callers**. The two macro cases remain explicit token evidence. All seven compiler targets remain **Unknown** in this workspace experiment. A lexical closure association does not establish that the outer function invokes the closure.

The bounded debug snapshot took **103.27s wall / 101.10s user / 0.50s system**. A one-second sample at 16s placed all 695 sampled main-thread stacks in existing macro extraction and validation, before method collection. This identifies a snapshot-construction cost to investigate separately; it is not a benchmark of the method collector alone. The first 90-second attempt produced no certified result.

The most effective next coverage steps are compiler-bound closure ownership and macro expansion locations, plus a path that reuses the existing graph artifact when collecting source evidence. Raising a name-match confidence score would not resolve those gaps.

In [71]:
# Fresh compiler capture from the integrated pilot, initiated through Notebook MCP.
import os, subprocess, time

pilot_capture_root = Path('/private/tmp/panic-evidence-root/notebook-capture-01')
assert not pilot_capture_root.exists(), 'choose a fresh capture directory; existing evidence is immutable'
pilot_capture_script = pilot_repo / 'scripts/experiments/panic-evidence-capture.py'
pilot_capture_script_sha = hashlib.sha256(pilot_capture_script.read_bytes()).hexdigest()
pilot_capture_argv = ['python3', str(pilot_capture_script), '--work-dir', str(pilot_capture_root), '--cga-driver', '/private/tmp/spur-cga-pilot-20261031/target/release/cga-driver', '--cargo-cga', '/private/tmp/spur-cga-pilot-20261031/target/release/cargo-cga', '--cga-source', '/private/tmp/spur-cga-pilot-20261031', '--producer-patch', '/private/tmp/spur-cga-pilot-macos-rpath.patch', '--expected-driver-sha256', '5647b7f2c7f857dda0dd0204e91d870287988181ca5ec4d7792810a561886172', '--expected-launcher-sha256', '7f84bc715138c09744fd351859ca02b240ea768d9756069229e9250fa9447b7a', '--example', '/private/tmp/panic-evidence-p1/target/debug/examples/panic_evidence']
pilot_capture_env = os.environ.copy()
pilot_capture_env.update(SPUR_REMOTE='0', TMPDIR='/private/tmp')
pilot_capture_env.pop('SPUR_GRAPH_BLESS', None)
pilot_capture_started = time.monotonic()
pilot_capture_process = subprocess.run(pilot_capture_argv, cwd=pilot_repo, env=pilot_capture_env, capture_output=True, text=True, timeout=90)
pilot_capture_elapsed = time.monotonic() - pilot_capture_started
Path('/private/tmp/panic-evidence-root/notebook-capture.stdout').write_text(pilot_capture_process.stdout)
Path('/private/tmp/panic-evidence-root/notebook-capture.stderr').write_text(pilot_capture_process.stderr)
assert pilot_capture_process.returncode == 0, pilot_capture_process.stderr
assert hashlib.sha256(pilot_capture_script.read_bytes()).hexdigest() == pilot_capture_script_sha
assert pilot_current.resolve() == pilot_graph_path
assert hashlib.sha256((pilot_graph_path / 'manifest.json').read_bytes()).hexdigest() == pilot_graph_manifest_sha
pilot_capture_run = {'argv': pilot_capture_argv, 'cwd': str(pilot_repo), 'environment_overrides': {'SPUR_REMOTE': '0', 'TMPDIR': '/private/tmp'}, 'unset': ['SPUR_GRAPH_BLESS'], 'exit': pilot_capture_process.returncode, 'elapsed_seconds': pilot_capture_elapsed, 'script_sha256': pilot_capture_script_sha, 'stdout_sha256': hashlib.sha256(pilot_capture_process.stdout.encode()).hexdigest(), 'stderr_sha256': hashlib.sha256(pilot_capture_process.stderr.encode()).hexdigest(), 'graph_pointer_unchanged': True, 'graph_manifest_unchanged': True}
Path('/private/tmp/panic-evidence-root/notebook-capture-run.json').write_text(json.dumps(pilot_capture_run, indent=2) + '\n')
print(json.dumps({'exit': pilot_capture_process.returncode, 'elapsed_seconds': round(pilot_capture_elapsed, 2), 'graph_pointer_unchanged': True, 'summary': pilot_capture_process.stdout.strip()}, indent=2))

{
  "exit": 0,
  "elapsed_seconds": 5.43,
  "graph_pointer_unchanged": true,
  "summary": "{\"schema\": \"panic-pilot-acceptance-v1\", \"producer\": {\"build\": {\"state\": \"patched\", \"patch_sha256\": \"72f3d4cd64439d988deba0ee1e0f449df8290e6f1d1e0ae83ba82d97b1592aca\"}, \"cargo_cga_sha256\": \"7f84bc715138c09744fd351859ca02b240ea768d9756069229e9250fa9447b7a\", \"cga_driver_sha256\": \"5647b7f2c7f857dda0dd0204e91d870287988181ca5ec4d7792810a561886172\"}, \"example_sha256\": \"ff60430d2c2d1fe151cb0de49067553f9c2039ccc632159c620a7aa057e0fa8b\", \"wrapper_sha256\": \"4e2e23d8c38abc8948f4b226d709cdfe0991dde3420d6842935805bb7a603ed4\", \"rustc_verbose\": \"rustc 1.101.0-nightly (8006091eb 2026-10-17)\\nbinary: rustc\\ncommit-hash: 8006091eb71ae7d9e9bccb66c6122c9d68af4466\\ncommit-date: 2026-10-17\\nhost: aarch64-apple-darwin\\nrelease: 1.101.0-nightly\\nLLVM version: 23.1.4\\n\", \"configurations\": [{\"name\": \"d0_o0\", \"build_id\": \"2b900ecac41d17532151aacd1dde8003d48c737d7dc3811049e

In [72]:
from collections import Counter
from IPython.display import display, Markdown

pilot_capture_summary = json.loads((pilot_capture_root / 'summary.json').read_text())
assert pilot_capture_summary['mandatory_gate'] == 'passed'
assert len(pilot_capture_summary['configurations']) == 5
pilot_config_rows = []
pilot_runtime_rows = []
pilot_compiler_sets = {}
for config in pilot_capture_summary['configurations']:
    name = config['name']
    directory = pilot_capture_root / name
    raw_bytes = (directory / 'raw-export.json').read_bytes()
    assert hashlib.sha256(raw_bytes).hexdigest() == config['raw_export_sha256']
    assert (directory / 'inventory-before.json').read_bytes() == (directory / 'inventory-after.json').read_bytes()
    evidence = json.loads((directory / 'import.stdout').read_text())['compiler_evidence']
    assert evidence['build_id'] == config['build_id']
    pilot_compiler_sets[name] = evidence
    associations = Counter(o['association']['state'] for o in evidence['observations'])
    assert dict(associations) == {k: v for k, v in config['association_counts'].items() if v}
    controls = json.loads((directory / 'runtime.json').read_text())
    for row in controls:
        stderr = (directory / ('runtime-' + row['mode'] + '.stderr')).read_text()
        assert (row['exit'] == 101 and 'panicked at' in stderr) if row['expected_panics'] else row['exit'] == 0
        pilot_runtime_rows.append({'configuration': name, **row})
    assert len(controls) == 9 and len(config['negative_controls_passed']) == 5
    for negative in ('source', 'ownership'):
        assert (directory / ('negative-' + negative + '.witness.json')).is_file()
    pilot_config_rows.append({'configuration': name, 'observations': len(evidence['observations']), 'exact': associations['exact'], 'unique_containing': associations['unique_containing'], 'unmapped': associations['unmapped'], 'external': associations['external'], 'runtime_children': len(controls), 'panic_exits': sum(r['exit'] == 101 for r in controls)})
assert len({r['build_id'] for r in pilot_capture_summary['configurations']}) == 5
assert len(pilot_runtime_rows) == 45
assert sum(r['exit'] == 101 for r in pilot_runtime_rows) == 32
print('All five captured builds, source inventories and 45 executable controls verified; 32 panic exits, 13 normal exits.')
print(json.dumps(pilot_config_rows, indent=2))
Path('/private/tmp/panic-evidence-root/configuration-analysis.json').write_text(json.dumps({'configurations': pilot_config_rows, 'runtime': pilot_runtime_rows}, indent=2) + '\n')

All five captured builds, source inventories and 45 executable controls verified; 32 panic exits, 13 normal exits.
[
  {
    "configuration": "d0_o0",
    "observations": 134,
    "exact": 18,
    "unique_containing": 1,
    "unmapped": 55,
    "external": 60,
    "runtime_children": 9,
    "panic_exits": 5
  },
  {
    "configuration": "d0_o1",
    "observations": 135,
    "exact": 18,
    "unique_containing": 1,
    "unmapped": 56,
    "external": 60,
    "runtime_children": 9,
    "panic_exits": 6
  },
  {
    "configuration": "d1_o0",
    "observations": 135,
    "exact": 18,
    "unique_containing": 1,
    "unmapped": 55,
    "external": 61,
    "runtime_children": 9,
    "panic_exits": 6
  },
  {
    "configuration": "d1_o1",
    "observations": 136,
    "exact": 18,
    "unique_containing": 1,
    "unmapped": 56,
    "external": 61,
    "runtime_children": 9,
    "panic_exits": 7
  },
  {
    "configuration": "d1_o1_extra",
    "observations": 137,
    "exact": 19,
    "unique_c

6500

In [75]:
pilot_high = pilot_compiler_sets['d1_o1']
pilot_observations = pilot_high['observations']
def pilot_from(caller):
    return [o for o in pilot_observations if o['canonical_caller'] == caller]

pilot_typed_cases = []
for caller, expected_count, target_fragment in [('panic_pilot::options', 4, 'std::option::Option'), ('panic_pilot::results', 4, 'std::result::Result'), ('panic_pilot::custom', 2, 'panic_pilot::Safe::')]:
    rows = [o for o in pilot_from(caller) if target_fragment in (o['canonical_callee'] or '')]
    assert len(rows) == expected_count
    assert all(o['association']['state'] == 'exact' for o in rows)
    pilot_typed_cases.append({'caller': caller, 'observations': len(rows), 'canonical_callees': sorted({o['canonical_callee'] for o in rows}), 'source_site_ids': sorted({o['association']['site_id'] for o in rows})})
pilot_generics = [o for o in pilot_observations if o['canonical_caller'].startswith('panic_pilot::generic::<')]
assert len(pilot_generics) == 2
assert len({o['canonical_caller'] for o in pilot_generics}) == 2
assert len({o['association']['site_id'] for o in pilot_generics}) == 1
pilot_unicode = [o for o in pilot_from('panic_pilot::unicode') if o['association']['state'] == 'exact']
assert len(pilot_unicode) == 1
pilot_unicode_obs = pilot_unicode[0]
pilot_unicode_source = (pilot_capture_root / 'fixture/src/lib.rs').read_bytes()
a, b = pilot_unicode_obs['converted_location']['byte_range']
assert pilot_unicode_source[a:b].decode() == 'x.expect("unicode")'
assert all(v == 'unavailable' for v in pilot_high['capabilities'].values())
pilot_check_rows = []
for name, evidence in pilot_compiler_sets.items():
    rows = evidence['observations']
    count = lambda caller, pred: sum(o['canonical_caller'] == caller and pred(o) for o in rows)
    pilot_check_rows.append({'configuration': name, 'index_builtin_asserts': count('panic_pilot::indexing', lambda o: o['kind'] == 'BuiltinAssert'), 'addition_builtin_asserts': count('panic_pilot::addition', lambda o: o['kind'] == 'BuiltinAssert'), 'debug_panic_calls': count('panic_pilot::debug_control', lambda o: 'panicking::panic' in (o['canonical_callee'] or '')), 'extra_feature_calls': count('panic_pilot::feature_control', lambda o: o['canonical_callee'] == 'std::option::Option::<u8>::expect')})
assert [r['index_builtin_asserts'] for r in pilot_check_rows] == [1, 1, 1, 1, 1]
assert [r['addition_builtin_asserts'] for r in pilot_check_rows] == [0, 1, 0, 1, 1]
assert [r['debug_panic_calls'] for r in pilot_check_rows] == [0, 0, 1, 1, 1]
assert [r['extra_feature_calls'] for r in pilot_check_rows] == [0, 0, 0, 0, 1]
pilot_typed_analysis = {'typed_cases': pilot_typed_cases, 'generics': pilot_generics, 'unicode': pilot_unicode_obs, 'configuration_checks': pilot_check_rows, 'capabilities': pilot_high['capabilities']}
_ = Path('/private/tmp/panic-evidence-root/typed-analysis.json').write_text(json.dumps(pilot_typed_analysis, indent=2) + '\n')
print(json.dumps({'typed_cases': pilot_typed_cases, 'generic_instances': [o['canonical_caller'] for o in pilot_generics], 'generic_source_sites': 1, 'unicode_span_bytes': [a, b], 'configuration_checks': pilot_check_rows, 'capabilities': pilot_high['capabilities']}, indent=2))

{
  "typed_cases": [
    {
      "caller": "panic_pilot::options",
      "observations": 4,
      "canonical_callees": [
        "std::option::Option::<u8>::expect",
        "std::option::Option::<u8>::unwrap"
      ],
      "source_site_ids": [
        "0df106b6f5f761d62ca8f7d78fb53baa956dd4da25faf7d60cdb2474464f4757",
        "ad38c513ef92e0ab89e452f0a44278e3c9bb9b82d55e5d8dc8d4502c488696cb",
        "d85a86b8a729403e4942189b7093d54a97f8b25b76e4ceebd60b97ff3d61cee0",
        "fca1c450817f1e5cbe6ba9cbabd2b61edc09abc2848647313de8858e96cefa4e"
      ]
    },
    {
      "caller": "panic_pilot::results",
      "observations": 4,
      "canonical_callees": [
        "std::result::Result::<u8, u8>::expect",
        "std::result::Result::<u8, u8>::unwrap"
      ],
      "source_site_ids": [
        "343db53bbfe01101380f6fd5f35baa3c55a3c3b4c2266892282511a0799e2938",
        "7a37c77d850b7e7d1f22953b96e56108adb82717cb5fecac8815221b0b2717f7",
        "a2a4573dba325254e69e14043cf9a8779163103c5c

In [74]:
pilot_reviewer_checks = []
for config in pilot_capture_summary['configurations']:
    env = os.environ.copy()
    env['P3_REVIEW_CAPTURE'] = str(pilot_capture_root / config['name'])
    checked = subprocess.run(['python3', '/private/tmp/panic-evidence-p3/brain-control-contract.py'], env=env, cwd=pilot_repo, capture_output=True, text=True, timeout=10)
    assert checked.returncode == 0, checked.stderr
    pilot_reviewer_checks.append({'configuration': config['name'], 'exit': checked.returncode, 'stdout': checked.stdout, 'stderr': checked.stderr})
_ = Path('/private/tmp/panic-evidence-root/notebook-control-review.json').write_text(json.dumps(pilot_reviewer_checks, indent=2) + '\n')
print('Independent saved-payload review: 10/10 assertions passed across five fresh configurations. Source control changes exactly one physical .rs Source input; ownership control changes exactly one caller to a different unique existing function/method in the same file.')

Independent saved-payload review: 10/10 assertions passed across five fresh configurations. Source control changes exactly one physical .rs Source input; ownership control changes exactly one caller to a different unique existing function/method in the same file.


### Fresh compiler result and effectiveness

The integrated pilot completed a new five-build capture through Notebook MCP in **5.43s**, including 45 child executions and 26 corruption/substitution rejections. The executable controls returned **32 panic exits and 13 normal exits** as expected. Five of the normal exits are the successful input batch; the other eight reflect disabled debug, overflow or feature behavior.

| Configuration | Compiler observations | Exact site | Unique containing site | Unmapped | External | Panic exits / 9 |
|---|---:|---:|---:|---:|---:|---:|
| `d0_o0` | 134 | 18 | 1 | 55 | 60 | 5 |
| `d0_o1` | 135 | 18 | 1 | 56 | 60 | 6 |
| `d1_o0` | 135 | 18 | 1 | 55 | 61 | 6 |
| `d1_o1` | 136 | 18 | 1 | 56 | 61 | 7 |
| `d1_o1_extra` | 137 | 19 | 1 | 56 | 61 | 8 |

Four Option and four Result calls retain separate exact source sites. Two same-spelled custom calls resolve to `panic_pilot::Safe::unwrap` / `Safe::expect`. The generic `u8` and `u16` instances share one source site without collapsing compiler observations. The Unicode case binds bytes **887..906**, exactly `x.expect("unicode")`, after a multibyte prefix.

| Compiler check | `d0_o0` | `d0_o1` | `d1_o0` | `d1_o1` | `d1_o1_extra` |
|---|---:|---:|---:|---:|---:|
| Index `BuiltinAssert` | 1 | 1 | 1 | 1 | 1 |
| Addition `BuiltinAssert` | 0 | 1 | 0 | 1 | 1 |
| Debug panic call | 0 | 0 | 1 | 1 | 1 |
| Feature-controlled Option expect | 0 | 0 | 0 | 0 | 1 |

These are compiler observations for the held-out corpus. Counts include standard/runtime code and assertions, so `19 / 136` is **not a workspace recall score**. Receiver types, assertion reasons and expansion maps remain unavailable. The macro-produced call survives as a compiler observation but has no P1 invocation association; the local dependency's direct expect does associate exactly.

The independent saved-request review passed **10/10 assertions**: source tampering changes only one physical `.rs` Source input, and caller tampering substitutes a different unique existing function/method in the same file. Source, configuration, checksum, reference and ownership failures are kept separately.

The original workspace gap is now measurable rather than hidden: three exact callers, two closure sites and two macro-token cases, with no workspace target resolution claimed. The next useful work is to import compiler captures for selected real workspace targets, resolve closure instances, add expansion-to-invocation locations, and reuse the existing graph during evidence collection. These steps need their own held-out controls and coverage denominators.

[Durable evidence and replay index](docs/research/artifacts/2026-11-01-panic-evidence-pilot/README.md) · [Root verification](docs/research/artifacts/2026-11-01-panic-evidence-pilot/verification.json)

In [76]:
# Standalone replay: no temporary capture directory, compiler, shellfs or graph MCP required.
from pathlib import Path, PurePosixPath
import hashlib, json, tarfile
from IPython.display import display, Markdown

pilot_evidence_dir = Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-panic-evidence-pilot')
pilot_index_bytes = (pilot_evidence_dir / 'evidence-index.json').read_bytes()
pilot_index = json.loads(pilot_index_bytes)
pilot_payloads_checked = 0
for row in pilot_index['payloads']:
    data = (pilot_evidence_dir / row['path']).read_bytes()
    assert len(data) == row['byte_length'] and hashlib.sha256(data).hexdigest() == row['sha256'], row['path']
    pilot_payloads_checked += 1
pilot_archive_entries = {}
for name, entries in pilot_index['archives'].items():
    expected = {row['path']: row for row in entries}
    loaded = {}
    with tarfile.open(pilot_evidence_dir / name, 'r:gz') as archive:
        assert {m.name for m in archive.getmembers()} == set(expected)
        for member in archive.getmembers():
            path = PurePosixPath(member.name)
            assert member.isfile() and not path.is_absolute() and '..' not in path.parts
            data = archive.extractfile(member).read()
            row = expected[member.name]
            assert len(data) == row['byte_length'] and hashlib.sha256(data).hexdigest() == row['sha256']
            loaded[member.name] = data
    pilot_archive_entries[name] = loaded
pilot_archived_capture = pilot_archive_entries['capture.tar.gz']
pilot_archive_json = lambda path: json.loads(pilot_archived_capture[path])
pilot_saved_summary = pilot_archive_json('summary.json')
pilot_saved_snapshot = pilot_archive_json('snapshot.json')
pilot_declared_sites = {s['site_id'] for s in pilot_saved_snapshot['method_sites']['sites']}
assert len(pilot_declared_sites) == 18
pilot_site_coverage = []
for config in pilot_saved_summary['configurations']:
    name = config['name']
    assert hashlib.sha256(pilot_archived_capture[name + '/raw-export.json']).hexdigest() == config['raw_export_sha256']
    evidence = pilot_archive_json(name + '/import.stdout')['compiler_evidence']
    associated = {o['association']['site_id'] for o in evidence['observations'] if o['association']['state'] in ('exact', 'unique_containing')}
    assert associated <= pilot_declared_sites
    missing = pilot_declared_sites - associated
    assert len(missing) == (0 if name.endswith('_extra') else 1)
    if missing:
        only = next(s for s in pilot_saved_snapshot['method_sites']['sites'] if s['site_id'] in missing)
        assert only['call_text'] == 'x.expect("feature")'
    pilot_site_coverage.append({'configuration': name, 'associated_unique_sites': len(associated), 'declared_ast_sites': len(pilot_declared_sites), 'unassociated': len(missing)})
pilot_archived_workspace = json.loads(pilot_archive_entries['workspace-subset.tar.gz']['snapshot.json'])
pilot_saved_frontier = json.loads((pilot_evidence_dir / 'workspace-frontier.json').read_text())
assert len(pilot_archived_workspace['method_sites']['sites']) == 6023
assert sum(r['exact_function_sites'] for r in pilot_saved_frontier['comparison']) == 3
assert sum(r['closure_sites'] for r in pilot_saved_frontier['comparison']) == 2
assert sum(r['macro_token_witnesses'] for r in pilot_saved_frontier['comparison']) == 2
print(f"Verified {pilot_payloads_checked} durable payloads and {sum(len(v) for v in pilot_archive_entries.values())} archive members without extraction.")
print('All capture results can be replayed after the temporary directories are removed.')
print('Declared fixture source-site association:')
print(json.dumps(pilot_site_coverage, indent=2))
display(Markdown('The held-out corpus associates **17/18 declared AST sites** in each base configuration and **18/18 with `extra` enabled**. The missing base site is the inactive cfg-gated `x.expect("feature")`. Macro token trees are excluded from this denominator. This measures source association, not panic safety or whole-workspace recall.'))

Verified 11 durable payloads and 496 archive members without extraction.
All capture results can be replayed after the temporary directories are removed.
Declared fixture source-site association:
[
  {
    "configuration": "d0_o0",
    "associated_unique_sites": 17,
    "declared_ast_sites": 18,
    "unassociated": 1
  },
  {
    "configuration": "d0_o1",
    "associated_unique_sites": 17,
    "declared_ast_sites": 18,
    "unassociated": 1
  },
  {
    "configuration": "d1_o0",
    "associated_unique_sites": 17,
    "declared_ast_sites": 18,
    "unassociated": 1
  },
  {
    "configuration": "d1_o1",
    "associated_unique_sites": 17,
    "declared_ast_sites": 18,
    "unassociated": 1
  },
  {
    "configuration": "d1_o1_extra",
    "associated_unique_sites": 18,
    "declared_ast_sites": 18,
    "unassociated": 0
  }
]


The held-out corpus associates **17/18 declared AST sites** in each base configuration and **18/18 with `extra` enabled**. The missing base site is the inactive cfg-gated `x.expect("feature")`. Macro token trees are excluded from this denominator. This measures source association, not panic safety or whole-workspace recall.

## Real workspace compilation — 2026-11-01

This experiment compiles the actual `spur-graph` library, with default embedding features disabled, through the pinned patched CGA driver and `scripts/spur-cargo`. It uses an isolated output directory and records compiler identity, selected build flags, Cargo manifests/configuration, declared workspace build inputs and before/after hashes. The published graph is the same v11 baseline.

This is **raw workspace compiler evidence**, separate from the accepted closed-fixture importer (`CaptureScope::BoundedFixture`). No fixture label is assigned to the workspace capture. We will measure which of the seven source-bound frontier cases the selected library target emits, preserving closure/macro ownership and any unmapped, external or missing observations. Compilation is not a runtime test or a proof of panic freedom.

In [77]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, time

ws_repo = Path('/Users/kevintruong/project/SpurCode')
ws_work = Path('/private/tmp/panic-workspace-20261101')
ws_driver = Path('/private/tmp/spur-cga-pilot-20261031/target/release/cga-driver')
ws_cargo_cga = ws_driver.with_name('cargo-cga')
ws_expected_driver = '5647b7f2c7f857dda0dd0204e91d870287988181ca5ec4d7792810a561886172'
ws_expected_launcher = '7f84bc715138c09744fd351859ca02b240ea768d9756069229e9250fa9447b7a'
ws_digest = lambda data: hashlib.sha256(data).hexdigest()
assert ws_digest(ws_driver.read_bytes()) == ws_expected_driver
assert ws_digest(ws_cargo_cga.read_bytes()) == ws_expected_launcher
ws_tracked = subprocess.check_output(['git', 'ls-files', '-z'], cwd=ws_repo).decode().split('\0')
ws_build_suffixes = {'.rs', '.toml', '.lock', '.scm', '.c', '.h', '.cpp', '.cc', '.proto', '.sh'}
ws_declared_paths = sorted(p for p in ws_tracked if p and Path(p).suffix in ws_build_suffixes and (ws_repo / p).is_file())
def ws_inventory():
    return [{'path': p, 'byte_length': (ws_repo / p).stat().st_size, 'sha256': ws_digest((ws_repo / p).read_bytes())} for p in ws_declared_paths]
ws_before_inventory = ws_inventory()
_ = (ws_work / 'workspace-inputs-before.json').write_text(json.dumps(ws_before_inventory, indent=2) + '\n')
ws_cargo_home = Path(os.environ.get('CARGO_HOME', str(Path.home() / '.cargo'))).resolve()
ws_rustup_home = Path(os.environ.get('RUSTUP_HOME', str(Path.home() / '.rustup'))).resolve()
ws_configs = []
for parent in [ws_repo, *ws_repo.parents, ws_cargo_home]:
    for path in [parent / '.cargo/config.toml', parent / '.cargo/config'] if parent != ws_cargo_home else [parent / 'config.toml', parent / 'config']:
        if path.is_file() and str(path) not in {r['path'] for r in ws_configs}:
            ws_configs.append({'path': str(path), 'sha256': ws_digest(path.read_bytes()), 'byte_length': path.stat().st_size})
ws_flags = ['-Zalways-encode-mir', '-Zhint-mostly-unused', '-Copt-level=0', '-Cpanic=unwind', '-Cdebug-assertions=on', '-Coverflow-checks=on', '-Cforce-frame-pointers=yes']
ws_env = {'PATH': os.pathsep.join([str(ws_cargo_home / 'bin'), '/opt/homebrew/bin', '/usr/bin', '/bin', '/usr/sbin', '/sbin']), 'RUSTUP_HOME': str(ws_rustup_home), 'CARGO_HOME': str(ws_cargo_home), 'RUSTUP_TOOLCHAIN': 'nightly-2026-10-18', 'SPUR_REMOTE': '0', 'SPUR_SCCACHE_S3': '0', 'RUSTC': str(ws_driver), 'RUSTC_WRAPPER': '', 'CARGO_INCREMENTAL': '0', 'CARGO_TARGET_DIR': str(ws_work / 'target'), 'RUSTFLAGS': ' '.join(ws_flags), 'LC_ALL': 'C', 'TMPDIR': str(ws_work)}
ws_rustc = subprocess.run([str(ws_driver), '-vV'], cwd=ws_repo, env=ws_env, capture_output=True, text=True, timeout=15)
assert ws_rustc.returncode == 0, ws_rustc.stderr
assert 'commit-hash: 8006091eb71ae7d9e9bccb66c6122c9d68af4466' in ws_rustc.stdout
ws_argv = [str(ws_repo / 'scripts/spur-cargo'), 'build', '--frozen', '--offline', '-p', 'spur-graph', '--lib', '--no-default-features', '--target', 'aarch64-apple-darwin', '-vv']
ws_plan = {'schema': 'panic-workspace-raw-capture-v1', 'scope': 'raw_workspace_library', 'typed_importer_supported': False, 'issue': 'bd-1lb', 'source_head': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ws_repo, text=True).strip(), 'argv': ws_argv, 'cwd': str(ws_repo), 'environment': ws_env, 'compiler_flags': ws_flags, 'rustc_verbose': ws_rustc.stdout, 'producer_revision': '2fc63108f0b674fe7a178191ffa15647a18879f1', 'producer_driver_sha256': ws_expected_driver, 'producer_launcher_sha256': ws_expected_launcher, 'producer_patch_sha256': '72f3d4cd64439d988deba0ee1e0f449df8290e6f1d1e0ae83ba82d97b1592aca', 'cargo_configs': ws_configs, 'declared_input_count': len(ws_before_inventory), 'inventory_scope': 'tracked workspace Rust, manifests, locks, queries, C/C++/proto and shell inputs; registry/git dependencies selected by Cargo.lock; external build effects not claimed closed', 'graph_target': str((ws_repo / '.spur/graph/CURRENT').resolve()), 'graph_manifest_sha256': ws_digest(((ws_repo / '.spur/graph/CURRENT').resolve() / 'manifest.json').read_bytes())}
assert not (ws_work / 'compile-plan.json').exists(), 'existing capture is immutable; use a fresh directory'
_ = (ws_work / 'compile-plan.json').write_text(json.dumps(ws_plan, indent=2) + '\n')
ws_started = time.monotonic()
with (ws_work / 'compiler.stdout').open('wb') as stdout, (ws_work / 'compiler.stderr').open('wb') as stderr:
    ws_process = subprocess.Popen(ws_argv, cwd=ws_repo, env=ws_env, stdout=stdout, stderr=stderr)
print(json.dumps({'state': 'running', 'pid': ws_process.pid, 'package': 'spur-graph', 'target': 'lib, no default features', 'declared_inputs': len(ws_before_inventory), 'rustc': ws_rustc.stdout.splitlines()[0], 'capture_directory': str(ws_work)}, indent=2))

{
  "state": "running",
  "pid": 86168,
  "package": "spur-graph",
  "target": "lib, no default features",
  "declared_inputs": 1711,
  "rustc": "rustc 1.101.0-nightly (8006091eb 2026-10-17)",
  "capture_directory": "/private/tmp/panic-workspace-20261101"
}


In [80]:
ws_exit = ws_process.poll()
ws_elapsed_at_poll = time.monotonic() - ws_started
ws_log = (ws_work / 'compiler.stderr').read_text(errors='replace')
print(json.dumps({'state': 'running' if ws_exit is None else 'finished', 'exit': ws_exit, 'elapsed_seconds_at_poll': round(ws_elapsed_at_poll, 2), 'stderr_bytes': (ws_work / 'compiler.stderr').stat().st_size, 'stdout_bytes': (ws_work / 'compiler.stdout').stat().st_size, 'recent_log': [line[:240] for line in ws_log.splitlines()[-8:]]}, indent=2))
if ws_exit is not None:
    ws_after_inventory = ws_inventory()
    _ = (ws_work / 'workspace-inputs-after.json').write_text(json.dumps(ws_after_inventory, indent=2) + '\n')
    ws_final = {'exit': ws_exit, 'elapsed_seconds_at_poll': ws_elapsed_at_poll, 'source_inputs_unchanged': ws_after_inventory == ws_before_inventory, 'stdout_sha256': ws_digest((ws_work / 'compiler.stdout').read_bytes()), 'stderr_sha256': ws_digest((ws_work / 'compiler.stderr').read_bytes()), 'graph_pointer_unchanged': str((ws_repo / '.spur/graph/CURRENT').resolve()) == ws_plan['graph_target'], 'graph_manifest_unchanged': ws_digest((Path(ws_plan['graph_target']) / 'manifest.json').read_bytes()) == ws_plan['graph_manifest_sha256']}
    _ = (ws_work / 'compile-result.json').write_text(json.dumps(ws_final, indent=2) + '\n')
    print(json.dumps(ws_final, indent=2))

{
  "state": "finished",
  "exit": 0,
  "elapsed_seconds_at_poll": 129.44,
  "stderr_bytes": 3177100,
  "stdout_bytes": 212430,
  "recent_log": [
    "    |     ----------------- method in this implementation",
    "908 |         fn as_bool(&self) -> Option<bool> {",
    "    |            ^^^^^^^",
    "    |",
    "    = note: `#[warn(dead_code)]` (part of `#[warn(unused)]`) on by default",
    "",
    "warning: `spur-graph` (lib) generated 2 warnings (run `cargo fix --lib -p spur-graph` to apply 1 suggestion)",
    "    Finished `dev` profile [unoptimized + debuginfo] target(s) in 1m 18s"
  ]
}


{
  "exit": 0,
  "elapsed_seconds_at_poll": 129.44279825000558,
  "source_inputs_unchanged": true,
  "stdout_sha256": "6536b0fba54c2b8e281d593512bfd9f479709104745b058328d67e12d9e3141b",
  "stderr_sha256": "3890c2539e5dce1fe461ef9a23344b9ffbd6cdfec5ec9e7a32024fe02fc4189b",
  "graph_pointer_unchanged": true,
  "graph_manifest_unchanged": true
}


In [79]:
import tarfile

ws_old_dossier = ws_repo / 'docs/research/artifacts/2026-11-01-panic-evidence-pilot'
ws_old_index = json.loads((ws_old_dossier / 'evidence-index.json').read_text())
ws_subset_archive = ws_old_dossier / 'workspace-subset.tar.gz'
ws_archive_row = next(r for r in ws_old_index['payloads'] if r['path'] == 'workspace-subset.tar.gz')
assert ws_digest(ws_subset_archive.read_bytes()) == ws_archive_row['sha256']
with tarfile.open(ws_subset_archive, 'r:gz') as archive:
    ws_source_graph = json.loads(archive.extractfile('snapshot.json').read())
    ws_frozen_sources = json.loads(archive.extractfile('sources.json').read())
ws_sources = {r['file_path']: r['text'].encode() for r in ws_frozen_sources}
ws_source_bindings = []
for row in ws_source_graph['method_sites']['sources']:
    path = row['file_path']
    data = (ws_repo / path).read_bytes()
    blob = hashlib.sha1(b'blob ' + str(len(data)).encode() + b'\0' + data).hexdigest()
    assert data == ws_sources[path] and blob == row['source_oid']
    ws_source_bindings.append({'file_path': path, 'source_oid': blob, 'sha256': ws_digest(data), 'byte_length': len(data)})
ws_frontier = json.loads((ws_old_dossier / 'workspace-frontier.json').read_text())['comparison']
ws_frontier_symbols = {}
for row in ws_frontier:
    matches = [s for s in ws_source_graph['graph']['symbols'] if s['stable_symbol_id'] == row['caller_id']]
    assert len(matches) == 1 and matches[0]['file_path'] == row['file_path']
    ws_frontier_symbols[row['caller_id']] = matches[0]
_ = (ws_work / 'frontier-source-bindings.json').write_text(json.dumps({'reused_subset_graph_hash': ws_source_graph['graph']['graph_content_hash'], 'published_graph_hash': Path(ws_plan['graph_target']).name.removesuffix('.parquet'), 'sources': ws_source_bindings, 'frontier': ws_frontier, 'symbols': ws_frontier_symbols}, indent=2) + '\n')
print('Reused validated source subset: all five files match their archived bytes and original graph blob OIDs. Seven frontier symbols retain their exact source ranges. No graph re-extraction or CURRENT update was needed.')

Reused validated source subset: all five files match their archived bytes and original graph blob OIDs. Seven frontier symbols retain their exact source ranges. No graph re-extraction or CURRENT update was needed.


### Compile baseline and extraction switch

The real library compiled successfully in **1m 18s** (Cargo-reported duration), exit 0, with all 1,711 declared input hashes unchanged. The first command omitted `CGA_INPUT`, so the pinned driver ran with extraction disabled and emitted **zero `cga-emit:` records**. This run establishes compilation only and is not accepted as compiler-graph evidence.

The next command explicitly sets `CGA_INPUT={"extract":true,"lint":null}`. The upstream driver tracks this environment variable in dependency information, so Cargo can invalidate the affected compiler invocations while retaining the same source, target and flags. Both command identities and their logs are retained separately. The poll elapsed value includes time before the notebook polled; it is not the exact compile duration.

In [81]:
ws_extract_dir = ws_work / 'extract-d1_o1'
ws_extract_dir.mkdir(exist_ok=False)
ws_extract_env = dict(ws_env, CGA_INPUT='{"extract":true,"lint":null}')
ws_extract_before = ws_inventory()
assert ws_extract_before == ws_before_inventory
_ = (ws_extract_dir / 'workspace-inputs-before.json').write_text(json.dumps(ws_extract_before, indent=2) + '\n')
ws_extract_plan = dict(ws_plan, phase='extraction_enabled', environment=ws_extract_env, extraction_input=json.loads(ws_extract_env['CGA_INPUT']), baseline_directory=str(ws_work), note='Same source/flags/target; CGA_INPUT enables extraction and is tracked by the pinned compiler callback.')
_ = (ws_extract_dir / 'compile-plan.json').write_text(json.dumps(ws_extract_plan, indent=2) + '\n')
ws_extract_started = time.monotonic()
with (ws_extract_dir / 'compiler.stdout').open('wb') as stdout, (ws_extract_dir / 'compiler.stderr').open('wb') as stderr:
    ws_extract_process = subprocess.Popen(ws_argv, cwd=ws_repo, env=ws_extract_env, stdout=stdout, stderr=stderr)
print(json.dumps({'state': 'running', 'pid': ws_extract_process.pid, 'extraction': ws_extract_env['CGA_INPUT'], 'log_directory': str(ws_extract_dir)}, indent=2))

{
  "state": "running",
  "pid": 90279,
  "extraction": "{\"extract\":true,\"lint\":null}",
  "log_directory": "/private/tmp/panic-workspace-20261101/extract-d1_o1"
}


In [85]:
ws_extract_exit = ws_extract_process.poll()
ws_extract_log = (ws_extract_dir / 'compiler.stderr').read_text(errors='replace')
ws_extract_markers = [line.removeprefix('cga-emit:') for line in ws_extract_log.splitlines() if line.startswith('cga-emit:')]
ws_extract_status = {'state': 'running' if ws_extract_exit is None else 'finished', 'exit': ws_extract_exit, 'elapsed_seconds_at_poll': round(time.monotonic() - ws_extract_started, 2), 'emitted_fragments': len(ws_extract_markers), 'recent_log': [line[:240] for line in ws_extract_log.splitlines()[-7:]]}
print(json.dumps(ws_extract_status, indent=2))
if ws_extract_exit is not None:
    ws_extract_after = ws_inventory()
    _ = (ws_extract_dir / 'workspace-inputs-after.json').write_text(json.dumps(ws_extract_after, indent=2) + '\n')
    ws_extract_result = dict(ws_extract_status, source_inputs_unchanged=ws_extract_after == ws_extract_before, graph_pointer_unchanged=str((ws_repo / '.spur/graph/CURRENT').resolve()) == ws_plan['graph_target'], graph_manifest_unchanged=ws_digest((Path(ws_plan['graph_target']) / 'manifest.json').read_bytes()) == ws_plan['graph_manifest_sha256'], stdout_sha256=ws_digest((ws_extract_dir / 'compiler.stdout').read_bytes()), stderr_sha256=ws_digest((ws_extract_dir / 'compiler.stderr').read_bytes()))
    _ = (ws_extract_dir / 'compile-result.json').write_text(json.dumps(ws_extract_result, indent=2) + '\n')
    print(json.dumps(ws_extract_result, indent=2))

{
  "state": "finished",
  "exit": 0,
  "elapsed_seconds_at_poll": 258.04,
  "emitted_fragments": 400,
  "recent_log": [
    "    |            ^^^^^^^",
    "    |",
    "    = note: `#[warn(dead_code)]` (part of `#[warn(unused)]`) on by default",
    "",
    "cga-emit:/private/tmp/panic-workspace-20261101/target/aarch64-apple-darwin/debug/build/spur-graph/b096b618732b4a04/out/spur_graph-b096b618732b4a04.callgraph.json",
    "warning: `spur-graph` (lib) generated 2 warnings (run `cargo fix --lib -p spur-graph` to apply 1 suggestion)",
    "    Finished `dev` profile [unoptimized + debuginfo] target(s) in 3m 55s"
  ]
}


{
  "state": "finished",
  "exit": 0,
  "elapsed_seconds_at_poll": 258.04,
  "emitted_fragments": 400,
  "recent_log": [
    "    |            ^^^^^^^",
    "    |",
    "    = note: `#[warn(dead_code)]` (part of `#[warn(unused)]`) on by default",
    "",
    "cga-emit:/private/tmp/panic-workspace-20261101/target/aarch64-apple-darwin/debug/build/spur-graph/b096b618732b4a04/out/spur_graph-b096b618732b4a04.callgraph.json",
    "warning: `spur-graph` (lib) generated 2 warnings (run `cargo fix --lib -p spur-graph` to apply 1 suggestion)",
    "    Finished `dev` profile [unoptimized + debuginfo] target(s) in 3m 55s"
  ],
  "source_inputs_unchanged": true,
  "graph_pointer_unchanged": true,
  "graph_manifest_unchanged": true,
  "stdout_sha256": "9497abadd86fbaa01e0b501a30c79abb64ee494953026c9974cad4d05bccd3d1",
  "stderr_sha256": "9fe4adb6c6e0e46cbd2d41bc32537640bd6ddec09d18c8e02e11bb5d8bfa99db"
}


In [84]:
# Observe only processes descended from this notebook's own compile command.
ws_child_info = []
ws_process_queue = [str(ws_extract_process.pid)]
while ws_process_queue:
    pid = ws_process_queue.pop(0)
    info = subprocess.run(['ps', '-p', pid, '-o', 'pid=,ppid=,etime=,time=,%cpu=,rss=,comm='], capture_output=True, text=True)
    if info.returncode == 0 and info.stdout.strip():
        ws_child_info.append(info.stdout.strip())
        children = subprocess.run(['pgrep', '-P', pid], capture_output=True, text=True)
        if children.returncode == 0:
            ws_process_queue.extend(children.stdout.split())
print(json.dumps({'compile_exit': ws_extract_process.poll(), 'owned_processes': ws_child_info, 'free_gib': round(shutil.disk_usage(ws_work).free / 1024**3, 2)}, indent=2))

{
  "compile_exit": null,
  "owned_processes": [
    "90279 61328 02:51   0:03.54   0.0  26080 /Users/kevintruong/.rustup/toolchains/nightly-2026-10-18-aarch64-apple-darwin/bin/cargo",
    "94061 90279 00:51   0:50.61  98.0 351968 /private/tmp/spur-cga-pilot-20261031/target/release/cga-driver"
  ],
  "free_gib": 47.59
}


In [86]:
import gzip, re

assert ws_extract_process.poll() == 0, 'compiler extraction must finish successfully before accepting exports'
assert ws_extract_result['source_inputs_unchanged'] and ws_extract_result['graph_pointer_unchanged'] and ws_extract_result['graph_manifest_unchanged']
ws_emit_paths = [Path(line.removeprefix('cga-emit:')).resolve() for line in (ws_extract_dir / 'compiler.stderr').read_text().splitlines() if line.startswith('cga-emit:')]
assert len(ws_emit_paths) == len(set(ws_emit_paths))
ws_export_index = []
for path in ws_emit_paths:
    assert path.is_relative_to(ws_work / 'target') and path.is_file()
    raw = path.read_bytes()
    ws_export_index.append({'path': str(path), 'byte_length': len(raw), 'sha256': ws_digest(raw)})
ws_selected_exports = [r for r in ws_export_index if Path(r['path']).name.startswith('spur_graph-')]
assert len(ws_selected_exports) == 1, ws_selected_exports
ws_selected = ws_selected_exports[0]
ws_raw_path = Path(ws_selected['path'])
ws_graph = json.loads(ws_raw_path.read_bytes())
ws_graph_edge_count = sum(len(v) for v in ws_graph['edges'].values())
ws_kind_counts = {}
for edge_list in ws_graph['edges'].values():
    for edge in edge_list:
        kind = edge['kind'] if isinstance(edge['kind'], str) else next(iter(edge['kind']))
        ws_kind_counts[kind] = ws_kind_counts.get(kind, 0) + 1
_ = (ws_extract_dir / 'exports.json').write_text(json.dumps(ws_export_index, indent=2) + '\n')
_ = (ws_extract_dir / 'selected-export.json').write_text(json.dumps({'raw_export': ws_selected, 'node_count': len(ws_graph['nodes']), 'root_count': len(ws_graph['roots']), 'edge_count': ws_graph_edge_count, 'kinds': ws_kind_counts}, indent=2) + '\n')
with ws_raw_path.open('rb') as raw, (ws_extract_dir / 'spur-graph.callgraph.json.gz').open('wb') as saved:
    with gzip.GzipFile(filename='', mode='wb', fileobj=saved, mtime=0) as compressed:
        shutil.copyfileobj(raw, compressed)
print(json.dumps({'export_count': len(ws_export_index), 'selected_export': ws_selected, 'nodes': len(ws_graph['nodes']), 'roots': len(ws_graph['roots']), 'edges': ws_graph_edge_count, 'edge_kinds': ws_kind_counts, 'first_node': ws_graph['nodes'][0], 'compressed_bytes': (ws_extract_dir / 'spur-graph.callgraph.json.gz').stat().st_size}, indent=2))

{
  "export_count": 400,
  "selected_export": {
    "path": "/private/tmp/panic-workspace-20261101/target/aarch64-apple-darwin/debug/build/spur-graph/b096b618732b4a04/out/spur_graph-b096b618732b4a04.callgraph.json",
    "byte_length": 156752107,
    "sha256": "1cccabaf49b6016c959c246b1e0a571a15aa7657d2332a92eeffd464d50620c2"
  },
  "nodes": 79008,
  "roots": 13,
  "edges": 250319,
  "edge_kinds": {
    "Called": 202261,
    "AsyncPanic": 59,
    "FnPtrCast": 584,
    "VTableCalled": 32611,
    "BuiltinAssert": 14804
  },
  "first_node": {
    "name": "spur_graph::mcp::overlaid_graph_artifact_from_base_seed_for_worktree::{closure#0}",
    "location": {
      "path": "crates/spur-graph/src/mcp/mod.rs",
      "span": {
        "start": {
          "line": 5393,
          "column": 55
        },
        "end": {
          "line": 5424,
          "column": 2
        }
      }
    }
  },
  "compressed_bytes": 4437776
}


In [87]:
ws_nodes = ws_graph['nodes']
assert all(isinstance(i, int) and 0 <= i < len(ws_nodes) for i in ws_graph['roots'])
for source, edges in ws_graph['edges'].items():
    assert str(int(source)) == source and 0 <= int(source) < len(ws_nodes)
    for edge in edges:
        kind = edge['kind']
        if isinstance(kind, dict):
            assert len(kind) == 1 and next(iter(kind)) in ('Called', 'VTableCalled', 'FnPtrCast')
            assert 0 <= next(iter(kind.values())) < len(ws_nodes)
        else:
            assert kind in ('BuiltinAssert', 'AsyncPanic')
print('All raw root and edge references are valid. Root names:', json.dumps([ws_nodes[i]['name'] for i in ws_graph['roots']]))
ws_discovery = []
for case in ws_frontier:
    entity = ws_frontier_symbols[case['caller_id']]['entity_name']
    matches = [(i, n) for i, n in enumerate(ws_nodes) if ('::' + entity) in n['name'] and n['location']['path'] == case['file_path']]
    records = []
    for i, node in matches:
        calls = []
        for edge_index, edge in enumerate(ws_graph['edges'].get(str(i), [])):
            kind = edge['kind']
            if isinstance(kind, dict) and next(iter(kind)) in ('Called', 'VTableCalled'):
                callee = ws_nodes[next(iter(kind.values()))]['name']
                if '::' + case['method'] in callee:
                    calls.append({'edge_index': edge_index, 'callee': callee, 'kind': next(iter(kind)), 'location': edge['location']})
        records.append({'node_id': i, 'compiler_caller': node['name'], 'definition_location': node['location'], 'matching_calls': calls})
    ws_discovery.append({'legacy_caller': case['caller'], 'method_label': case['method'], 'instances': records})
_ = (ws_extract_dir / 'frontier-discovery.json').write_text(json.dumps(ws_discovery, indent=2) + '\n')
print(json.dumps(ws_discovery, indent=2))

All raw root and edge references are valid. Root names: ["spur_graph::mcp::overlaid_graph_artifact_from_base_seed_for_worktree::{closure#0}", "spur_graph::mcp::code_callees::{closure#0}", "spur_graph::mcp::code_search::{closure#0}", "spur_graph::mcp::code_symbol_history::{closure#0}", "spur_graph::mcp::code_callers::{closure#0}", "spur_graph::mcp::code_resolve::{closure#0}", "spur_graph::mcp::code_symbol_info_rebuild_aware::{closure#0}", "spur_graph::mcp::code_symbol_info::{closure#0}", "spur_graph::mcp::code_read_symbol::{closure#0}", "spur_graph::mcp::code_file_symbols::{closure#0}", "spur_graph::mcp::code_subgraph::{closure#0}", "spur_graph::mcp::markdown_overlay_base_delta_for_worktree::{closure#0}", "spur_graph::memory_eval::qa::evaluate_longmem::{closure#0}"]
[
  {
    "legacy_caller": "run_full_walk_into_with_stats",
    "method_label": "expect",
    "instances": []
  },
  {
    "legacy_caller": "write_edges",
    "method_label": "expect",
    "instances": []
  },
  {
    "legac

In [88]:
ws_frontier_name_probe = []
for case in ws_frontier:
    entity = ws_frontier_symbols[case['caller_id']]['entity_name']
    hits = [(i, n) for i, n in enumerate(ws_nodes) if entity in n['name']]
    ws_frontier_name_probe.append({'caller': case['caller'], 'raw_name_hits_before_path_filter': len(hits), 'sample': [{'node_id': i, 'name': n['name'], 'location': n['location']} for i, n in hits[:3]]})
print(json.dumps(ws_frontier_name_probe, indent=2))
_ = (ws_extract_dir / 'frontier-name-probe.json').write_text(json.dumps(ws_frontier_name_probe, indent=2) + '\n')

[
  {
    "caller": "run_full_walk_into_with_stats",
    "raw_name_hits_before_path_filter": 0,
    "sample": []
  },
  {
    "caller": "write_edges",
    "raw_name_hits_before_path_filter": 0,
    "sample": []
  },
  {
    "caller": "rebuild_from_buckets",
    "raw_name_hits_before_path_filter": 0,
    "sample": []
  },
  {
    "caller": "stable_file_id_from_path",
    "raw_name_hits_before_path_filter": 0,
    "sample": []
  },
  {
    "caller": "impl ExternalImportRegistry::write_to_buckets",
    "raw_name_hits_before_path_filter": 0,
    "sample": []
  },
  {
    "caller": "compute_anchor_hash",
    "raw_name_hits_before_path_filter": 0,
    "sample": []
  },
  {
    "caller": "resolve_bare_pending_edge",
    "raw_name_hits_before_path_filter": 0,
    "sample": []
  }
]


### Root selection is a measured coverage boundary

The selected library export has **13 roots**, all public async coroutine bodies. None of the seven legacy frontier function names is present, even before filtering by source path. The pinned driver uses `NoConstOrStatic`: normal Rust functions are not automatic roots; exported/unmangled functions, public async bodies and binary entry roots receive special handling. The 250,319-edge library graph therefore does **not** establish coverage of those seven cases.

The next compile selects the actual **`spur-cli` binary `spur`**, with default features disabled and the same explicit compiler flags. This supplies its real `main` entry point and the graph-build command dispatch. The experiment will check emitted coverage rather than treating a larger graph as sufficient. Shared dependency artifacts are reused; each selected raw export is retained before further compiler invocations.

In [89]:
ws_cli_dir = ws_work / 'cli-d1_o1'
ws_cli_dir.mkdir(exist_ok=False)
ws_cli_paths = sorted(p for p in ws_tracked if p and not p.endswith('.ipynb') and (ws_repo / p).is_file())
def ws_cli_inventory():
    return [{'path': p, 'byte_length': (ws_repo / p).stat().st_size, 'sha256': ws_digest((ws_repo / p).read_bytes())} for p in ws_cli_paths]
ws_cli_before = ws_cli_inventory()
_ = (ws_cli_dir / 'workspace-inputs-before.json').write_text(json.dumps(ws_cli_before, indent=2) + '\n')
ws_cli_argv = [str(ws_repo / 'scripts/spur-cargo'), 'build', '--frozen', '--offline', '-p', 'spur-cli', '--bin', 'spur', '--no-default-features', '--target', 'aarch64-apple-darwin', '-vv']
ws_cli_plan = dict(ws_extract_plan, phase='cli_binary_extraction', argv=ws_cli_argv, packages=['spur-cli'], targets=['bin:spur'], declared_input_count=len(ws_cli_before), inventory_scope='all tracked workspace files except observational notebooks; cached registry/git dependency sources are selected by Cargo.lock, not claimed fully closed', baseline_library_export=ws_selected, environment=ws_extract_env)
_ = (ws_cli_dir / 'compile-plan.json').write_text(json.dumps(ws_cli_plan, indent=2) + '\n')
ws_cli_started = time.monotonic()
with (ws_cli_dir / 'compiler.stdout').open('wb') as stdout, (ws_cli_dir / 'compiler.stderr').open('wb') as stderr:
    ws_cli_process = subprocess.Popen(ws_cli_argv, cwd=ws_repo, env=ws_extract_env, stdout=stdout, stderr=stderr)
print(json.dumps({'state': 'running', 'pid': ws_cli_process.pid, 'target': 'spur-cli bin spur, default features disabled', 'declared_inputs': len(ws_cli_before), 'capture_directory': str(ws_cli_dir)}, indent=2))

{
  "state": "running",
  "pid": 99213,
  "target": "spur-cli bin spur, default features disabled",
  "declared_inputs": 3507,
  "capture_directory": "/private/tmp/panic-workspace-20261101/cli-d1_o1"
}


In [112]:
ws_cli_exit = ws_cli_process.poll()
ws_cli_log = (ws_cli_dir / 'compiler.stderr').read_text(errors='replace')
ws_cli_markers = [line.removeprefix('cga-emit:') for line in ws_cli_log.splitlines() if line.startswith('cga-emit:')]
ws_cli_status = {'state': 'running' if ws_cli_exit is None else 'finished', 'exit': ws_cli_exit, 'elapsed_seconds_at_poll': round(time.monotonic() - ws_cli_started, 2), 'emitted_markers': len(ws_cli_markers), 'recent_log': [line[:260] for line in ws_cli_log.splitlines()[-8:]], 'free_gib': round(shutil.disk_usage(ws_work).free / 1024**3, 2)}
print(json.dumps(ws_cli_status, indent=2))
if ws_cli_exit is not None:
    ws_cli_after = ws_cli_inventory()
    _ = (ws_cli_dir / 'workspace-inputs-after.json').write_text(json.dumps(ws_cli_after, indent=2) + '\n')
    ws_cli_result = dict(ws_cli_status, source_inputs_unchanged=ws_cli_after == ws_cli_before, graph_pointer_unchanged=str((ws_repo / '.spur/graph/CURRENT').resolve()) == ws_plan['graph_target'], graph_manifest_unchanged=ws_digest((Path(ws_plan['graph_target']) / 'manifest.json').read_bytes()) == ws_plan['graph_manifest_sha256'], stdout_sha256=ws_digest((ws_cli_dir / 'compiler.stdout').read_bytes()), stderr_sha256=ws_digest((ws_cli_dir / 'compiler.stderr').read_bytes()))
    _ = (ws_cli_dir / 'compile-result.json').write_text(json.dumps(ws_cli_result, indent=2) + '\n')
    print(json.dumps(ws_cli_result, indent=2))

{
  "state": "finished",
  "exit": 101,
  "elapsed_seconds_at_poll": 1916.63,
  "emitted_markers": 847,
  "recent_log": [
    "",
    "thread 'rustc' (9024487) panicked at /rustc-dev/8006091eb71ae7d9e9bccb66c6122c9d68af4466/compiler/rustc_middle/src/ty/instance.rs:640:21:",
    "Box<dyn Any>",
    "note: run with `RUST_BACKTRACE=1` environment variable to display a backtrace",
    "error: could not compile `spur-cli` (lib)",
    "",
    "Caused by:",
    "  process didn't exit successfully: `CARGO=/Users/kevintruong/.rustup/toolchains/nightly-2026-10-18-aarch64-apple-darwin/bin/cargo CARGO_CRATE_NAME=spur_cli CARGO_MANIFEST_DIR=/Users/kevintruong/project/SpurCode/crates/spur-cli CARGO_MANIFEST_PATH=/Users/kevin"
  ],
  "free_gib": 37.91
}


{
  "state": "finished",
  "exit": 101,
  "elapsed_seconds_at_poll": 1916.63,
  "emitted_markers": 847,
  "recent_log": [
    "",
    "thread 'rustc' (9024487) panicked at /rustc-dev/8006091eb71ae7d9e9bccb66c6122c9d68af4466/compiler/rustc_middle/src/ty/instance.rs:640:21:",
    "Box<dyn Any>",
    "note: run with `RUST_BACKTRACE=1` environment variable to display a backtrace",
    "error: could not compile `spur-cli` (lib)",
    "",
    "Caused by:",
    "  process didn't exit successfully: `CARGO=/Users/kevintruong/.rustup/toolchains/nightly-2026-10-18-aarch64-apple-darwin/bin/cargo CARGO_CRATE_NAME=spur_cli CARGO_MANIFEST_DIR=/Users/kevintruong/project/SpurCode/crates/spur-cli CARGO_MANIFEST_PATH=/Users/kevin"
  ],
  "free_gib": 37.91,
  "source_inputs_unchanged": true,
  "graph_pointer_unchanged": true,
  "graph_manifest_unchanged": true,
  "stdout_sha256": "f2f5e01c3657e1a3d2661346793f2d1e48502fa6d87d75411b3ae0c028ee06b2",
  "stderr_sha256": "f8af3dfd79ca4eb98d19f9085f3b6449751c862

In [91]:
# Flatten the immutable library export for SQL inspection; retain every edge kind.
ws_flat_dir = ws_extract_dir / 'flat'
ws_flat_dir.mkdir(exist_ok=True)
with (ws_flat_dir / 'nodes.jsonl').open('w') as nodes_out:
    for node_id, node in enumerate(ws_nodes):
        nodes_out.write(json.dumps({'node_id': node_id, 'canonical_name': node['name'], 'file_path': node['location']['path'], 'is_root': node_id in ws_graph['roots']}) + '\n')
with (ws_flat_dir / 'edges.jsonl').open('w') as edges_out:
    for source, edges in ws_graph['edges'].items():
        for edge_index, edge in enumerate(edges):
            kind = edge['kind']
            target = None if isinstance(kind, str) else next(iter(kind.values()))
            kind_name = kind if isinstance(kind, str) else next(iter(kind))
            edges_out.write(json.dumps({'source_node': int(source), 'edge_index': edge_index, 'target_node': target, 'edge_kind': kind_name, 'file_path': edge['location']['path'], 'start_line': edge['location']['span']['start']['line'], 'start_column': edge['location']['span']['start']['column'], 'end_line': edge['location']['span']['end']['line'], 'end_column': edge['location']['span']['end']['column']}) + '\n')
print(json.dumps({'nodes_jsonl_bytes': (ws_flat_dir / 'nodes.jsonl').stat().st_size, 'edges_jsonl_bytes': (ws_flat_dir / 'edges.jsonl').stat().st_size, 'classification': 'Called and VTableCalled are kept distinct; FnPtrCast remains address flow and will not be treated as a call.'}))

{"nodes_jsonl_bytes": 27391663, "edges_jsonl_bytes": 72353056, "classification": "Called and VTableCalled are kept distinct; FnPtrCast remains address flow and will not be treated as a call."}


In [92]:
-- Discover the selected managed connection before introducing compiler tables.
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'main' AND table_name IN ('ws_cga_nodes', 'ws_cga_edges')
ORDER BY table_name, ordinal_position;


table_name,column_name,data_type


In [93]:
DESCRIBE SELECT * FROM read_ndjson_auto('/private/tmp/panic-workspace-20261101/extract-d1_o1/flat/edges.jsonl');

column_name,column_type,null,key,default,extra
source_node,BIGINT,YES,NaN,NaN,NaN
edge_index,BIGINT,YES,NaN,NaN,NaN
target_node,BIGINT,YES,NaN,NaN,NaN
edge_kind,VARCHAR,YES,NaN,NaN,NaN
file_path,VARCHAR,YES,NaN,NaN,NaN
start_line,BIGINT,YES,NaN,NaN,NaN
start_column,BIGINT,YES,NaN,NaN,NaN
end_line,BIGINT,YES,NaN,NaN,NaN
end_column,BIGINT,YES,NaN,NaN,NaN


In [95]:
DESCRIBE SELECT * FROM read_ndjson_auto('/private/tmp/panic-workspace-20261101/extract-d1_o1/flat/nodes.jsonl');

column_name,column_type,null,key,default,extra
node_id,BIGINT,YES,NaN,NaN,NaN
canonical_name,VARCHAR,YES,NaN,NaN,NaN
file_path,VARCHAR,YES,NaN,NaN,NaN
is_root,BOOLEAN,YES,NaN,NaN,NaN


In [96]:
CREATE OR REPLACE TEMP TABLE ws_cga_nodes AS
SELECT * FROM read_ndjson_auto('/private/tmp/panic-workspace-20261101/extract-d1_o1/flat/nodes.jsonl');
CREATE OR REPLACE TEMP TABLE ws_cga_edges AS
SELECT * FROM read_ndjson_auto('/private/tmp/panic-workspace-20261101/extract-d1_o1/flat/edges.jsonl');
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_name IN ('ws_cga_nodes', 'ws_cga_edges')
ORDER BY table_name, ordinal_position;

table_name,column_name,data_type
ws_cga_edges,source_node,BIGINT
ws_cga_edges,edge_index,BIGINT
ws_cga_edges,target_node,BIGINT
ws_cga_edges,edge_kind,VARCHAR
ws_cga_edges,file_path,VARCHAR
ws_cga_edges,start_line,BIGINT
ws_cga_edges,start_column,BIGINT
ws_cga_edges,end_line,BIGINT
ws_cga_edges,end_column,BIGINT
ws_cga_nodes,node_id,BIGINT


In [98]:
-- Typed compiler edge kinds remain separate from syntax graph relations.
COPY ws_cga_nodes TO '/private/tmp/panic-workspace-20261101/extract-d1_o1/flat/nodes.parquet' (FORMAT PARQUET);
COPY ws_cga_edges TO '/private/tmp/panic-workspace-20261101/extract-d1_o1/flat/edges.parquet' (FORMAT PARQUET);
SELECT edge_kind, count(*) AS observations,
       count(*) FILTER (WHERE file_path LIKE 'crates/spur-graph/%') AS graph_crate_locations
FROM ws_cga_edges
GROUP BY edge_kind ORDER BY observations DESC;

edge_kind,observations,graph_crate_locations
Called,202261,8502
VTableCalled,32611,23
BuiltinAssert,14804,77
FnPtrCast,584,1
AsyncPanic,59,54


In [99]:
SELECT node_id, canonical_name, file_path
FROM ws_cga_nodes
WHERE canonical_name IN ('core::panicking::panic_fmt', 'std::panicking::panic_fmt')
   OR canonical_name = 'spur_graph::mcp::code_read_symbol::{closure#0}'
ORDER BY canonical_name, node_id;

node_id,canonical_name,file_path
55,core::panicking::panic_fmt,/Users/kevintruong/.rustup/toolchains/nightly-2026-10-18-aarch64-apple-darwin/lib/rustlib/src/rust/library/core/src/panicking.rs
8,spur_graph::mcp::code_read_symbol::{closure#0},crates/spur-graph/src/mcp/mod.rs


In [100]:
-- The library node IDs below are bound to raw export SHA256 1cccabaf...20c2.
-- Read Parquet in the edge query because duckrouting uses another connection.
DESCRIBE SELECT * FROM duckrouting_dijkstra(
 $$SELECT row_number() OVER (ORDER BY source_node, edge_index)::BIGINT AS id,
          source_node AS source, target_node AS target,
          1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
   FROM read_parquet('/private/tmp/panic-workspace-20261101/extract-d1_o1/flat/edges.parquet')
   WHERE edge_kind = 'Called' AND target_node IS NOT NULL$$,
 8::BIGINT, 55::BIGINT
);

column_name,column_type,null,key,default,extra
seq,BIGINT,YES,NaN,NaN,NaN
path_seq,BIGINT,YES,NaN,NaN,NaN
start_vid,BIGINT,YES,NaN,NaN,NaN
end_vid,BIGINT,YES,NaN,NaN,NaN
node,BIGINT,YES,NaN,NaN,NaN
edge,BIGINT,YES,NaN,NaN,NaN
cost,DOUBLE,YES,NaN,NaN,NaN
agg_cost,DOUBLE,YES,NaN,NaN,NaN


In [102]:
CREATE OR REPLACE TEMP TABLE ws_library_panic_path AS
SELECT * FROM duckrouting_dijkstra(
 $$SELECT row_number() OVER (ORDER BY source_node, edge_index)::BIGINT AS id,
          source_node AS source, target_node AS target,
          1.0::DOUBLE AS cost, -1.0::DOUBLE AS reverse_cost
   FROM read_parquet('/private/tmp/panic-workspace-20261101/extract-d1_o1/flat/edges.parquet')
   WHERE edge_kind = 'Called' AND target_node IS NOT NULL$$,
 8::BIGINT, 55::BIGINT
);
COPY ws_library_panic_path TO '/private/tmp/panic-workspace-20261101/extract-d1_o1/flat/panic-path.parquet' (FORMAT PARQUET);
SELECT p.seq, p.node, p.agg_cost AS hops, n.canonical_name, n.file_path
FROM ws_library_panic_path p JOIN ws_cga_nodes n ON n.node_id = p.node
ORDER BY p.seq;

seq,node,hops,canonical_name,file_path
1,8,0.0,spur_graph::mcp::code_read_symbol::{closure#0},crates/spur-graph/src/mcp/mod.rs
2,77808,1.0,"spur_graph::mcp::code_graph_backend_value_allowing_source::<for<'a, 'b> fn(&'a serde_json::Value, &'b (dyn spur_graph::query_client::GraphQueryClient + 'b)) -> std::result::Result<serde_json::Value, spur_graph::mcp::CodeGraphError> {spur_graph::mcp::code_read_symbol_with_client}>::{closure#0}",crates/spur-graph/src/mcp/mod.rs
3,77814,2.0,"spur_graph::mcp::code_graph_backend_value_with_format::<for<'a, 'b> fn(&'a serde_json::Value, &'b (dyn spur_graph::query_client::GraphQueryClient + 'b)) -> std::result::Result<serde_json::Value, spur_graph::mcp::CodeGraphError> {spur_graph::mcp::code_read_symbol_with_client}>::{closure#0}",crates/spur-graph/src/mcp/mod.rs
4,72515,3.0,spur_graph::mcp::open_code_search_backend_for_request::{closure#0},crates/spur-graph/src/mcp/mod.rs
5,57493,4.0,spur_graph::store::pointer::resolve_artifact_location,crates/spur-graph/src/store/pointer.rs
6,380,5.0,tracing::level_filters::LevelFilter::current,/Users/kevintruong/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/tracing-core-0.1.36/src/metadata.rs
7,55,6.0,core::panicking::panic_fmt,/Users/kevintruong/.rustup/toolchains/nightly-2026-10-18-aarch64-apple-darwin/lib/rustlib/src/rust/library/core/src/panicking.rs


In [104]:
ws_last_edges = [dict(edge_index=i, **e) for i, e in enumerate(ws_graph['edges'].get('380', [])) if e['kind'] == {'Called': 55}]
assert len(ws_last_edges) == 1
ws_path_node_ids = [8, 77808, 77814, 72515, 57493, 380, 55]
ws_path_verified_edges = []
for src, dst in zip(ws_path_node_ids, ws_path_node_ids[1:]):
    matches = [(i, e) for i, e in enumerate(ws_graph['edges'].get(str(src), [])) if e['kind'] == {'Called': dst}]
    assert matches, (src, dst)
    ws_path_verified_edges.append({'from_node': src, 'to_node': dst, 'compiler_caller': ws_nodes[src]['name'], 'compiler_callee': ws_nodes[dst]['name'], 'observed_edges': [{'edge_index': i, **edge} for i, edge in matches]})
_ = (ws_extract_dir / 'shortest-path-verified.json').write_text(json.dumps({'raw_export_sha256': ws_selected['sha256'], 'node_ids': ws_path_node_ids, 'hops': 6, 'edge_policy': 'Called only; FnPtrCast and VTableCalled excluded', 'edges': ws_path_verified_edges, 'interpretation': 'Compiler path candidate; branch/input feasibility not established'}, indent=2) + '\n')
print(json.dumps({'verified_hops': len(ws_path_verified_edges), 'last_compiler_edge': ws_last_edges[0], 'interpretation': 'A compiler call path is not a demonstrated runtime panic.'}, indent=2))

{
  "verified_hops": 6,
  "last_compiler_edge": {
    "edge_index": 3,
    "kind": {
      "Called": 55
    },
    "location": {
      "path": "/Users/kevintruong/.rustup/toolchains/nightly-2026-10-18-aarch64-apple-darwin/lib/rustlib/src/rust/library/core/src/panic.rs",
      "span": {
        "start": {
          "line": 61,
          "column": 9
        },
        "end": {
          "line": 61,
          "column": 73
        }
      }
    }
  },
  "interpretation": "A compiler call path is not a demonstrated runtime panic."
}


In [105]:
ws_cli_owned_processes = []
ws_queue = [str(ws_cli_process.pid)]
while ws_queue:
    pid = ws_queue.pop(0)
    info = subprocess.run(['ps', '-p', pid, '-o', 'pid=,ppid=,etime=,time=,%cpu=,rss=,comm='], capture_output=True, text=True)
    if info.returncode == 0 and info.stdout.strip():
        ws_cli_owned_processes.append(info.stdout.strip())
        children = subprocess.run(['pgrep', '-P', pid], capture_output=True, text=True)
        if children.returncode == 0:
            ws_queue.extend(children.stdout.split())
print(json.dumps({'compile_exit': ws_cli_process.poll(), 'owned_processes': ws_cli_owned_processes, 'free_gib': round(shutil.disk_usage(ws_work).free / 1024**3, 2)}, indent=2))

{
  "compile_exit": null,
  "owned_processes": [
    "99213 61328 12:18   0:07.24   0.0  42624 /Users/kevintruong/.rustup/toolchains/nightly-2026-10-18-aarch64-apple-darwin/bin/cargo",
    "9080 99213 03:26   3:23.74  97.6 942624 /private/tmp/spur-cga-pilot-20261031/target/release/cga-driver"
  ],
  "free_gib": 40.74
}


In [108]:
ws_guard = json.loads((ws_work / 'tracing-guard-solves.json').read_text())
ws_guard_source = json.loads((ws_work / 'tracing-source-contract.json').read_text())
def ws_unwrap_result(result):
    return result.get('structuredContent') or json.loads(next(r['text'] for r in result['content'] if r['type'] == 'text'))
ws_guard_rows = []
for record in ws_guard['records']:
    solved = ws_unwrap_result(record['result'])
    reloaded = ws_unwrap_result(record['reloaded'])
    assert solved['solve_id'] == reloaded['solve_id'] and solved['status'] == reloaded['status']
    ws_guard_rows.append({'case': record['name'], 'status': solved['status'], 'solve_id': solved['solve_id'], 'model': solved.get('model')})
assert [r['status'] for r in ws_guard_rows] == ['sat', 'unsat', 'sat']
assert ws_guard_source['source_matches_registry_checksum']
print(json.dumps({'source_contract': ws_guard_source['source_sha256'], 'horizon': 1, 'results': ws_guard_rows, 'scope': ws_guard['scope']}, indent=2))

{
  "source_contract": "87b7dc2f5e1da4ae4146da2c641e71972b5412c243918ca484027bc06922a7e5",
  "horizon": 1,
  "results": [
    {
      "case": "valid_witness",
      "status": "sat",
      "solve_id": "sol_c977ef0429054438",
      "model": {
        "workflow_u_0": "off",
        "workflow_u_1": "trace"
      }
    },
    {
      "case": "invalid_under_typed_setter",
      "status": "unsat",
      "solve_id": "sol_f84468625d604da4",
      "model": null
    },
    {
      "case": "hypothetical_invalid_write",
      "status": "sat",
      "solve_id": "sol_681709bbd37a4ca9",
      "model": {
        "workflow_u_0": "off",
        "workflow_u_1": "invalid"
      }
    }
  ],
  "scope": "finite one-step abstraction of tracing-core0.1.36 typed setter; not Rust execution or unbounded proof"
}


In [111]:
import shlex
ws_active_crates = []
if ws_cli_process.poll() is None:
    children = subprocess.run(['pgrep', '-P', str(ws_cli_process.pid)], capture_output=True, text=True)
    for pid in children.stdout.split():
        command = subprocess.run(['ps', '-p', pid, '-o', 'command='], capture_output=True, text=True)
        args = shlex.split(command.stdout)
        if '--crate-name' in args:
            ws_active_crates.append({'pid': int(pid), 'crate': args[args.index('--crate-name') + 1]})
print(json.dumps({'cli_exit': ws_cli_process.poll(), 'active_compiler_crates': ws_active_crates, 'elapsed_seconds': round(time.monotonic() - ws_cli_started, 1)}, indent=2))

{
  "cli_exit": null,
  "active_compiler_crates": [
    {
      "pid": 16619,
      "crate": "spur_tui"
    }
  ],
  "elapsed_seconds": 1439.0
}


In [113]:
# The full CLI capture failed inside the pinned rustc/CGA producer (exit 101).
# Try the existing smaller executable without changing production source or flags.
assert ws_cli_process.poll() == 101
ws_example_dir = ws_work / 'example-d1_o1'
ws_example_dir.mkdir(exist_ok=False)
ws_example_before = ws_cli_inventory()
_ = (ws_example_dir / 'workspace-inputs-before.json').write_text(json.dumps(ws_example_before, indent=2) + '\n')
ws_example_argv = [str(ws_repo / 'scripts/spur-cargo'), 'build', '--frozen', '--offline', '-p', 'spur-graph', '--example', 'panic_evidence', '--no-default-features', '--target', 'aarch64-apple-darwin', '-vv']
ws_example_plan = dict(ws_cli_plan, phase='existing_example_extraction_after_cli_producer_failure', argv=ws_example_argv, packages=['spur-graph'], targets=['example:panic_evidence'], declared_input_count=len(ws_example_before), previous_cli_exit=101)
_ = (ws_example_dir / 'compile-plan.json').write_text(json.dumps(ws_example_plan, indent=2) + '\n')
ws_example_started = time.monotonic()
with (ws_example_dir / 'compiler.stdout').open('wb') as stdout, (ws_example_dir / 'compiler.stderr').open('wb') as stderr:
    ws_example_process = subprocess.Popen(ws_example_argv, cwd=ws_repo, env=ws_extract_env, stdout=stdout, stderr=stderr)
print(json.dumps({'state':'running','pid':ws_example_process.pid,'target':'existing spur-graph example panic_evidence','declared_inputs':len(ws_example_before)}, indent=2))


{
  "state": "running",
  "pid": 40740,
  "target": "existing spur-graph example panic_evidence",
  "declared_inputs": 3507
}


In [120]:
ws_example_exit = ws_example_process.poll()
ws_example_log = (ws_example_dir / 'compiler.stderr').read_text(errors='replace')
ws_example_markers = [line.removeprefix('cga-emit:') for line in ws_example_log.splitlines() if line.startswith('cga-emit:')]
ws_example_status = {'state':'running' if ws_example_exit is None else 'finished','exit':ws_example_exit,'elapsed_seconds_at_poll':round(time.monotonic()-ws_example_started,2),'emitted_markers':len(ws_example_markers),'unique_emitted_paths':len(set(ws_example_markers)),'diagnostics':[line[:240] for line in ws_example_log.splitlines() if line.startswith(('error:',"thread '",'    Finished','   Compiling'))][-6:]}
if ws_example_exit is not None:
    ws_example_after = ws_cli_inventory()
    _ = (ws_example_dir / 'workspace-inputs-after.json').write_text(json.dumps(ws_example_after, indent=2) + '\n')
    ws_example_status.update(source_inputs_unchanged=ws_example_after == ws_example_before,graph_pointer_unchanged=str((ws_repo/'.spur/graph/CURRENT').resolve()) == ws_plan['graph_target'],graph_manifest_unchanged=ws_digest((Path(ws_plan['graph_target'])/'manifest.json').read_bytes()) == ws_plan['graph_manifest_sha256'],stdout_sha256=ws_digest((ws_example_dir/'compiler.stdout').read_bytes()),stderr_sha256=ws_digest((ws_example_dir/'compiler.stderr').read_bytes()))
    _ = (ws_example_dir / 'compile-result.json').write_text(json.dumps(ws_example_status, indent=2) + '\n')
print(json.dumps(ws_example_status,indent=2))


{
  "state": "finished",
  "exit": 0,
  "elapsed_seconds_at_poll": 244.61,
  "emitted_markers": 427,
  "unique_emitted_paths": 427,
  "diagnostics": [
    "   Compiling parquet v58.1.0",
    "   Compiling spur-telemetry v1.24.0 (/Users/kevintruong/project/SpurCode/crates/spur-telemetry)",
    "   Compiling spur-acp v1.24.0 (/Users/kevintruong/project/SpurCode/crates/spur-acp)",
    "   Compiling spur-mcp v1.24.0 (/Users/kevintruong/project/SpurCode/crates/spur-mcp)",
    "   Compiling spur-graph v1.24.0 (/Users/kevintruong/project/SpurCode/crates/spur-graph)",
    "    Finished `dev` profile [unoptimized + debuginfo] target(s) in 3m 30s"
  ],
  "source_inputs_unchanged": true,
  "graph_pointer_unchanged": true,
  "graph_manifest_unchanged": true,
  "stdout_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
  "stderr_sha256": "85c54e1ab295b311598f94af8f9ff77b5710516a8b3337dd3481d41b94b4cc71"
}


In [119]:
# Independently verify one source match in an emitted dependency fragment.
# This is partial evidence from a FAILED CLI build, not a spur::main graph.
import gzip, hashlib, gc
ws_worker_report = json.loads((ws_cli_dir / 'worker-result.json').read_text())
ws_tui_record = next(r for r in ws_worker_report['retained_partial_exports'] if '/spur-tui/' in r['path'])
ws_tui_raw = gzip.decompress(Path(ws_tui_record['verbatim_gzip']['path']).read_bytes())
assert hashlib.sha256(ws_tui_raw).hexdigest() == ws_tui_record['sha256']
assert len(ws_tui_raw) == ws_tui_record['byte_length']
ws_tui_graph = json.loads(ws_tui_raw)
del ws_tui_raw
ws_binding = next(c for c in json.loads((ws_work/'frontier-source-bindings.json').read_text())['frontier'] if c['caller_id']=='f6b27fe2f9cf6602')
ws_site = ws_binding['sites'][0]
ws_text_bytes = (ws_repo/ws_binding['file_path']).read_bytes()
assert hashlib.sha1(b'blob '+str(len(ws_text_bytes)).encode()+b'\0'+ws_text_bytes).hexdigest() == ws_site['source_oid']
def ws_scalar_offset(position, text):
    lines=text.splitlines(keepends=True)
    line,column=position['line'],position['column']
    assert 1<=line<=len(lines) and 1<=column<=len(lines[line-1])+1
    return sum(len(s.encode()) for s in lines[:line-1])+len(lines[line-1][:column-1].encode())
def ws_span_bytes(location):
    assert location['path']==ws_binding['file_path']
    text=ws_text_bytes.decode()
    return [ws_scalar_offset(location['span'][p],text) for p in ['start','end']]
ws_partial_matches=[]
for caller_key,edges in ws_tui_graph['edges'].items():
    for edge_index,edge in enumerate(edges):
        loc=edge.get('location')
        if not loc or loc['path']!=ws_binding['file_path'] or ws_span_bytes(loc)!=ws_site['byte_range']:
            continue
        kind=edge['kind']
        if not isinstance(kind,dict) or 'Called' not in kind:
            continue
        caller_id=int(caller_key)
        caller=ws_tui_graph['nodes'][caller_id]
        callee=ws_tui_graph['nodes'][kind['Called']]
        assert caller['location']['path']==ws_binding['file_path']
        owner=ws_site['lexical_owners'][0]['byte_range']
        definition=ws_span_bytes(caller['location'])
        assert definition[0]==owner[0] and definition[1]<=owner[1]
        assert ws_text_bytes[slice(*ws_site['byte_range'])].decode()==ws_site['call_text']
        ws_partial_matches.append({'caller_id':caller_id,'edge_index':edge_index,'callee_id':kind['Called'],'caller':caller,'callee':callee,'edge':edge,'call_byte_range':ws_site['byte_range'],'caller_definition_bytes':definition,'lexical_function_bytes':owner})
assert len(ws_partial_matches)==1
ws_partial_proof={'scope':'raw_workspace_partial_dependency_export','cli_cargo_exit':101,'actual_spur_binary_graph_available':False,'raw_export_sha256':ws_tui_record['sha256'],'raw_export_byte_length':ws_tui_record['byte_length'],'source_oid':ws_site['source_oid'],'site_id':ws_site['site_id'],'matches':ws_partial_matches,'interpretation':'One exact Result::expect observation, source span and immediate function header bound. Panic feasibility and validated workspace importer remain unavailable.'}
_=(ws_work/'partial-tui-exact-expect.json').write_text(json.dumps(ws_partial_proof,indent=2)+'\n')
print(json.dumps({'verified_exact_site':ws_site['site_id'],'compiler_target':ws_partial_matches[0]['callee']['name'],'partial_export_only':True,'full_cli_exit':101},indent=2))
del ws_tui_graph
gc.collect()


{
  "verified_exact_site": "f0fe225fb2fda3fa98abd4927348d78c18443cf435ac9672bf8ca2d6360e3851",
  "compiler_target": "std::result::Result::<[u8; 8], std::array::TryFromSliceError>::expect",
  "partial_export_only": true,
  "full_cli_exit": 101
}


1837

In [121]:
from collections import Counter
assert ws_example_exit == 0
ws_example_candidates = [Path(p) for p in dict.fromkeys(ws_example_markers) if Path(p).name.startswith('panic_evidence')]
assert len(ws_example_candidates)==1, [str(p) for p in ws_example_candidates]
ws_example_export_path=ws_example_candidates[0]
ws_example_raw=ws_example_export_path.read_bytes()
ws_example_selected={'path':str(ws_example_export_path),'byte_length':len(ws_example_raw),'sha256':hashlib.sha256(ws_example_raw).hexdigest()}
ws_example_graph=json.loads(ws_example_raw)
ws_example_nodes=ws_example_graph['nodes']
ws_example_roots=[{'node_id':n,**ws_example_nodes[n]} for n in ws_example_graph['roots']]
assert any(n['name']=='panic_evidence::main' for n in ws_example_roots), ws_example_roots
ws_example_kinds=Counter()
for caller,edges in ws_example_graph['edges'].items():
    assert 0<=int(caller)<len(ws_example_nodes)
    for edge in edges:
        kind=edge['kind']
        if isinstance(kind,dict):
            assert len(kind)==1
            name,target=next(iter(kind.items()))
            assert 0<=target<len(ws_example_nodes)
        else:
            name=kind
        ws_example_kinds[name]+=1
assert all(0<=n<len(ws_example_nodes) for n in ws_example_graph['roots'])
with (ws_example_dir/'panic-evidence.callgraph.json.gz').open('wb') as out:
    with gzip.GzipFile(filename='',mode='wb',fileobj=out,mtime=0,compresslevel=6) as zipped:
        zipped.write(ws_example_raw)
ws_example_export_index=[]
for index,p in enumerate(dict.fromkeys(ws_example_markers)):
    path=Path(p)
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):digest.update(chunk)
    ws_example_export_index.append({'emit_index':index,'path':p,'byte_length':path.stat().st_size,'sha256':digest.hexdigest()})
_=(ws_example_dir/'exports.json').write_text(json.dumps(ws_example_export_index,indent=2)+'\n')
ws_example_selected_result={'raw_export':ws_example_selected,'node_count':len(ws_example_nodes),'edge_count':sum(ws_example_kinds.values()),'roots':ws_example_roots,'root_count':len(ws_example_roots),'kinds':dict(ws_example_kinds),'all_references_valid':True,'scope':'existing_workspace_example_binary','typed_workspace_importer_supported':False}
_=(ws_example_dir/'selected-export.json').write_text(json.dumps(ws_example_selected_result,indent=2)+'\n')
del ws_example_raw
print(json.dumps(ws_example_selected_result,indent=2))


{
  "raw_export": {
    "path": "/private/tmp/panic-workspace-20261101/target/aarch64-apple-darwin/debug/build/spur-graph/a35c77b0cc44f073/out/panic_evidence.callgraph.json",
    "byte_length": 155067054,
    "sha256": "fa1f11f94164756f69f53b0011d4fecf17f9978127d363dc263361cc4d4a50d4"
  },
  "node_count": 77468,
  "edge_count": 246520,
  "roots": [
    {
      "node_id": 0,
      "name": "panic_evidence::main",
      "location": {
        "path": "crates/spur-graph/examples/panic_evidence.rs",
        "span": {
          "start": {
            "line": 161,
            "column": 1
          },
          "end": {
            "line": 161,
            "column": 32
          }
        }
      }
    }
  ],
  "root_count": 1,
  "kinds": {
    "Called": 200036,
    "VTableCalled": 31862,
    "BuiltinAssert": 14060,
    "FnPtrCast": 562
  },
  "all_references_valid": true,
  "scope": "existing_workspace_example_binary",
  "typed_workspace_importer_supported": false
}


In [122]:
ws_frontier_bundle=json.loads((ws_work/'frontier-source-bindings.json').read_text())
ws_bound_bytes={row['file_path']:(ws_repo/row['file_path']).read_bytes() for row in ws_frontier_bundle['sources']}
for row in ws_frontier_bundle['sources']:
    assert hashlib.sha256(ws_bound_bytes[row['file_path']]).hexdigest()==row['sha256']
ws_lines={p:b.decode().splitlines(keepends=True) for p,b in ws_bound_bytes.items()}
ws_line_offsets={}
for p,lines in ws_lines.items():
    offsets=[0]
    for line in lines:offsets.append(offsets[-1]+len(line.encode()))
    ws_line_offsets[p]=offsets
def ws_bound_range(location):
    p=location['path']
    out=[]
    for which in ['start','end']:
        line=location['span'][which]['line'];col=location['span'][which]['column']
        assert 1<=line<=len(ws_lines[p]) and 1<=col<=len(ws_lines[p][line-1])+1
        out.append(ws_line_offsets[p][line-1]+len(ws_lines[p][line-1][:col-1].encode()))
    return out
ws_example_cases=[]
for binding in ws_frontier_bundle['frontier']:
    symbol=ws_frontier_bundle['symbols'][binding['caller_id']]
    p=binding['file_path'];lo,hi=symbol['byte_range']
    definitions=[];observations=[]
    for node_id,node in enumerate(ws_example_nodes):
        loc=node['location']
        if loc['path']!=p:continue
        r=ws_bound_range(loc)
        if lo<=r[0]<=r[1]<=hi:
            definitions.append({'node_id':node_id,'raw_node':node,'byte_range':r,'classification':'same_function_start' if r[0]==lo else 'nested_definition_or_unknown_owner'})
    for caller,edges in ws_example_graph['edges'].items():
        for edge_index,edge in enumerate(edges):
            loc=edge['location']
            if loc['path']!=p:continue
            r=ws_bound_range(loc)
            if not lo<=r[0]<=r[1]<=hi:continue
            node=ws_example_nodes[int(caller)]
            node_loc=node['location']
            owner='inlined_or_unknown_owner'
            node_range=ws_bound_range(node_loc) if node_loc['path']==p else None
            if node_range and node_range[0]==lo and node_range[1]<=hi:
                owner='same_function_start'
            for site in binding['sites']:
                lexical=site['lexical_owners'][0]
                if site['lexical_phase']=='closure_body_syntax' and node_range and node_range[0]==lexical['byte_range'][0] and node_range[1]<=lexical['byte_range'][1]:
                    owner='same_closure_start'
            correlations=[]
            for site in binding['sites']:
                if r==site['byte_range']:
                    correlations.append({'relation':'exact_source_call_span','site_id':site['site_id'],'lexical_phase':site['lexical_phase'],'outer_caller_id':site['caller_stable_symbol_id']})
            for macro in binding['macro_evidence']:
                ml,mh=macro['macro_site']['byte_range']
                if ml<=r[0]<=r[1]<=mh:correlations.append({'relation':'within_recorded_macro_invocation_span','macro_site_id':macro['macro_site']['site_id'],'expansion_mapping_proven':False})
            kind=edge['kind'];target=next(iter(kind.values())) if isinstance(kind,dict) else None
            kind_name=next(iter(kind)) if isinstance(kind,dict) else kind
            observations.append({'caller_node':int(caller),'caller':node,'caller_byte_range':node_range,'owner_classification':owner,'edge_index':edge_index,'edge':edge,'kind':kind_name,'callee_node':target,'callee':ws_example_nodes[target] if target is not None else None,'source_byte_range':r,'source_text':ws_bound_bytes[p][slice(*r)].decode(),'correlations':correlations,'semantics':'address_flow' if kind_name=='FnPtrCast' else 'compiler_observation'})
    row={'caller':binding['caller'],'caller_id':binding['caller_id'],'source_state':binding['state'],'source_file':p,'source_sha256':hashlib.sha256(ws_bound_bytes[p]).hexdigest(),'definition_count':len(definitions),'observation_count':len(observations),'exact_call_span_observations':sum(any(c['relation']=='exact_source_call_span' for c in e['correlations']) for e in observations),'macro_span_observations':sum(any(c['relation']=='within_recorded_macro_invocation_span' for c in e['correlations']) for e in observations),'kinds':dict(Counter(e['kind'] for e in observations)),'definitions':definitions,'observations':observations,'status':'source_bound_observations' if observations else 'no_emitted_source_match'}
    ws_example_cases.append(row)
ws_example_frontier={'scope':'raw_workspace_existing_example_binary','raw_export_sha256':ws_example_selected['sha256'],'cargo_exit':0,'main_root':'panic_evidence::main','cases':ws_example_cases,'measured_source_match_count':sum(c['observation_count']>0 for c in ws_example_cases),'frontier_size':7,'typed_workspace_importer_supported':False,'limits':['Source matches do not prove panic feasibility.','Closure ownership is never assigned to the outer function.','Within macro invocation spans is a raw source correlation, not expansion provenance.','No emitted match is not absence of panic.']}
_=(ws_example_dir/'frontier-correlations.json').write_text(json.dumps(ws_example_frontier,indent=2)+'\n')
print(json.dumps({k:v for k,v in ws_example_frontier.items() if k!='cases'},indent=2))
for row in ws_example_cases:
    print(json.dumps({k:v for k,v in row.items() if k not in ['definitions','observations']},ensure_ascii=False))


{
  "scope": "raw_workspace_existing_example_binary",
  "raw_export_sha256": "fa1f11f94164756f69f53b0011d4fecf17f9978127d363dc263361cc4d4a50d4",
  "cargo_exit": 0,
  "main_root": "panic_evidence::main",
  "measured_source_match_count": 5,
  "frontier_size": 7,
  "typed_workspace_importer_supported": false,
  "limits": [
    "Source matches do not prove panic feasibility.",
    "Closure ownership is never assigned to the outer function.",
    "Within macro invocation spans is a raw source correlation, not expansion provenance.",
    "No emitted match is not absence of panic."
  ]
}
{"caller": "run_full_walk_into_with_stats", "caller_id": "10dad659a68a8b0f", "source_state": "exact function caller", "source_file": "crates/spur-graph/src/git_walk.rs", "source_sha256": "13de1fe19d15b2c94ac49e65be2933ce62610fa2087f43ed5ef2de990b5e7d15", "definition_count": 0, "observation_count": 0, "exact_call_span_observations": 0, "macro_span_observations": 0, "kinds": {}, "status": "no_emitted_source_mat

In [123]:
ws_example_flat=ws_example_dir/'flat'
ws_example_flat.mkdir(exist_ok=True)
ws_case_sql=[]
ws_observation_sql=[]
for case in ws_example_cases:
    exact=[e for e in case['observations'] if any(c['relation']=='exact_source_call_span' for c in e['correlations'])]
    ws_case_sql.append({'caller':case['caller'],'source_state':case['source_state'],'source_bound_nodes':case['definition_count'],'source_bound_observations':case['observation_count'],'exact_call_span_observations':case['exact_call_span_observations'],'same_function_exact_calls':sum(e['kind']=='Called' and e['owner_classification']=='same_function_start' for e in exact),'same_closure_exact_calls':sum(e['kind']=='Called' and e['owner_classification']=='same_closure_start' for e in exact),'macro_span_observations':case['macro_span_observations'],'builtin_asserts':case['kinds'].get('BuiltinAssert',0),'availability':case['status']})
    for e in case['observations']:
        ws_observation_sql.append({'frontier_caller':case['caller'],'compiler_caller':e['caller']['name'],'compiler_callee':e['callee']['name'] if e['callee'] else None,'owner':e['owner_classification'],'edge_kind':e['kind'],'source_file':case['source_file'],'start_byte':e['source_byte_range'][0],'end_byte':e['source_byte_range'][1],'source_text':e['source_text'],'exact_call_span':any(c['relation']=='exact_source_call_span' for c in e['correlations']),'within_macro_span':any(c['relation']=='within_recorded_macro_invocation_span' for c in e['correlations'])})
_=(ws_example_flat/'frontier.jsonl').write_text(''.join(json.dumps(row)+'\n' for row in ws_case_sql))
_=(ws_example_flat/'observations.jsonl').write_text(''.join(json.dumps(row)+'\n' for row in ws_observation_sql))
print(json.dumps({'case_rows':len(ws_case_sql),'source_observation_rows':len(ws_observation_sql),'exact_function_calls':sum(r['same_function_exact_calls'] for r in ws_case_sql),'exact_closure_calls':sum(r['same_closure_exact_calls'] for r in ws_case_sql),'implicit_check_observations':sum(r['builtin_asserts'] for r in ws_case_sql)},indent=2))


{
  "case_rows": 7,
  "source_observation_rows": 277,
  "exact_function_calls": 2,
  "exact_closure_calls": 1,
  "implicit_check_observations": 8
}


In [124]:
DESCRIBE SELECT * FROM read_json_auto('/private/tmp/panic-workspace-20261101/example-d1_o1/flat/frontier.jsonl');

column_name,column_type,null,key,default,extra
caller,VARCHAR,YES,NaN,NaN,NaN
source_state,VARCHAR,YES,NaN,NaN,NaN
source_bound_nodes,BIGINT,YES,NaN,NaN,NaN
source_bound_observations,BIGINT,YES,NaN,NaN,NaN
exact_call_span_observations,BIGINT,YES,NaN,NaN,NaN
same_function_exact_calls,BIGINT,YES,NaN,NaN,NaN
same_closure_exact_calls,BIGINT,YES,NaN,NaN,NaN
macro_span_observations,BIGINT,YES,NaN,NaN,NaN
builtin_asserts,BIGINT,YES,NaN,NaN,NaN
availability,VARCHAR,YES,NaN,NaN,NaN


In [125]:
CREATE OR REPLACE TEMP TABLE ws_example_frontier AS
SELECT * FROM read_json_auto('/private/tmp/panic-workspace-20261101/example-d1_o1/flat/frontier.jsonl');
COPY ws_example_frontier TO '/private/tmp/panic-workspace-20261101/example-d1_o1/flat/frontier.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
SELECT caller, source_bound_observations, same_function_exact_calls, same_closure_exact_calls,
       macro_span_observations, builtin_asserts, availability
FROM ws_example_frontier;
DESCRIBE SELECT * FROM read_json_auto('/private/tmp/panic-workspace-20261101/example-d1_o1/flat/observations.jsonl');

column_name,column_type,null,key,default,extra
frontier_caller,VARCHAR,YES,NaN,NaN,NaN
compiler_caller,VARCHAR,YES,NaN,NaN,NaN
compiler_callee,VARCHAR,YES,NaN,NaN,NaN
owner,VARCHAR,YES,NaN,NaN,NaN
edge_kind,VARCHAR,YES,NaN,NaN,NaN
source_file,VARCHAR,YES,NaN,NaN,NaN
start_byte,BIGINT,YES,NaN,NaN,NaN
end_byte,BIGINT,YES,NaN,NaN,NaN
source_text,VARCHAR,YES,NaN,NaN,NaN
exact_call_span,BOOLEAN,YES,NaN,NaN,NaN


In [126]:
SELECT caller, source_bound_observations, same_function_exact_calls, same_closure_exact_calls,
       macro_span_observations, builtin_asserts, availability
FROM ws_example_frontier;

caller,source_bound_observations,same_function_exact_calls,same_closure_exact_calls,macro_span_observations,builtin_asserts,availability
run_full_walk_into_with_stats,0,0,0,0,0,no_emitted_source_match
write_edges,0,0,0,0,0,no_emitted_source_match
rebuild_from_buckets,135,0,0,0,0,source_bound_observations
stable_file_id_from_path,10,0,0,7,0,source_bound_observations
impl ExternalImportRegistry::write_to_buckets,21,0,1,0,0,source_bound_observations
compute_anchor_hash,19,1,0,0,0,source_bound_observations
resolve_bare_pending_edge,92,1,0,0,8,source_bound_observations


In [127]:
CREATE OR REPLACE TEMP TABLE ws_example_observations AS
SELECT * FROM read_json_auto('/private/tmp/panic-workspace-20261101/example-d1_o1/flat/observations.jsonl');
COPY ws_example_observations TO '/private/tmp/panic-workspace-20261101/example-d1_o1/flat/observations.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
SELECT frontier_caller, owner, edge_kind, compiler_callee, start_byte, end_byte, source_text
FROM ws_example_observations
WHERE exact_call_span OR edge_kind='BuiltinAssert'
ORDER BY frontier_caller, start_byte, end_byte;

frontier_caller,owner,edge_kind,compiler_callee,start_byte,end_byte,source_text
compute_anchor_hash,same_function_start,Called,"std::result::Result::<[u8; 8], std::array::TryFromSliceError>::expect",3776,3881,"digest[..8]\n .try_into()\n .expect(""SHA-256 digest always has at least eight bytes"")"
impl ExternalImportRegistry::write_to_buckets,same_closure_start,Called,std::option::Option::<u64>::expect,55651,55798,"self\n .next_node_id\n .checked_add(1)\n .expect(""exhausted synthetic external NodeId space"")"
resolve_bare_pending_edge,same_function_start,BuiltinAssert,NaN,35267,35293,*ambiguous_unresolved += 1
resolve_bare_pending_edge,same_function_start,BuiltinAssert,NaN,37216,37242,*ambiguous_unresolved += 1
resolve_bare_pending_edge,same_function_start,Called,std::option::Option::<&spur_graph::tree_sitter::PendingEdge>::expect,37885,37931,"fallback_edge.as_ref().expect(""fallback edge"")"
resolve_bare_pending_edge,same_function_start,BuiltinAssert,NaN,39011,39037,*ambiguous_unresolved += 1
resolve_bare_pending_edge,same_function_start,BuiltinAssert,NaN,39739,39765,*ambiguous_unresolved += 1
resolve_bare_pending_edge,same_function_start,BuiltinAssert,NaN,40421,40447,*ambiguous_unresolved += 1
resolve_bare_pending_edge,same_function_start,BuiltinAssert,NaN,41642,41668,*ambiguous_unresolved += 1
resolve_bare_pending_edge,same_function_start,BuiltinAssert,NaN,42561,42587,*ambiguous_unresolved += 1


In [128]:
ws_example_binary=ws_work/'target/aarch64-apple-darwin/debug/examples/panic_evidence'
ws_smoke_root=ws_example_dir/'runtime-smoke-fixture'
(ws_smoke_root/'src').mkdir(parents=True,exist_ok=False)
ws_smoke_text='pub fn probe(x: Option<u8>) -> u8 { x.expect("smoke") }\n'
_=(ws_smoke_root/'src/lib.rs').write_text(ws_smoke_text)
ws_smoke_request={'command':'snapshot','root':str(ws_smoke_root),'source_paths':['src/lib.rs']}
ws_smoke_run=subprocess.run([str(ws_example_binary)],input=json.dumps(ws_smoke_request).encode(),stdout=subprocess.PIPE,stderr=subprocess.PIPE,cwd=ws_repo,env=ws_env,timeout=30)
_=(ws_example_dir/'runtime-smoke.stdout').write_bytes(ws_smoke_run.stdout)
_=(ws_example_dir/'runtime-smoke.stderr').write_bytes(ws_smoke_run.stderr)
assert ws_smoke_run.returncode==0,ws_smoke_run.stderr.decode(errors='replace')[:500]
ws_smoke_response=json.loads(ws_smoke_run.stdout)
assert 'graph' in ws_smoke_response and 'method_sites' in ws_smoke_response
ws_smoke_result={'exit':ws_smoke_run.returncode,'binary_sha256':hashlib.sha256(ws_example_binary.read_bytes()).hexdigest(),'binary_byte_length':ws_example_binary.stat().st_size,'argv':[str(ws_example_binary)],'request':ws_smoke_request,'fixture_source_text':ws_smoke_text,'response_keys':sorted(ws_smoke_response),'stdout_sha256':hashlib.sha256(ws_smoke_run.stdout).hexdigest(),'stderr_sha256':hashlib.sha256(ws_smoke_run.stderr).hexdigest(),'scope':'Runtime smoke of actual compiled sidecar on separate synthetic source; does not compile or run probe, prove a panic, or publish CURRENT.'}
_=(ws_example_dir/'runtime-smoke.json').write_text(json.dumps(ws_smoke_result,indent=2)+'\n')
print(json.dumps(ws_smoke_result,indent=2))


{
  "exit": 0,
  "binary_sha256": "9d515d0b0160afef11b9063e24383f621006332be14cba6db9259899a5d46777",
  "binary_byte_length": 36696272,
  "argv": [
    "/private/tmp/panic-workspace-20261101/target/aarch64-apple-darwin/debug/examples/panic_evidence"
  ],
  "request": {
    "command": "snapshot",
    "root": "/private/tmp/panic-workspace-20261101/example-d1_o1/runtime-smoke-fixture",
    "source_paths": [
      "src/lib.rs"
    ]
  },
  "fixture_source_text": "pub fn probe(x: Option<u8>) -> u8 { x.expect(\"smoke\") }\n",
  "response_keys": [
    "graph",
    "method_sites"
  ],
  "stdout_sha256": "fd903e5a9153e1847e12da400f3be7e40d5b7088cccf6ed6bd37ebed7f5654be",
  "stderr_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
  "scope": "Runtime smoke of actual compiled sidecar on separate synthetic source; does not compile or run probe, prove a panic, or publish CURRENT."
}


In [129]:
# Standalone replay: no earlier notebook variables or temporary build outputs.
from pathlib import Path
import json, runpy
panic_workspace_bundle=Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-panic-workspace')
panic_workspace_verifier=runpy.run_path(str(panic_workspace_bundle/'replay.py'))
panic_workspace_replay=panic_workspace_verifier['verify'](panic_workspace_bundle)
print(json.dumps(panic_workspace_replay,indent=2))


{
  "payloads_verified": 29,
  "source_files_verified": 5,
  "captures": [
    {
      "label": "library",
      "nodes": 79008,
      "observations": 250319,
      "roots": 13,
      "kinds": {
        "Called": 202261,
        "AsyncPanic": 59,
        "FnPtrCast": 584,
        "VTableCalled": 32611,
        "BuiltinAssert": 14804
      }
    },
    {
      "label": "example",
      "nodes": 77468,
      "observations": 246520,
      "roots": 1,
      "kinds": {
        "Called": 200036,
        "VTableCalled": 31862,
        "BuiltinAssert": 14060,
        "FnPtrCast": 562
      }
    },
    {
      "label": "cli-partial-graph",
      "nodes": 81244,
      "observations": 257936,
      "roots": 13,
      "kinds": {
        "Called": 208904,
        "AsyncPanic": 59,
        "FnPtrCast": 606,
        "BuiltinAssert": 14996,
        "VTableCalled": 33371
      }
    },
    {
      "label": "cli-partial-tui",
      "nodes": 152128,
      "observations": 525486,
      "roots": 4,
      

In [130]:
WITH coverage AS (
  SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-panic-workspace/sql/example-frontier.parquet')
), observations AS (
  SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-panic-workspace/sql/example-observations.parquet')
), checks AS (
  SELECT 'frontier regions' AS metric, count(*)::BIGINT AS actual, 7::BIGINT AS expected FROM coverage
  UNION ALL SELECT 'regions with source observations', count(*) FILTER (WHERE source_bound_observations>0), 5 FROM coverage
  UNION ALL SELECT 'exact function calls', sum(same_function_exact_calls)::BIGINT, 2 FROM coverage
  UNION ALL SELECT 'exact closure calls', sum(same_closure_exact_calls)::BIGINT, 1 FROM coverage
  UNION ALL SELECT 'implicit checks', sum(builtin_asserts)::BIGINT, 8 FROM coverage
  UNION ALL SELECT 'source observations', count(*), 277 FROM observations
  UNION ALL SELECT 'exact source call spans', count(*) FILTER (WHERE exact_call_span), 3 FROM observations
  UNION ALL SELECT 'raw macro unwrap correlations', count(*) FILTER (WHERE within_macro_span AND ends_with(compiler_callee,'::unwrap')), 1 FROM observations
  UNION ALL SELECT 'library nodes', count(*), 79008 FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-panic-workspace/sql/library-nodes.parquet')
  UNION ALL SELECT 'library observations', count(*), 250319 FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-panic-workspace/sql/library-edges.parquet')
)
SELECT metric,actual,expected,actual=expected AS matches FROM checks;

metric,actual,expected,matches
frontier regions,7,7,True
regions with source observations,5,5,True
exact function calls,2,2,True
exact closure calls,1,1,True
implicit checks,8,8,True
source observations,277,277,True
exact source call spans,3,3,True
raw macro unwrap correlations,1,1,True
library nodes,79008,79008,True
library observations,250319,250319,True


## Workspace compile experiment: measured outcome

The local `panic_evidence` executable compiled successfully with the pinned CGA producer, and its JSON-command smoke run exited 0. The main-rooted export has **77,468 nodes / 246,520 observations**. The separate full CLI attempt exited **101** with a producer ICE resolving `std::ptr::drop_glue::<S>`; it has no `spur::main` export.

| Measure | Result |
|---|---:|
| Function regions with source-bound observations | 5 / 7 |
| Exact collected method-call spans | 3: two functions, one closure |
| Raw `unwrap` correlation inside recorded macro tokens | 1 |
| Additional resolver `BuiltinAssert` observations | 8 |
| Archived payloads verified by standalone replay | 29 |
| SQL checks against retained Parquet | 10 / 10 |

The five observed regions are not five validated panic sites: `rebuild_from_buckets` lacks an exact match for its collected closure `expect`, and two other functions have no emitted source match. The matched closure keeps its own compiler identity; the outer graph caller remains null. The `format!` token correlation does not supply expansion provenance. The implicit checks carry source locations but no exported reason payload.

The library's six-call path to `panic_fmt` is a structural candidate. The tracing setter experiment rejects an invalid state in a one-step model and finds a witness after adding a hypothetical invalid write. These results do not demonstrate a workspace runtime panic or prove whole-workspace safety.

For replay without compilation, run cell `2e2a3e01-7aec-416e-87a6-4bc2b8929330` and the following SQL check cell. Historical compiler launch cells are retained as capture provenance. All replay inputs now come from [the durable dossier](docs/research/artifacts/2026-11-01-panic-workspace/README.md), whose payload-index SHA256 is `e8c4ed7d6b3bfd39508a96df377dc135d0b05bd9ba55d147ce83f862a35ec794`.

The published v11 graph, manifest and five protected user files remain unchanged. The next implementation gap is an explicit workspace-target import contract, followed by compiler-backed macro/closure association and value-flow conditions. A full CLI capture additionally needs a producer fix for the generic drop-glue ICE.


## CGA failure isolation — approved investigation

Issue `bd-3qw` isolates the CLI producer failure before proposing a fix. The original rustc/CGA binaries and toolchain remain pinned. A controlled comparison will distinguish normal compilation, analysis without export, and analysis with export; an analysis-disabled success is a control, not a panic graph.

The saved failure is `failed to resolve instance for std::ptr::drop_glue::<S>` at pinned rustc `instance.rs:640`. That branch is reached when `try_resolve` returns no instance below the type-length limit. Pinned rustc's `resolve_drop_glue` uses `TypingEnv::fully_monomorphized()`. Upstream CGA's `visit_drop_use` calls that resolver directly. The failing visitor/root and reason generic `S` remains unresolved are still under investigation.

Codex (`gpt-6.1-sol`, `xhigh`) owns reproduction and reduction in an isolated worktree. Root owns this notebook and review. Reproducer and diagnostic patches will remain separate from a production fix.


In [131]:
# Baseline only. Compiler/reduction commands are owned by the isolated worker.
from pathlib import Path
import hashlib,json,subprocess
cga_iso_repo=Path('/Users/kevintruong/project/SpurCode')
cga_iso_prior=cga_iso_repo/'docs/research/artifacts/2026-11-01-panic-workspace'
cga_iso_before=json.loads((cga_iso_prior/'before.json').read_text())
cga_iso_hash=lambda path:hashlib.sha256(path.read_bytes()).hexdigest()
cga_iso_head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=cga_iso_repo,text=True).strip()
assert cga_iso_head=='a9ee997c925d243cd1bbe4e4895867c2ff11b70e'
assert all(cga_iso_hash(cga_iso_repo/p)==h for p,h in cga_iso_before['user_dirty_files'].items())
assert str((cga_iso_repo/'.spur/graph/CURRENT').resolve())==cga_iso_before['graph_target']
assert cga_iso_hash(Path(cga_iso_before['graph_target'])/'manifest.json')==cga_iso_before['manifest_sha256']
cga_iso_driver=Path('/private/tmp/spur-cga-pilot-20261031/target/release/cga-driver')
assert cga_iso_hash(cga_iso_driver)=='5647b7f2c7f857dda0dd0204e91d870287988181ca5ec4d7792810a561886172'
cga_iso_root_dir=Path('/private/tmp/cga-isolation-root-20261101')
cga_iso_root_dir.mkdir(exist_ok=True)
cga_iso_baseline={'issue':'bd-3qw','delegation_id':'8e8983be-9105-47a0-8204-cf92383c97c3','source_head':cga_iso_head,'protected_files':cga_iso_before['user_dirty_files'],'graph_target':cga_iso_before['graph_target'],'manifest_sha256':cga_iso_before['manifest_sha256'],'original_driver_sha256':cga_iso_hash(cga_iso_driver),'historical_cli_exit':101,'current_status':'isolation in progress; no minimized failure or fix claimed'}
_=(cga_iso_root_dir/'baseline.json').write_text(json.dumps(cga_iso_baseline,indent=2)+'\n')
print(json.dumps({'source_head':cga_iso_head,'protected_file_hashes_verified':5,'published_graph_preserved':True,'original_producer_verified':True,'issue':'bd-3qw','status':'worker reproduction/reduction running'},indent=2))


{
  "source_head": "a9ee997c925d243cd1bbe4e4895867c2ff11b70e",
  "protected_file_hashes_verified": 5,
  "published_graph_preserved": true,
  "original_producer_verified": true,
  "issue": "bd-3qw",
  "status": "worker reproduction/reduction running"
}


In [132]:
# Root verification uses seven fresh dependency-free compiler targets.
from pathlib import Path
import hashlib,json,subprocess,sys,tempfile,time
cga_root_repo=Path('/Users/kevintruong/project/SpurCode')
cga_root_bundle=cga_root_repo/'docs/research/artifacts/2026-11-01-cga-isolation'
cga_root_prior=json.loads((cga_root_repo/'docs/research/artifacts/2026-11-01-panic-workspace/before.json').read_text())
cga_root_hash=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
cga_root_head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=cga_root_repo,text=True).strip()
assert cga_root_head=='e0e6ebef676fc51c2fffc8c7581dbfdeab54e72d'
assert all(cga_root_hash(cga_root_repo/p)==h for p,h in cga_root_prior['user_dirty_files'].items())
assert str((cga_root_repo/'.spur/graph/CURRENT').resolve())==cga_root_prior['graph_target']
assert cga_root_hash(Path(cga_root_prior['graph_target'])/'manifest.json')==cga_root_prior['manifest_sha256']
cga_root_scratch=Path(tempfile.mkdtemp(prefix='cga-isolation-root-controls-',dir='/private/tmp'))
cga_root_output=cga_root_scratch/'controls'
cga_root_argv=[sys.executable,str(cga_root_bundle/'run.py'),'--output',str(cga_root_output)]
cga_root_plan={'issue':'bd-3qw','source_head':cga_root_head,'argv':cga_root_argv,'cwd':str(cga_root_repo),'harness_sha256':cga_root_hash(cga_root_bundle/'run.py'),'producer_sha256':cga_root_hash(Path('/private/tmp/spur-cga-pilot-20261031/target/release/cga-driver')),'scope':'root fresh compiler verification only; no runtime tests or producer fix'}
_=(cga_root_scratch/'root-plan.json').write_text(json.dumps(cga_root_plan,indent=2)+'\n')
cga_root_started=time.monotonic()
with (cga_root_scratch/'harness.stdout').open('wb') as out,(cga_root_scratch/'harness.stderr').open('wb') as err:
    cga_root_process=subprocess.Popen(cga_root_argv,cwd=cga_root_repo,stdout=out,stderr=err)
print(json.dumps({'state':'running','pid':cga_root_process.pid,'output':str(cga_root_output),'source_head':cga_root_head},indent=2))


{
  "state": "running",
  "pid": 90029,
  "output": "/private/tmp/cga-isolation-root-controls-eo4gmtnc/controls",
  "source_head": "e0e6ebef676fc51c2fffc8c7581dbfdeab54e72d"
}


In [133]:
cga_root_exit=cga_root_process.poll()
if cga_root_exit is None:
    print(json.dumps({'state':'running','elapsed_seconds_at_poll':round(time.monotonic()-cga_root_started,2)}))
else:
    assert cga_root_exit==0,(cga_root_scratch/'harness.stderr').read_text(errors='replace')[-2000:]
    cga_root_verification=json.loads((cga_root_output/'verification.json').read_text())
    cga_root_inventory=json.loads((cga_root_output/'inventory.json').read_text())
    assert cga_root_verification['all_seven_controls_match'] is True
    cga_root_cases=cga_root_verification['cases']
    assert len(cga_root_cases)==7
    assert all(c['fresh_compiler_invocation'] for c in cga_root_cases)
    assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=cga_root_repo,text=True).strip()==cga_root_head
    assert all(cga_root_hash(cga_root_repo/p)==h for p,h in cga_root_prior['user_dirty_files'].items())
    assert str((cga_root_repo/'.spur/graph/CURRENT').resolve())==cga_root_prior['graph_target']
    assert cga_root_hash(Path(cga_root_prior['graph_target'])/'manifest.json')==cga_root_prior['manifest_sha256']
    cga_root_rows=[{'case':c['name'],'actual_exit':c['exit'],'expected_exit':c['expected_exit'],'exact_ice':c['ice_signature'],'exports':len(c['exports']),'fresh_invocation':c['fresh_compiler_invocation'],'elapsed_seconds':c['elapsed_seconds'],'compiler_nodes':[n for g in c['exports'] for n in g['graph']['node_names']]} for c in cga_root_cases]
    cga_root_result={'issue':'bd-3qw','worker_commit':cga_root_head,'harness_exit':cga_root_exit,'all_seven_controls_match':True,'cases':cga_root_rows,'protected_hashes_match':True,'graph_pointer_manifest_match':True,'source_head_unchanged_during_root_controls':True,'driver_sha256':cga_root_inventory['driver_sha256'],'root_control_inventory_sha256':cga_root_hash(cga_root_output/'inventory.json'),'production_fix_tested':False,'runtime_tests':False}
    _=(cga_root_scratch/'root-result.json').write_text(json.dumps(cga_root_result,indent=2)+'\n')
    print(json.dumps(cga_root_result,indent=2))


{
  "issue": "bd-3qw",
  "worker_commit": "e0e6ebef676fc51c2fffc8c7581dbfdeab54e72d",
  "harness_exit": 0,
  "all_seven_controls_match": true,
  "cases": [
    {
      "case": "enabled",
      "actual_exit": 101,
      "expected_exit": 101,
      "exact_ice": true,
      "exports": 0,
      "fresh_invocation": true,
      "elapsed_seconds": 0.2854486250143964,
      "compiler_nodes": []
    },
    {
      "case": "extract-false",
      "actual_exit": 101,
      "expected_exit": 101,
      "exact_ice": true,
      "exports": 0,
      "fresh_invocation": true,
      "elapsed_seconds": 0.08531629201024771,
      "compiler_nodes": []
    },
    {
      "case": "analysis-unset",
      "actual_exit": 0,
      "expected_exit": 0,
      "exact_ice": false,
      "exports": 0,
      "fresh_invocation": true,
      "elapsed_seconds": 0.0955298749904614,
      "compiler_nodes": []
    },
    {
      "case": "plain-rustc",
      "actual_exit": 0,
      "expected_exit": 0,
      "exact_ice": false,

In [134]:
# Preserve root controls and their raw evidence; exclude target/cache directories.
from collections import Counter
import gzip,io,shutil,tarfile
assert cga_root_exit==0
cga_root_review=cga_root_bundle/'review'
cga_root_review.mkdir(exist_ok=False)
cga_root_records=[]
for file in sorted(cga_root_scratch.rglob('*')):
    rel=file.relative_to(cga_root_scratch)
    if file.is_file() and 'target' not in rel.parts:
        cga_root_records.append((str(rel),file))
with (cga_root_review/'root-controls.tar.gz').open('wb') as out:
    with gzip.GzipFile(filename='',mode='wb',fileobj=out,mtime=0,compresslevel=9) as zipped:
        with tarfile.open(fileobj=zipped,mode='w|') as tar:
            for name,file in cga_root_records:
                data=file.read_bytes()
                info=tarfile.TarInfo(name);info.size=len(data);info.mtime=0;info.mode=0o644
                tar.addfile(info,io.BytesIO(data))
cga_root_capture_index={'archive':'root-controls.tar.gz','archive_sha256':cga_root_hash(cga_root_review/'root-controls.tar.gz'),'members':[{'path':name,'byte_length':file.stat().st_size,'sha256':cga_root_hash(file)} for name,file in cga_root_records]}
_=(cga_root_review/'root-controls-index.json').write_text(json.dumps(cga_root_capture_index,indent=2)+'\n')
shutil.copyfile(cga_root_scratch/'root-result.json',cga_root_review/'root-result.json')
shutil.copyfile(Path('/private/tmp/cga-isolation-root-20261101/audit-repair.json'),cga_root_review/'audit-repair.json')
shutil.copyfile(Path('/private/tmp/cga-isolation-root-20261101/rustc-source-identity.json'),cga_root_review/'rustc-source-identity.json')
with (cga_root_review/'rustc-instance.rs.gz').open('wb') as out:
    with gzip.GzipFile(filename='',mode='wb',fileobj=out,mtime=0,compresslevel=9) as zipped:
        zipped.write(Path('/private/tmp/cga-isolation-root-20261101/rustc-instance.rs').read_bytes())
cga_root_flat=[]
for case in cga_root_cases:
    nodes=[n for export in case['exports'] for n in export['graph']['node_names']]
    cga_root_flat.append({'case':case['name'],'compiler':'pinned rustc' if case['name']=='plain-rustc' else 'original CGA driver','cga_input':case['environment'].get('CGA_INPUT','unset'),'actual_exit':case['exit'],'expected_exit':case['expected_exit'],'matches':case['exit']==case['expected_exit'],'exact_ice':case['ice_signature'],'fresh_compiler':case['fresh_compiler_invocation'],'exports':len(case['exports']),'unresolved_generic_in_export':any('Trait<S>' in n for n in nodes) if case['exports'] else None})
_=(cga_root_review/'root-controls.jsonl').write_text(''.join(json.dumps(row)+'\n' for row in cga_root_flat))
cga_root_final={'source_head_during_fresh_controls':cga_root_head,'worker_commit':cga_root_head,'protected_files':{p:{'expected':h,'actual':cga_root_hash(cga_root_repo/p)} for p,h in cga_root_prior['user_dirty_files'].items()},'graph_target':str((cga_root_repo/'.spur/graph/CURRENT').resolve()),'manifest_sha256':cga_root_hash(Path(cga_root_prior['graph_target'])/'manifest.json'),'original_driver_sha256':cga_root_hash(Path('/private/tmp/spur-cga-pilot-20261031/target/release/cga-driver')),'worker_evidence_index_sha256':cga_root_hash(cga_root_bundle/'evidence-index.json'),'prior_panic_workspace_index_sha256':cga_root_hash(cga_root_repo/'docs/research/artifacts/2026-11-01-panic-workspace/evidence-index.json'),'root_notebook_cells':{'launch':'36938cf1-38ed-4938-8fcf-df67c5b8f061','actual_exit_and_controls':'509a691e-422c-47e7-832f-f2e89a0e6159'}}
assert all(r['expected']==r['actual'] for r in cga_root_final['protected_files'].values())
_=(cga_root_review/'final-protection.json').write_text(json.dumps(cga_root_final,indent=2)+'\n')
print(json.dumps({'archive_members':len(cga_root_records),'archive_bytes':(cga_root_review/'root-controls.tar.gz').stat().st_size,'root_review':str(cga_root_review),'ledger_rows':len(cga_root_flat),'target_directories_retained':False},indent=2))


{
  "archive_members": 65,
  "archive_bytes": 12969,
  "root_review": "/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-cga-isolation/review",
  "ledger_rows": 7,
  "target_directories_retained": false
}


In [135]:
DESCRIBE SELECT * FROM read_json_auto('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-cga-isolation/review/root-controls.jsonl');

column_name,column_type,null,key,default,extra
case,VARCHAR,YES,NaN,NaN,NaN
compiler,VARCHAR,YES,NaN,NaN,NaN
cga_input,VARCHAR,YES,NaN,NaN,NaN
actual_exit,BIGINT,YES,NaN,NaN,NaN
expected_exit,BIGINT,YES,NaN,NaN,NaN
matches,BOOLEAN,YES,NaN,NaN,NaN
exact_ice,BOOLEAN,YES,NaN,NaN,NaN
fresh_compiler,BOOLEAN,YES,NaN,NaN,NaN
exports,BIGINT,YES,NaN,NaN,NaN
unresolved_generic_in_export,BOOLEAN,YES,NaN,NaN,NaN


In [136]:
COPY (SELECT * FROM read_json_auto('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-cga-isolation/review/root-controls.jsonl'))
TO '/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-cga-isolation/review/root-controls.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
SELECT "case",compiler,actual_exit,expected_exit,matches,exact_ice,exports,unresolved_generic_in_export
FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-cga-isolation/review/root-controls.parquet');

case,compiler,actual_exit,expected_exit,matches,exact_ice,exports,unresolved_generic_in_export
enabled,original CGA driver,101,101,True,True,0,None
extract-false,original CGA driver,101,101,True,True,0,None
analysis-unset,original CGA driver,0,0,True,False,0,None
plain-rustc,pinned rustc,0,0,True,False,0,None
static-dispatch,original CGA driver,0,0,True,False,1,False
concrete-trait,original CGA driver,0,0,True,False,1,False
reference-value,original CGA driver,0,0,True,False,1,True


In [137]:
# Durable replay: no compiler, live build cache, or earlier notebook variables.
from pathlib import Path
import json,runpy
cga_review_path=Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-cga-isolation/review')
cga_review_module=runpy.run_path(str(cga_review_path/'replay.py'))
cga_review_replay=cga_review_module['verify']()
print(json.dumps(cga_review_replay,indent=2))


{
  "isolation_verified": true,
  "root_payloads": 12,
  "worker_archive_members": 138,
  "root_archive_members": 65,
  "fresh_compiler_controls": [
    {
      "case": "enabled",
      "exit": 101,
      "exports": 0
    },
    {
      "case": "extract-false",
      "exit": 101,
      "exports": 0
    },
    {
      "case": "analysis-unset",
      "exit": 0,
      "exports": 0
    },
    {
      "case": "plain-rustc",
      "exit": 0,
      "exports": 0
    },
    {
      "case": "static-dispatch",
      "exit": 0,
      "exports": 1
    },
    {
      "case": "concrete-trait",
      "exit": 0,
      "exports": 1
    },
    {
      "case": "reference-value",
      "exit": 0,
      "exports": 1
    }
  ],
  "production_fix_tested": false,
  "runtime_tests": false,
  "meaning": "Expected failures and passing controls reproduced; reference-value export still has unresolved S."
}


## Isolated result: CGA virtual-call substitution

A four-line, dependency-free library reproduces the exact producer ICE:

```rust
pub trait Trait<S> { fn consume(&self, value: S); }
impl<S> Trait<S> for () { fn consume(&self, _value: S) {} }
#[no_mangle]
pub fn entry(object: &dyn Trait<()>) { object.consume(()); }
```

CGA's pinned `dyn_object_call_to_impl_methods` ignores the incoming concrete `_args` and reconstructs trait arguments with identity parameters. Here the actual argument is unit `()`, but the visited implementation has `S/#1`. Its owned-value cleanup asks rustc to resolve `drop_glue::<S>` under a fully monomorphized environment and triggers the recorded ICE. The CLI trace identifies the same transition in Octocrab's `AuthHeaderLayer` implementation of `Layer<S>`. The earlier root-collector candidate is not the failing stack.

**Root independently reproduced all seven expected compiler outcomes** in fresh targets. Analysis enabled fails with exit 101 whether export is on or off. Ordinary rustc and analysis-disabled compilation succeed. Static and nongeneric virtual controls emit concrete graphs. Passing the generic argument by reference avoids the ICE but still exports unresolved `Trait<S>`; exit 0 alone therefore misses an analysis defect.

The accepted [reproducer dossier](docs/research/artifacts/2026-11-01-cga-isolation/README.md) is preserved in commit `e0e6ebef676fc51c2fffc8c7581dbfdeab54e72d`. The [root review](docs/research/artifacts/2026-11-01-cga-isolation/review/README.md) retains fresh controls, SQL data, upstream byte checks and exact audit-repair evidence. Durable replay verified **138 worker archive members + 65 root archive members**, including **12 root payloads**. Root review index SHA256: `f055c207cd5978fc92d22df8ebc63c962dd1e80865cbc581c5cf88dd396eaa89`.

The next producer regression must require analysis-enabled compilation **and concrete exported implementation arguments** for both the owned-value and reference-value inputs. Skipping unresolved implementations would leave a coverage gap. No producer fix or successful full CLI capture is claimed here. The published graph, original producer and five protected user files remain unchanged.
